# Lib Installation

In [ ]:
!pip install  lightning pytorch-lightning torchvision tqdm python-dotenv PyYAML wilds

In [ ]:
!pip install --upgrade scipy
# Then restart runtime

In [ ]:
# Option 1: Install older albumentations
!pip install --upgrade albumentations

In [ ]:
!pip install numpy==1.24.3 scipy==1.11.4 lightning==2.1.0
# Then restart runtime

# Config

In [56]:
# =============================================================================
# YOUR EXISTING CONFIG - JUST CHANGE THE "algo" FIELD
# =============================================================================

config = {
    "algo": "FOND",  # <--- CHANGE THIS (was "FOND")
    "dataset": ["PACS", "VLCS", "OfficeHome"],
    "test_set_id": 0,
    
    # Paths
    "data_dir": {
        "PACS": "/kaggle/input/pacs-dataset/kfold",
        "VLCS": "/kaggle/input/vlcsdataset/",
        "OfficeHome": "/kaggle/input/officehome/OfficeHome/",
    },
    "log_dir": "./logs",
    
    # Training settings
    "n_epochs": 5001,
    "checkpoint_freq": 300,
    "holdout_fraction": 0.2,
    
    # Model checkpointing
    "model_checkpoint": {
        "metric": "val/oacc",
        "maximize": True
    },
    
    # Seeds
    "overall_seed": 1,
    "trial_id": 0,
    "hparam_id": 1,
    
    # System
    "num_workers": 4,
    
    # Dataset config
    "overlap": "high",
    "num_classes": None,
    "num_domain_linked_classes": None,
    "cdsa_y_l_multiplier": 1.0,
    # AutoAugment (RandAugment)
    "auto_augment": True,
    "augment_search_epochs": 5,
    "augment_policy_size": 5,
    "augment_num_policies": 3,
    
    # Teacher paths
    "teacher_paths": {}
}



# Imports

In [2]:
import math
from typing import List, Any, Dict, Optional, Callable, Tuple
import argparse
import logging
import hashlib
import numpy as np
import pandas as pd
from collections import Counter
import collections
import time
from datetime import datetime
import lightning as L
from tqdm import tqdm
import csv
import json
from pathlib import Path
from os.path import basename, dirname, join, exists, splitext
import os
from tqdm import tqdm
import shutil
from glob import glob
import torch
import torchmetrics
import torchvision.datasets.folder
from torch.utils.data import ConcatDataset, Dataset, Subset, TensorDataset
from torchvision import transforms
from torchvision.datasets import MNIST, ImageFolder
from torchvision.transforms.functional import rotate
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models
from wilds.datasets.camelyon17_dataset import Camelyon17Dataset
import scipy.misc
import gc
from scipy import linalg
from skimage.util import dtype, dtype_limits
from skimage.exposure import rescale_intensity
import skimage.exposure
import skimage.color
import scipy.ndimage
import math
from scipy import ndimage
import inspect

#from tensorflow.contrib import training as contrib_training
from PIL import Image, ImageEnhance, ImageOps, ImageFilter , ImageFile

import albumentations as A
from albumentations.augmentations.geometric.resize import RandomScale

import random

# SRC

## Domain Creation

In [3]:

def create_domains(
    num_classes: int, num_linked: int, num_train_domains: int
) -> List[List[int]]:
    """
    Determine and distribute domain-linked and domain-shared classes.
    Domain-linked classes will come from the same domain, i.e., idx=0.
    Domain-shared classes will exist in each of the remaining domains

    Args:
        num_classes: number of overall classes within all the domains
        num_linked: number of classes that are linked to an individual domain
        num_train_domains: number of different training domains
    """
    assert num_linked <= num_classes
    domain_shared = [i for i in range(num_linked, num_classes)]
    domain_linked = [i for i in range(num_linked)]
    logging.info(f"Domain-shared classes: {domain_shared}")
    logging.info(f"Domain-linked classes: {domain_linked}")
    # domains = [domain_shared.copy() for i in range(num_train_domains)]
    domains = [[] for i in range(num_train_domains)]

    # first domain is domain-linked only
    domains[0] = domain_linked
    # other domains contain domain-shared only
    domains[1:] = [domain_shared.copy() for i in range(1, num_train_domains)]

    logging.info(f"domains[0]: {domains[0]}")
    logging.info(f"domains[1:]: {domains[1:]}")

    return domains


def create_domains_1(
    num_classes: int, num_linked: int, num_domains: int
) -> List[List[int]]:
    """
    Args:
        num_classes: number of overall classes within all the domains
        num_linked: number of classes that are linked to an individual domain
        num_domains: number of different domains, including test domain
    """
    assert num_linked < num_classes
    domain_shared = [i for i in range(num_linked, num_classes)]
    print(f"Shared classes: {domain_shared}")
    num_train_domains = num_domains - 1
    domains = [domain_shared.copy() for i in range(num_train_domains)]

    for class_idx in range(num_linked):
        domain_idx = class_idx % num_train_domains
        domains[domain_idx].append(class_idx)
    return domains


def create_domains_2(
    num_classes: int, num_linked_ratio: float, num_domains: int
) -> List[List[int]]:
    """
    Args:
        num_classes: number of overall classes within all the domains
        num_linked_ratio: ratio of linked classes to the total number of classes
        num_domains: number of different domains, including test domain
    """
    num_linked = math.floor(num_linked_ratio * num_classes)
    return create_domains_1(num_classes, num_linked, num_domains)



# Testing
create_domains(num_classes=10, num_linked=5, num_train_domains=3)
# Testing domain linked only
create_domains(num_classes=10, num_linked=10, num_train_domains=3)

# Testing same classes with different overlap
domains = create_domains_1(num_classes=10, num_linked=3, num_domains=4)
print(domains)
domains = create_domains_1(num_classes=10, num_linked=5, num_domains=4)
print(domains)

# Testing different classes with same overlap percentage
domains = create_domains_2(num_classes=10, num_linked_ratio=0.2, num_domains=4)
print(domains)
domains = create_domains_2(num_classes=5, num_linked_ratio=0.2, num_domains=4)
print(domains)


Shared classes: [3, 4, 5, 6, 7, 8, 9]
[[3, 4, 5, 6, 7, 8, 9, 0], [3, 4, 5, 6, 7, 8, 9, 1], [3, 4, 5, 6, 7, 8, 9, 2]]
Shared classes: [5, 6, 7, 8, 9]
[[5, 6, 7, 8, 9, 0, 3], [5, 6, 7, 8, 9, 1, 4], [5, 6, 7, 8, 9, 2]]
Shared classes: [2, 3, 4, 5, 6, 7, 8, 9]
[[2, 3, 4, 5, 6, 7, 8, 9, 0], [2, 3, 4, 5, 6, 7, 8, 9, 1], [2, 3, 4, 5, 6, 7, 8, 9]]
Shared classes: [1, 2, 3, 4]
[[1, 2, 3, 4, 0], [1, 2, 3, 4], [1, 2, 3, 4]]


## Hparams

In [4]:
"""
Hyper-parameter registry
"""

def seed_hash(*args):
    """
    Derive an integer hash from all args, for use as a random seed.
    """
    args_str = str(args)
    return int(hashlib.md5(args_str.encode("utf-8")).hexdigest(), 16) % (2**31)


def _define_hparam(hparams, hparam_name, default_val, random_val_fn):
    hparams[hparam_name] = (hparams, hparam_name, default_val, random_val_fn)


def _hparams(algorithm, dataset, random_seed):
    """
    Global registry of hyperparams. Each entry is a (default, random) tuple.
    New algorithms / networks / etc. should add entries here.
    """
    SMALL_IMAGES = ["Debug28", "RotatedMNIST", "ColoredMNIST"]

    hparams = {}

    def _hparam(name, default_val, random_val_fn):
        """Define a hyperparameter. random_val_fn takes a RandomState and
        returns a random hyperparameter value."""
        assert name not in hparams
        random_state = np.random.RandomState(seed_hash(random_seed, name))
        hparams[name] = (default_val, random_val_fn(random_state))

    # Unconditional hparam definitions.
    _hparam("data_augmentation", True, lambda r: True)
    _hparam("resnet18", True, lambda r: True)
    _hparam("resnet_dropout", 0.0, lambda r: r.choice([0.0, 0.1, 0.5]))
    _hparam("class_balanced", False, lambda r: False)
    _hparam("nonlinear_classifier", False, lambda r: False)

    # Algorithm-specific hparam definitions. Each block of code below
    # corresponds to exactly one algorithm.

    if (
        algorithm == "FOND"
        or algorithm == "FOND_NC"
        or algorithm == "FOND_N"
        or algorithm == "NOC"
        or algorithm == "FOND_CDSA"  # <--- ADD THIS LINE
        or algorithm == "FOND_DANN"    
    ):
        _hparam("temperature", 0.07, lambda r: 0.07 * r.uniform(0.75, 1.25))
        _hparam("base_temperature", 0.07, lambda r: 0.07)
        _hparam("xdom_lmbd", 1, lambda r: 10 ** r.uniform(-1, 3))
        _hparam("error_lmbd", 1, lambda r: 10 ** r.uniform(-1, 3))
        _hparam("xda_alpha", 1, lambda r: 10 ** r.uniform(0, 1))
        _hparam("xda_beta", 1, lambda r: 10 ** r.uniform(0, 1))
        
        # CDSA-specific hyperparameters (only for FOND_CDSA)
        if algorithm == "FOND_CDSA":
            _hparam("cdsa_lambda", 0.5, lambda r: r.uniform(0.3, 0.7))
            _hparam("cdsa_lambda_0", 0.3, lambda r: r.uniform(0.1, 0.5))
            _hparam("cdsa_y_l_multiplier", 2.0, lambda r: r.uniform(1.5, 3.0))

    elif (
        algorithm == "FOND_Distillation_Separate_Projector"
        or algorithm == "FOND_Distillation_Teacher_Projector"
        or algorithm == "FOND_Distillation_Student_Projector"
    ):
        _hparam("temperature", 0.07, lambda r: 0.07 * r.uniform(0.75, 1.25))
        _hparam("base_temperature", 0.07, lambda r: 0.07)
        _hparam("xdom_lmbd", 1, lambda r: 10 ** r.uniform(-1, 3))
        _hparam("error_lmbd", 1, lambda r: 10 ** r.uniform(-1, 3))
        _hparam("xda_alpha", 1, lambda r: 10 ** r.uniform(0, 1))
        _hparam("xda_beta", 1, lambda r: 10 ** r.uniform(0, 1))
        _hparam("distillation_temperature", 1, lambda r: r.uniform(1, 20))

    # Dataset-and-algorithm-specific hparam definitions. Each block of code
    # below corresponds to exactly one hparam. Avoid nested conditionals.

    if dataset in SMALL_IMAGES:
        _hparam("lr", 1e-3, lambda r: 10 ** r.uniform(-4.5, -2.5))
    else:
        _hparam("lr", 5e-5, lambda r: 10 ** r.uniform(-5, -3.5))

    if dataset in SMALL_IMAGES:
        _hparam("weight_decay", 0.0, lambda r: 0.0)
    else:
        _hparam("weight_decay", 0.0, lambda r: 10 ** r.uniform(-6, -2))

    if dataset in SMALL_IMAGES:
        _hparam("batch_size", 64, lambda r: int(2 ** r.uniform(3, 9)))
    elif algorithm == "ARM":
        _hparam("batch_size", 8, lambda r: 8)
    elif dataset == "DomainNet":
        _hparam("batch_size", 32, lambda r: int(2 ** r.uniform(3, 5)))
    else:
        _hparam("batch_size", 32, lambda r: int(2 ** r.uniform(3, 5.5)))

    return hparams


def default_hparams(algorithm, dataset):
    return {a: b for a, (b, c) in _hparams(algorithm, dataset, 0).items()}


def random_hparams(algorithm, dataset, seed):
    return {a: c for a, (b, c) in _hparams(algorithm, dataset, seed).items()}

## Misc

In [5]:

class _SplitDataset(torch.utils.data.Dataset):
    """Used by split_dataset"""

    def __init__(self, underlying_dataset, keys):
        super(_SplitDataset, self).__init__()
        self.underlying_dataset = underlying_dataset
        self.keys = keys

    def __getitem__(self, key):
        return self.underlying_dataset[self.keys[key]]

    def __len__(self):
        return len(self.keys)


def split_dataset(dataset, n, seed=0):
    """
    Return a pair of datasets corresponding to a random split of the given
    dataset, with n datapoints in the first dataset and the rest in the last,
    using the given random seed
    """
    assert n <= len(dataset)
    keys = list(range(len(dataset)))
    np.random.RandomState(seed).shuffle(keys)
    keys_1 = keys[:n]
    keys_2 = keys[n:]
    return _SplitDataset(dataset, keys_1), _SplitDataset(dataset, keys_2)


def seed_hash(*args):
    """
    Derive an integer hash from all args, for use as a random seed.
    """
    args_str = str(args)
    return int(hashlib.md5(args_str.encode("utf-8")).hexdigest(), 16) % (2**31)


def make_weights_for_balanced_classes(dataset):
    counts = Counter()
    classes = []
    for _, y in dataset:
        y = int(y)
        counts[y] += 1
        classes.append(y)

    n_classes = len(counts)

    weight_per_class = {}
    for y in counts:
        weight_per_class[y] = 1 / (counts[y] * n_classes)

    weights = torch.zeros(len(dataset))
    for i, y in enumerate(classes):
        weights[i] = weight_per_class[int(y)]

    return weights
def compute_loss(network, loader, device):
    """
    Compute average loss for a given loader.
    Returns: average loss value
    """
    total_loss = 0.0
    num_batches = 0
    
    network.eval()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            p = network.predict(x)
            
            # Compute loss
            loss = torch.nn.functional.cross_entropy(p, y, reduction='mean')
            total_loss += loss.item()
            num_batches += 1
    
    network.train()
    
    avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
    return avg_loss

def accuracy(network, loader, weights, device, dataset):
    correct = 0
    total = 0
    weights_offset = 0
    overlapping_classes = dataset.overlapping_classes
    num_classes = dataset.num_classes
    
    f1_score = torchmetrics.F1Score(
        task="multiclass", num_classes=num_classes, average="macro"
    ).to(device)
    per_class_accuracy = torchmetrics.Accuracy(
        task="multiclass",
        num_classes=num_classes,
        average=None,
    ).to(device)
    accuracy = torchmetrics.Accuracy(
        task="multiclass",
        num_classes=num_classes,
        average="macro",
    ).to(device)
    recall = torchmetrics.Recall(
        task="multiclass",
        num_classes=num_classes,
        average="macro",
    ).to(device)
    precision = torchmetrics.Precision(
        task='multiclass',  # ✅ Fixed typo
        num_classes=num_classes,
        average='macro'
    ).to(device)
    conf_mat = torchmetrics.ConfusionMatrix(
        task="multiclass",
        num_classes=num_classes
    ).to(device)  # ✅ Added .to(device)
    
    network.eval()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            p = network.predict(x)
            
            if weights is None:
                batch_weights = torch.ones(len(x))
            else:
                batch_weights = weights[weights_offset : weights_offset + len(x)]
                weights_offset += len(x)
            batch_weights = batch_weights.to(device)
            
            if p.size(1) == 1:
                correct += (
                    (p.gt(0).eq(y).float() * batch_weights.view(-1, 1)).sum().item()
                )
            else:
                correct += (p.argmax(1).eq(y).float() * batch_weights).sum().item()
            total += batch_weights.sum().item()
            
            # update metrics
            accuracy.update(p, y)
            recall.update(p, y)
            precision.update(p, y)
            f1_score.update(p, y)
            per_class_accuracy.update(p, y)
            conf_mat.update(p, y)
    
    network.train()
    compute_acc = accuracy.compute().item()
    compute_recall = recall.compute().item()
    compute_precision = precision.compute().item()  # ✅ Fixed typo
    compute_f1 = f1_score.compute().item()
    compute_per_class_acc = per_class_accuracy.compute().cpu().numpy()
    cm_compute = conf_mat.compute().cpu().numpy()  # ✅ Convert to numpy
    
    overlap_class_acc = []
    non_overlap_class_acc = []
    per_class_acc_dict = {}
    for i in range(num_classes):
        per_class_acc_dict[i] = float(compute_per_class_acc[i])
        if i in overlapping_classes:
            overlap_class_acc.append(compute_per_class_acc[i])
        else:
            non_overlap_class_acc.append(compute_per_class_acc[i])
    
    if len(non_overlap_class_acc) == 0:
        non_overlap_class_acc = -1
    else:
        non_overlap_class_acc = np.mean(non_overlap_class_acc)
    if len(overlap_class_acc) == 0:
        overlap_class_acc = -1
    else:
        overlap_class_acc = np.mean(overlap_class_acc)
    
    other_acc = correct / total
    
    return (
        float(compute_acc),
        float(compute_recall),
        float(compute_f1),
        float(compute_precision),      # ✅ Moved before oacc/nacc
        float(overlap_class_acc),
        float(non_overlap_class_acc),
        per_class_acc_dict,
        cm_compute
    )
class _InfiniteSampler(torch.utils.data.Sampler):
    """Wraps another Sampler to yield an infinite stream."""

    def __init__(self, sampler):
        self.sampler = sampler

    def __iter__(self):
        while True:
            for batch in self.sampler:
                yield batch


class InfiniteDataLoader:
    def __init__(self, dataset, weights, batch_size, num_workers):
        super().__init__()

        if weights is not None:
            sampler = torch.utils.data.WeightedRandomSampler(
                weights, replacement=True, num_samples=batch_size
            )
        else:
            sampler = torch.utils.data.RandomSampler(dataset, replacement=True)

        if weights == None:
            weights = torch.ones(len(dataset))

        batch_sampler = torch.utils.data.BatchSampler(
            sampler, batch_size=batch_size, drop_last=True
        )

        self._infinite_iterator = iter(
            torch.utils.data.DataLoader(
                dataset,
                num_workers=num_workers,
                batch_sampler=_InfiniteSampler(batch_sampler),
            )
        )

    def __iter__(self):
        while True:
            yield next(self._infinite_iterator)

    def __len__(self):
        raise ValueError


class FastDataLoader:
    """DataLoader wrapper with slightly improved speed by not respawning worker
    processes at every epoch."""

    def __init__(self, dataset, batch_size, num_workers):
        super().__init__()

        batch_sampler = torch.utils.data.BatchSampler(
            torch.utils.data.RandomSampler(dataset, replacement=False),
            batch_size=batch_size,
            drop_last=False,
        )

        self._infinite_iterator = iter(
            torch.utils.data.DataLoader(
                dataset,
                num_workers=num_workers,
                batch_sampler=_InfiniteSampler(batch_sampler),
            )
        )

        self._length = len(batch_sampler)

    def __iter__(self):
        for _ in range(len(self)):
            yield next(self._infinite_iterator)

    def __len__(self):
        return self._length


def config_logging():
    """
    Reusable code for formatting the logger
    """
    logging.basicConfig(
        format="%(asctime)s,%(msecs)03d %(levelname)-8s [%(filename)s:%(funcName)s:%(lineno)d] %(message)s",
        datefmt="%Y-%m-%d:%H:%M:%S",
        level=logging.INFO,
    )


## Simple Logger

In [6]:
"""
Simple CSV logger to replace W&B during development
Keeps the same interface for easy swap later
"""
class CSVLogger:
    """Lightweight logger that writes metrics to CSV"""
    
    def __init__(self, csv_path: str, root_dir:str=None):
        self.csv_path = Path(csv_path)
        self.csv_path.parent.mkdir(parents=True, exist_ok=True)
        self.fieldnames = None
        self.file = None
        self.writer = None
        self._init_csv()
        self.root = root_dir if root_dir is not None else csv_path.parent
    def _init_csv(self):
        """Initialize CSV file with headers"""
        self.file = open(self.csv_path, 'w', newline='')
        self.writer = None  # Will be created on first log
    def return_root(self):
        return self.root
    def log(self, metrics: Dict[str, Any], step: Optional[int] = None):
        """
        Log metrics to CSV
        
        Args:
            metrics: Dictionary of metric_name -> value
            step: Training step (optional, will be added if provided)
        """
        if step is not None:
            metrics = {"step": step, **metrics}
            
        # Flatten nested dictionaries (e.g., {"train/acc": 0.9})
        flat_metrics = {}
        for key, value in metrics.items():
            if isinstance(value, dict):
                for subkey, subvalue in value.items():
                    flat_metrics[f"{key}/{subkey}"] = subvalue
            else:
                flat_metrics[key] = value
        
        # Initialize writer with fieldnames on first call
        if self.writer is None:
            self.fieldnames = list(flat_metrics.keys())
            self.writer = csv.DictWriter(self.file, fieldnames=self.fieldnames)
            self.writer.writeheader()
            
        # Add new fields if they appear
        new_fields = set(flat_metrics.keys()) - set(self.fieldnames)
        if new_fields:
            self.fieldnames.extend(sorted(new_fields))
            # Rewrite file with new headers
            self.file.close()
            self._rewrite_with_new_fields(flat_metrics)
            return
            
        self.writer.writerow(flat_metrics)
        self.file.flush()  # Ensure immediate write
        
    def _rewrite_with_new_fields(self, new_row: Dict[str, Any]):
        """Rewrite CSV when new fields are discovered"""
        # Read existing rows
        with open(self.csv_path, 'r') as f:
            reader = csv.DictReader(f)
            existing_rows = list(reader)
        
        # Rewrite with updated fieldnames
        self.file = open(self.csv_path, 'w', newline='')
        self.writer = csv.DictWriter(self.file, fieldnames=self.fieldnames)
        self.writer.writeheader()
        for row in existing_rows:
            self.writer.writerow(row)
        self.writer.writerow(new_row)
        self.file.flush()
        
    def save_config(self, config: Dict[str, Any], path: Optional[str] = None):
        """Save experiment config as JSON"""
        if path is None:
            path = self.csv_path.parent / "config.json"
        with open(path, 'w') as f:
            json.dump(config, f, indent=2)
            
    def close(self):
        """Close CSV file"""
        if self.file:
            self.file.close()
            
    def __enter__(self):
        return self
        
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close()


class PrintLogger:
    """Even simpler logger that just prints to console"""
    
    def log(self, metrics: Dict[str, Any], step: Optional[int] = None):
        step_str = f"[Step {step}] " if step is not None else ""
        metric_str = ", ".join(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}" 
                               for k, v in metrics.items())
        print(f"{step_str}{metric_str}")
        
    def save_config(self, config: Dict[str, Any], path: Optional[str] = None):
        print(f"Config: {json.dumps(config, indent=2)}")
        
    def close(self):
        pass


# RandAugment

## AugmenterBase

In [7]:
"""
This file contains base class for augmenting patches from whole slide images.
"""


#----------------------------------------------------------------------------------------------------

class AugmenterBase(object):
    """Base class for patch augmentation."""

    def __init__(self, keyword):
        """
        Initialize the object.

        Args:
            keyword (str): Short name for the transformation.
        """

        # Initialize the base class.
        #
        super().__init__()

        # Initialize members.
        #
        self.__keyword = keyword

    @property
    def keyword(self):
        """
        Get the keyword for the augmenter.

        Returns:
            str: Keyword.
        """

        return self.__keyword

    def shapes(self, target_shapes):
        """
        Calculate the required shape of the input to achieve the target output shape.

        Args:
            target_shapes (dict): Target output shape per level.

        Returns:
            (dict): Required input shape per level.
        """

        # By default the output shapes match the input shapes.
        #
        return target_shapes

    def transform(self, patch):
        """
        Transform the given patch.

        Args:
            patch (np.ndarray): Patch to transform.

        Returns:
            np.ndarray: Transformed patch.
        """

        pass

    def randomize(self):
        """Randomize the parameters of the augmenter."""

        pass

## AugmenterPool

In [8]:
"""
This file contains a class for handling and applying multiple augmentations on patches from whole slide images.
"""


#----------------------------------------------------------------------------------------------------

class AugmenterPool(object):
    """Class for augmenting patches from whole slide images."""

    def __init__(self):
        """Initialize object."""

        # Initialize the base class.
        #
        super().__init__()

        # Initialize members.
        #
        self.__groups = []          # List of augmenter groups in order of application.
        self.__augmenters = {}      # List of augmenter object by groups.
        self.__probabilities = {}   # Probability of selection for each augmenter.
        self.__current_set = {}     # Index of augmenters that currently selected for execution.
        self.__distributed = False  # Flag to indicate if proper group probabilities have been pre-calculated.

    def __cropbatch(self, patches, labels, shape):
        """
        Crop the batch of patches to the target shape. For efficiency considerations this is a separate function from the crop() public function.

        Args:
            patches (np.ndarray): Image patches to crop.
            labels (np.ndarray): Label images to crop.
            shape (tuple): Target shape.

        Returns:
            (np.ndarray): Cropped batch.

        Raises:
            BatchCroppingError: The target shape for cropping is smaller than the batch to crop itself.
        """

        # Check quickly if cropping is needed at all.
        #
        if patches.shape[2:] == shape:
            # Return the original batch.
            #
            return patches, labels

        else:
            # Check if the shape is larger or equal to the target.
           
            shift = ((patches.shape[2] - shape[0]) // 2, (patches.shape[3] - shape[1]) // 2)

            # Return the cropped patch.
            #
            return patches[:, :, shift[0]: shift[0] + shape[0], shift[1]: shift[1] + shape[1]], labels[:, :, shift[0]: shift[0] + shape[0], shift[1]: shift[1] + shape[1]] if 2 < labels.ndim else labels

    def appendgroup(self, group, randomized):
        """
        Append augmentation group. If the created group is randomized the augmenter pool will choose one augmenter from the pool for execution, otherwise it is sequential and the augmenter pool
        will execute all the augmenters in the group in order.

        Args:
            group (str): Group identifier.
            randomized (bool): Flag indicating group type.

        Raises:
            AugmentationGroupAlreadyExistsError: Augmentation group already exists.
        """

        # Check if group exists already.
        #
        
        # Set up the per-group data.
        #
        self.__groups.append(group)
        self.__augmenters[group] = []

        if randomized:
            self.__probabilities[group] = []

    def appendaugmenter(self, augmenter, group, ratio=0.0):
        """
        Append an augmenter object to the group.

        Args:
            augmenter (dptaugmenterbase.AugmenterBase): Augmenter object.
            group (str): Group identifier.
            ratio (float): Ratio for selection.

        Raises:
            UnknownAugmentationGroupError: Unknown augmentation group.
            InvalidAugmentationRatioError: The ratio of the augmentation selection in its group is invalid.
        """

        # Check if the group is known.
      
        # Check the ratio for selection. It must be larger than 0.0 if the group is randomized.
        #
       
        # Append the augmenter object and its relative selection probability to the per-group list.
        #
        self.__augmenters[group].append(augmenter)

        if group in self.__probabilities:
            self.__probabilities[group].append(ratio)

            # Reset the distributed flag, only if the affected group is randomized.
            #
            self.__distributed = False

    def distribute(self):
        """Calculate the distribution of the augmenters based on their relative probability."""

        # Normalize the probability of selection to 1.0 sum in the groups.
        #
        for group_id in self.__groups:
            if group_id in self.__probabilities:
                group_p_sum = sum(self.__probabilities[group_id])
                for index in range(len(self.__probabilities[group_id])):
                    self.__probabilities[group_id][index] /= group_p_sum

        # Mark the flag.
        #
        self.__distributed = True

    def shapes(self, target_shapes):
        """
        Calculate the required shape of the input to achieve the target output shape.

        Args:
            target_shapes (dict): Target output shape per level.

        Returns:
            (dict): Required input shape per level.
        """

        # Copy the target sizes.
        #
        required_shapes = target_shapes.copy()

        # Get the maximal required sizes.
        #
        for group_id in self.__augmenters:
            for augmenter_item in self.__augmenters[group_id]:
                required_shapes_for_item = augmenter_item.shapes(target_shapes)

                for level in required_shapes:
                    required_shapes[level] = tuple(max(required_shapes[level][index], required_shapes_for_item[level][index]) for index in range(len(required_shapes[level])))

        return required_shapes

    def transform(self, patch, label=None):
        """
        Randomly select one from each group and apply transformations on the patch and the label map in order.

        Args:
            patch (np.ndarray): Patch to transform.
            label (np.ndarray, None): Patch labels to transform.

        Returns:
            np.ndarray, (np.ndarray, None): Transformed patch, transformed labels.

        Raises:
            MissingAugmentationRandomizationError: Augmentations are configured but not randomized.
        """

        # Check if randomization is done.
        #
       
        # Prepare the result.
        #
        transformed_patch = patch
        transformed_label = label

        # Go through the groups in order.
        #
        for group_id in self.__groups:
            if group_id in self.__probabilities:
                # Apply transformation on the patch and if there is a label image and the transformation is spatial apply it to the label image too.
                #
                current_augmenter = self.__augmenters[group_id][self.__current_set[group_id]]
                transformed_patch = current_augmenter.transform(patch=transformed_patch)

                if transformed_label is not None and transformed_label.ndim == 3 and isinstance(current_augmenter, dptspatialaugmenterbase.SpatialAugmenterBase):
                    transformed_label = current_augmenter.transform(patch=transformed_label)
            else:
                # This a sequential group. Apply all the augmenters in order.
                #
                for augmenter_item in self.__augmenters[group_id]:
                    transformed_patch = augmenter_item.transform(patch=transformed_patch)

                    if transformed_label is not None and transformed_label.ndim == 3 and isinstance(augmenter_item, dptspatialaugmenterbase.SpatialAugmenterBase):
                        transformed_label = augmenter_item.transform(patch=transformed_label)

        # Return the result patch, label pair.
        #
        return transformed_patch, transformed_label

    def randomize(self):
        """
        Randomize the parameters of the augmenters.

        Raises:
            AugmentationPoolRandomizationBeforeDistribution: The augmentation pool randomized before distribution.
            EmptyAugmentationGroupError: Empty augmentation group.
        """

        # Proper distribution calculation is necessary before randomization.
       
        # Check if there is an empty group.
        #
        if any(len(self.__augmenters[group_id]) == 0 for group_id in self.__groups):
            empty_groups = [group_id for group_id in self.__groups if len(self.__augmenters[group_id]) == 0]
            
        for group_id in self.__groups:
            if group_id in self.__probabilities:
                # Randomly select an augmenter based on the configured probability and randomize its parameters.
                #
                augmentation_index = np.random.choice(a=len(self.__probabilities[group_id]), size=None, p=self.__probabilities[group_id])

                self.__current_set[group_id] = augmentation_index
                self.__augmenters[group_id][augmentation_index].randomize()
            else:
                # This is a sequential group. Randomize the parameters of all the augmenters.
                #
                for augmenter_item in self.__augmenters[group_id]:
                    augmenter_item.randomize()

    def crop(self, patch, shape):
        """
        Crop the patch to the target shape.

        Args:
            patch (np.ndarray): Patch to crop.
            shape (tuple): Target shape.

        Returns:
            (np.ndarray): Cropped patch.

        Raises:
            PatchCroppingError: The target shape for cropping is smaller than the patch to crop itself.
        """

        # Check quickly if cropping is needed at all.
        #
        if patch.shape[1:] == shape:
            # Return the original patch.
            #
            return patch

        else:
            # Check if the patch shape is larger or equal to the target.
            #
            
            # Calculate the shift.
            #
            shift = ((patch.shape[1] - shape[0]) // 2, (patch.shape[2] - shape[1]) // 2)

            # Return the cropped patch.
            #
            return patch[:, shift[0]: shift[0] + shape[0], shift[1]: shift[1] + shape[1]]

    def process(self, patches, shapes=None, randomize=True):
        """
        Process a batch of multi-level patches.

        Args:
            patches (dict): RGB patches and labels per level as given by the PatchSampler.
            randomize (flag to control if parameters should be randomized before each patch augmentation.
            shapes (dict, None): Target patch shapes (rows, columns) per level.

        Returns:
            dict: Augmented patch collection.

        Raises:
            MissingTargetShapeForLevelError: Target shape for cropping is missing for a level.
            MissingAugmentationRandomizationError: Augmentations are configured but not randomized.
            AugmentationPoolRandomizationBeforeDistribution: The augmentation pool randomized before distribution.
            EmptyAugmentationGroupError: Empty augmentation group.
            BatchCroppingError: The target shape for cropping is smaller than the batch to crop itself.
        """

        # Check if the target shapes are valid.
        #
       
        # Get patch collection length.
        #
        patch_count = next(iter(patches.values()))['patches'].shape[0]

        # Go through all the patches: randomize the augmenters and apply the same augmentation across all the levels for the same patch.
        #
        for index in range(patch_count):
            if randomize:
                self.randomize()

            for level in patches:
                patches[level]['patches'][index], patches[level]['labels'][index] = self.transform(patch=patches[level]['patches'][index], label=patches[level]['labels'][index])

        # Crop the central part of the patches to remove augmentation artifacts.
        #
        if shapes:
            for level in patches:
                patches[level]['patches'], patches[level]['labels'] = self.__cropbatch(patches=patches[level]['patches'], labels=patches[level]['labels'], shape=shapes[level])

        # Return the transformed patch set.
        #
        return patches

## PassThroughAugmenter

In [9]:
"""
This file contains a pass-through augmentation cass.
"""

class PassThroughAugmenter(AugmenterBase):
    """Pass through augmenter that does noting."""

    def __init__(self):
        """Initialize the object."""

        # Initialize the base class.
        #
        super().__init__(keyword='pass_through')

    def transform(self, patch):
        """
        Return the given patch without transformation.

        Args:
            patch (np.ndarray): Patch to return.

        Returns:
            np.ndarray: The patch.
        """

        return patch

## Colors: Utils

### copy_files

### Data Generator

### data handler

### ColorAugmenterBase

In [10]:
class ColorAugmenterBase(AugmenterBase):
    """Base class for color patch augmentation."""

    def __init__(self, keyword):
        """
        Initialize the object.

        Args:
            keyword (str): Short name for the transformation.
        """

        # Initialize the base class.
        #
        super().__init__(keyword=keyword)

### ContrastAugmenter

In [11]:
"""
This file contains a class for augmenting patches from whole slide images with contrast changes.
"""



#----------------------------------------------------------------------------------------------------

class ContrastAugmenter(ColorAugmenterBase):
    """Apply contrast enhancements on the patch."""

    def __init__(self, sigma_range):
        """
        Initialize the object.

        Args:
            sigma_range (tuple): Range for contrast adjustment from the [-1.0, 1.0] range. For example: (-0.4, 0.4).

        Raises:
            InvalidContrastSigmaRangeError: The contrast adjustment range is not valid.
        """

        # Initialize base class.
        #
        super().__init__(keyword='contrast')

        # Initialize members.
        #
        self.__sigma_range = None  # Configured sigma range for contrast enhancement.
        self.__sigma = None        # Randomized sigma.

        # Save configuration.
        #
        self.__setsigmarange(sigma_range=sigma_range)

    def __setsigmarange(self, sigma_range):
        """
        Set the interval.

        Args:
            sigma_range (tuple): Range for contrast adjustment.

        Raises:
            InvalidContrastSigmaRangeError: The contrast adjustment range is not valid.
        """

        # Check the interval.
        #
       
        # Store the settings.
        #
        self.__sigma_range = list(sigma_range)
        self.__sigma = sigma_range[0]

    def transform(self, patch):
        """
        Apply contrast deformation on the patch.

        Args:
            patch (np.ndarray): Patch to transform.

        Returns:
            np.ndarray: Transformed patch.
        """

        # Reorder the patch to channel last format.
        #
        patch_image = np.transpose(a=patch, axes=(1, 2, 0))

        # Augment the contrast.
        #
        patch_center = skimage.color.rgb2gray(rgb=patch_image).mean() * 255.0
        patch_range = (self.__sigma * patch_center, 255.0 - self.__sigma * (255.0 - patch_center))
        patch_contrast = skimage.exposure.rescale_intensity(image=patch_image, in_range=patch_range, out_range='dtype')

        # Order back to channels first order.
        #
        patch_transformed = np.transpose(a=patch_contrast, axes=(2, 0, 1))

        return patch_transformed

    def randomize(self):
        """Randomize the parameters of the augmenter."""

        # Randomize sigma.
        #
        self.__sigma = np.random.uniform(low=self.__sigma_range[0], high=self.__sigma_range[1], size=None)

### custom_hed_transform

In [12]:

rgb_from_hed = np.array([[0.65, 0.70, 0.29],
                         [0.07, 0.99, 0.11],
                         [0.27, 0.57, 0.78]]).astype('float32')
hed_from_rgb = linalg.inv(rgb_from_hed).astype('float32')


def rgb2hed(rgb):

    return separate_stains(rgb, hed_from_rgb)

def hed2rgb(hed):

    return combine_stains(hed, rgb_from_hed)

def separate_stains(rgb, conv_matrix):

    # # t = time.time()
    # rgb = dtype.img_as_float(rgb, force_copy=True).astype('float32')
    # # print('{f} took {s} s'.format(f='separate img_as_float', s=(time.time() - t)), flush=True)
    # # print('rgb type is {r}, matrix type is {m}'.format(r=rgb.dtype, m=conv_matrix.dtype), flush=True)
    #
    # # t = time.time()
    # rgb = -np.log(rgb)
    # # print('{f} took {s} s'.format(f='separate np.log', s=(time.time() - t)), flush=True)
    #
    # # t = time.time()
    # rgb += 2
    # rgb = np.reshape(rgb, (-1, 3))
    # # print('{f} took {s} s'.format(f='separate add reshape', s=(time.time() - t)), flush=True)
    #
    # # print('x shape is {s}, conv_matrix shape is {c}'.format(s=x.shape, c=conv_matrix.shape))
    #
    # # t = time.time()
    # stains = np.dot(rgb, conv_matrix)
    # # print('{f} took {s} s'.format(f='separate np.dot', s=(time.time() - t)), flush=True)
    #
    # return np.reshape(stains, rgb.shape)

    rgb = dtype.img_as_float(rgb, force_copy=True).astype('float32')
    rgb += 2
    stains = np.dot(np.reshape(-np.log(rgb), (-1, 3)), conv_matrix)
    return np.reshape(stains, rgb.shape)


def combine_stains(stains, conv_matrix):

    # # t = time.time()
    # stains = dtype.img_as_float(stains).astype('float32')
    # # stains = stains.astype('float32')
    # # conv_matrix = conv_matrix.astype('float32')
    # # print('{f} took {s} s'.format(f='separate img_as_float', s=(time.time() - t)), flush=True)
    # # print('stains type is {r}, matrix type is {m}'.format(r=stains.dtype, m=conv_matrix.dtype), flush=True)
    #
    # stains = -np.reshape(stains, (-1, 3))
    # # print('x shape is {s}, conv_matrix shape is {c}'.format(s=x.shape, c=conv_matrix.shape))
    #
    # # t = time.time()
    # logrgb2 = np.dot(stains, conv_matrix)
    # # print('{f} took {s} s'.format(f='combine np.dot', s=(time.time() - t)), flush=True)
    #
    # # t = time.time()
    # rgb2 = np.exp(logrgb2)
    # # print('{f} took {s} s'.format(f='combine np.exp', s=(time.time() - t)), flush=True)
    #
    # t = time.time()
    # x = rescale_intensity(np.reshape(rgb2 - 2, stains.shape),
    #                          in_range=(-1, 1))
    # print('{f} took {s} s'.format(f='combine rescale_intensity', s=(time.time() - t)), flush=True)
    #
    # return x

    stains = dtype.img_as_float(stains.astype('float64')).astype('float32')  # stains are out of range [-1, 1] so dtype.img_as_float complains if not float64
    logrgb2 = np.dot(-np.reshape(stains, (-1, 3)), conv_matrix)
    rgb2 = np.exp(logrgb2)
    return rescale_intensity(np.reshape(rgb2 - 2, stains.shape),
                             in_range=(-1, 1))

### hedcoloraugmenter

In [13]:
"""
This file contains a class for augmenting patches from whole slide images by applying color correction in HED color space.
"""

rgb_from_hed = np.array([[0.65, 0.70, 0.29],
                         [0.07, 0.99, 0.11],
                         [0.27, 0.57, 0.78]]).astype('float32')
hed_from_rgb = linalg.inv(rgb_from_hed).astype('float32')


def rgb2hed(rgb):

    return separate_stains(rgb, hed_from_rgb)

def hed2rgb(hed):

    return combine_stains(hed, rgb_from_hed)

def separate_stains(rgb, conv_matrix):

    # # t = time.time()
    # rgb = dtype.img_as_float(rgb, force_copy=True).astype('float32')
    # # print('{f} took {s} s'.format(f='separate img_as_float', s=(time.time() - t)), flush=True)
    # # print('rgb type is {r}, matrix type is {m}'.format(r=rgb.dtype, m=conv_matrix.dtype), flush=True)
    #
    # # t = time.time()
    # rgb = -np.log(rgb)
    # # print('{f} took {s} s'.format(f='separate np.log', s=(time.time() - t)), flush=True)
    #
    # # t = time.time()
    # rgb += 2
    # rgb = np.reshape(rgb, (-1, 3))
    # # print('{f} took {s} s'.format(f='separate add reshape', s=(time.time() - t)), flush=True)
    #
    # # print('x shape is {s}, conv_matrix shape is {c}'.format(s=x.shape, c=conv_matrix.shape))
    #
    # # t = time.time()
    # stains = np.dot(rgb, conv_matrix)
    # # print('{f} took {s} s'.format(f='separate np.dot', s=(time.time() - t)), flush=True)
    #
    # return np.reshape(stains, rgb.shape)

    rgb = dtype.img_as_float(rgb, force_copy=True).astype('float32')
    rgb += 2
    stains = np.dot(np.reshape(-np.log(rgb), (-1, 3)), conv_matrix)
    return np.reshape(stains, rgb.shape)


def combine_stains(stains, conv_matrix):

    # # t = time.time()
    # stains = dtype.img_as_float(stains).astype('float32')
    # # stains = stains.astype('float32')
    # # conv_matrix = conv_matrix.astype('float32')
    # # print('{f} took {s} s'.format(f='separate img_as_float', s=(time.time() - t)), flush=True)
    # # print('stains type is {r}, matrix type is {m}'.format(r=stains.dtype, m=conv_matrix.dtype), flush=True)
    #
    # stains = -np.reshape(stains, (-1, 3))
    # # print('x shape is {s}, conv_matrix shape is {c}'.format(s=x.shape, c=conv_matrix.shape))
    #
    # # t = time.time()
    # logrgb2 = np.dot(stains, conv_matrix)
    # # print('{f} took {s} s'.format(f='combine np.dot', s=(time.time() - t)), flush=True)
    #
    # # t = time.time()
    # rgb2 = np.exp(logrgb2)
    # # print('{f} took {s} s'.format(f='combine np.exp', s=(time.time() - t)), flush=True)
    #
    # t = time.time()
    # x = rescale_intensity(np.reshape(rgb2 - 2, stains.shape),
    #                          in_range=(-1, 1))
    # print('{f} took {s} s'.format(f='combine rescale_intensity', s=(time.time() - t)), flush=True)
    #
    # return x

    stains = dtype.img_as_float(stains.astype('float64')).astype('float32')  # stains are out of range [-1, 1] so dtype.img_as_float complains if not float64
    logrgb2 = np.dot(-np.reshape(stains, (-1, 3)), conv_matrix)
    rgb2 = np.exp(logrgb2)
    return rescale_intensity(np.reshape(rgb2 - 2, stains.shape),
                             in_range=(-1, 1))

#----------------------------------------------------------------------------------------------------

class HedColorAugmenter(ColorAugmenterBase):
    """Apply color correction in HED color space on the RGB patch."""

    def __init__(self, haematoxylin_sigma_range, haematoxylin_bias_range, eosin_sigma_range, eosin_bias_range, dab_sigma_range, dab_bias_range, cutoff_range):
        """
        Initialize the object. For each channel the augmented value is calculated as value = value * sigma + bias

        Args:
            haematoxylin_sigma_range (tuple, None): Adjustment range for the Haematoxylin channel from the [-1.0, 1.0] range where 0.0 means no change. For example (-0.1, 0.1).
            haematoxylin_bias_range (tuple, None): Bias range for the Haematoxylin channel from the [-1.0, 1.0] range where 0.0 means no change. For example (-0.2, 0.2).
            eosin_sigma_range (tuple, None): Adjustment range for the Eosin channel from the [-1.0, 1.0] range where 0.0 means no change.
            eosin_bias_range (tuple, None) Bias range for the Eosin channel from the [-1.0, 1.0] range where 0.0 means no change.
            dab_sigma_range (tuple, None): Adjustment range for the DAB channel from the [-1.0, 1.0] range where 0.0 means no change.
            dab_bias_range (tuple, None): Bias range for the DAB channel from the [-1.0, 1.0] range where 0.0 means no change.
            cutoff_range (tuple, None): Patches with mean value outside the cutoff interval will not be augmented. Values from the [0.0, 1.0] range. The RGB channel values are from the same range.

        Raises:
            InvalidHaematoxylinSigmaRangeError: The sigma range for Haematoxylin channel adjustment is not valid.
            InvalidHaematoxylinBiasRangeError: The bias range for Haematoxylin channel adjustment is not valid.
            InvalidEosinSigmaRangeError: The sigma range for Eosin channel adjustment is not valid.
            InvalidEosinBiasRangeError: The bias range for Eosin channel adjustment is not valid.
            InvalidDabSigmaRangeError: The sigma range for DAB channel adjustment is not valid.
            InvalidDabBiasRangeError: The bias range for DAB channel adjustment is not valid.
            InvalidCutoffRangeError: The cutoff range is not valid.
        """

        # Initialize base class.
        #
        super().__init__(keyword='hed_color')

        # Initialize members.
        #
        self.__sigma_ranges = None  # Configured sigma ranges for H, E, and D channels.
        self.__bias_ranges = None   # Configured bias ranges for H, E, and D channels.
        self.__cutoff_range = None  # Cutoff interval.
        self.__sigmas = None        # Randomized sigmas for H, E, and D channels.
        self.__biases = None        # Randomized biases for H, E, and D channels.

        # Save configuration.
        #
        self.__setsigmaranges(haematoxylin_sigma_range=haematoxylin_sigma_range, eosin_sigma_range=eosin_sigma_range, dab_sigma_range=dab_sigma_range)
        self.__setbiasranges(haematoxylin_bias_range=haematoxylin_bias_range, eosin_bias_range=eosin_bias_range, dab_bias_range=dab_bias_range)
        self.__setcutoffrange(cutoff_range=cutoff_range)

    def __setsigmaranges(self, haematoxylin_sigma_range, eosin_sigma_range, dab_sigma_range):
        """
        Set the sigma intervals.

        Args:
            haematoxylin_sigma_range (tuple, None): Adjustment range for the Haematoxylin channel.
            eosin_sigma_range (tuple, None): Adjustment range for the Eosin channel.
            dab_sigma_range (tuple, None): Adjustment range for the DAB channel.

        Raises:
            InvalidHaematoxylinSigmaRangeError: The sigma range for Haematoxylin channel adjustment is not valid.
            InvalidEosinSigmaRangeError: The sigma range for Eosin channel adjustment is not valid.
            InvalidDabSigmaRangeError: The sigma range for DAB channel adjustment is not valid.
        """

        # Check the intervals.
        #
        '''
        if haematoxylin_sigma_range is not None:
            if len(haematoxylin_sigma_range) != 2 or haematoxylin_sigma_range[1] < haematoxylin_sigma_range[0] or haematoxylin_sigma_range[0] < -1.0 or 1.0 < haematoxylin_sigma_range[1]:
                raise dptaugmentationerrors.InvalidHaematoxylinSigmaRangeError(haematoxylin_sigma_range)

        if eosin_sigma_range is not None:
            if len(eosin_sigma_range) != 2 or eosin_sigma_range[1] < eosin_sigma_range[0] or eosin_sigma_range[0] < -1.0 or 1.0 < eosin_sigma_range[1]:
                raise dptaugmentationerrors.InvalidEosinSigmaRangeError(eosin_sigma_range)

        if dab_sigma_range is not None:
            if len(dab_sigma_range) != 2 or dab_sigma_range[1] < dab_sigma_range[0] or dab_sigma_range[0] < -1.0 or 1.0 < dab_sigma_range[1]:
                raise dptaugmentationerrors.InvalidDabSigmaRangeError(dab_sigma_range)
        '''
        # Store the settings.
        #
        self.__sigma_ranges = [haematoxylin_sigma_range, eosin_sigma_range, dab_sigma_range]

        self.__sigmas = [haematoxylin_sigma_range if haematoxylin_sigma_range is not None else 0.0,
                         eosin_sigma_range if eosin_sigma_range is not None else 0.0,
                         dab_sigma_range if dab_sigma_range is not None else 0.0]
        

    def __setbiasranges(self, haematoxylin_bias_range, eosin_bias_range, dab_bias_range):
        """
        Set the bias intervals.

        Args:
            haematoxylin_bias_range (tuple, None): Bias range for the Haematoxylin channel.
            eosin_bias_range (tuple, None) Bias range for the Eosin channel.
            dab_bias_range (tuple, None): Bias range for the DAB channel.

        Raises:
            InvalidHaematoxylinBiasRangeError: The bias range for Haematoxylin channel adjustment is not valid.
            InvalidEosinBiasRangeError: The bias range for Eosin channel adjustment is not valid.
            InvalidDabBiasRangeError: The bias range for DAB channel adjustment is not valid.
        """

        # Check the intervals.
        #
        '''
        if haematoxylin_bias_range is not None:
            if len(haematoxylin_bias_range) != 2 or haematoxylin_bias_range[1] < haematoxylin_bias_range[0] or haematoxylin_bias_range[0] < -1.0 or 1.0 < haematoxylin_bias_range[1]:
                raise dptaugmentationerrors.InvalidHaematoxylinBiasRangeError(haematoxylin_bias_range)

        if eosin_bias_range is not None:
            if len(eosin_bias_range) != 2 or eosin_bias_range[1] < eosin_bias_range[0] or eosin_bias_range[0] < -1.0 or 1.0 < eosin_bias_range[1]:
                raise dptaugmentationerrors.InvalidEosinBiasRangeError(eosin_bias_range)

        if dab_bias_range is not None:
            if len(dab_bias_range) != 2 or dab_bias_range[1] < dab_bias_range[0] or dab_bias_range[0] < -1.0 or 1.0 < dab_bias_range[1]:
                raise dptaugmentationerrors.InvalidDabBiasRangeError(dab_bias_range)
        '''
        # Store the settings.
        #
        self.__bias_ranges = [haematoxylin_bias_range, eosin_bias_range, dab_bias_range]

        self.__biases = [haematoxylin_bias_range if haematoxylin_bias_range is not None else 0.0,
                         eosin_bias_range if eosin_bias_range is not None else 0.0,
                         dab_bias_range if dab_bias_range is not None else 0.0]
        
    def __setcutoffrange(self, cutoff_range):
        """
        Set the cutoff value. Patches with mean value outside the cutoff interval will not be augmented.

        Args:
            cutoff_range (tuple, None): Patches with mean value outside the cutoff interval will not be augmented.

        Raises:
            InvalidCutoffRangeError: The cutoff range is not valid.
        """

        # Check the interval.
        #
       
        # Store the setting.
        #
        self.__cutoff_range = cutoff_range if cutoff_range is not None else [0.0, 1.0]

    def transform(self, patch):
        """
        Apply color deformation on the patch.

        Args:
            patch (np.ndarray): Patch to transform.

        Returns:
            np.ndarray: Transformed patch.
        """
        #print('hed self.__biases',self.__biases)
        #print('hed self.__sigmas',self.__sigmas)

        # Check if the patch is inside the cutoff values.
        #
        patch_mean = np.mean(a=patch) / 255.0
        if self.__cutoff_range[0] <= patch_mean <= self.__cutoff_range[1]:
            # Reorder the patch to channel last format and convert the image patch to HED color coding.
            #
            #patch_image = np.transpose(a=patch, axes=(1, 2, 0))
            patch_hed = rgb2hed(rgb=patch)

            # Augment the Haematoxylin channel.
            #
            if self.__sigmas[0] != 0.0:
                patch_hed[:, :, 0] *= (1.0 + self.__sigmas[0])

            if self.__biases[0] != 0.0:
                patch_hed[:, :, 0] += self.__biases[0]

            # Augment the Eosin channel.
            #
            if self.__sigmas[1] != 0.0:
                patch_hed[:, :, 1] *= (1.0 + self.__sigmas[1])

            if self.__biases[1] != 0.0:
                patch_hed[:, :, 1] += self.__biases[1]

            # Augment the DAB channel.
            #
            if self.__sigmas[2] != 0.0:
                patch_hed[:, :, 2] *= (1.0 + self.__sigmas[2])

            if self.__biases[2] != 0.0:
                patch_hed[:, :, 2] += self.__biases[2]

            # Convert back to RGB color coding and order back to channels first order.
            #
            patch_rgb = hed2rgb(hed=patch_hed)
            patch_rgb = np.clip(a=patch_rgb, a_min=0.0, a_max=1.0)
            patch_rgb *= 255.0
            patch_rgb = patch_rgb.astype(dtype=np.uint8)

            #patch_transformed = np.transpose(a=patch_rgb, axes=(2, 0, 1))

            return patch_rgb

        else:
            # The image patch is outside the cutoff interval.
            #
            return patch


    # def transform(self, patch):
    #     """
    #     Apply color deformation on the patch.
    #
    #     Args:
    #         patch (np.ndarray): Patch to transform.
    #
    #     Returns:
    #         np.ndarray: Transformed patch.
    #     """
    #     import time
    #
    #     print('### Timing ###')
    #     t_init = time.time()
    #
    #     # Check if the patch is inside the cutoff values.
    #     #
    #     patch_mean = np.mean(a=patch) / 255.0
    #     if self.__cutoff_range[0] <= patch_mean <= self.__cutoff_range[1]:
    #         # Reorder the patch to channel last format and convert the image patch to HED color coding.
    #         #
    #         # t = time.time()
    #         patch_image = np.transpose(a=patch, axes=(1, 2, 0))
    #         # print('{f} took {s} s'.format(f='initial transpose', s=(time.time() - t)), flush=True)
    #
    #         t = time.time()
    #         patch_hed = rgb2hed(rgb=patch_image)
    #         print('{f} took {s} s'.format(f='rgb2hed', s=(time.time() - t)), flush=True)
    #
    #         # Augment the Haematoxylin channel.
    #         #
    #         # t = time.time()
    #         if self.__sigmas[0] != 0.0:
    #             patch_hed[:, :, 0] *= (1.0 + self.__sigmas[0])
    #
    #         if self.__biases[0] != 0.0:
    #             patch_hed[:, :, 0] += self.__biases[0]
    #         # print('{f} took {s} s'.format(f='H variation', s=(time.time() - t)), flush=True)
    #
    #         # Augment the Eosin channel.
    #         #
    #         # t = time.time()
    #         if self.__sigmas[1] != 0.0:
    #             patch_hed[:, :, 1] *= (1.0 + self.__sigmas[1])
    #
    #         if self.__biases[1] != 0.0:
    #             patch_hed[:, :, 1] += self.__biases[1]
    #         # print('{f} took {s} s'.format(f='E variation', s=(time.time() - t)), flush=True)
    #
    #         # Augment the DAB channel.
    #         #
    #         # t = time.time()
    #         if self.__sigmas[2] != 0.0:
    #             patch_hed[:, :, 2] *= (1.0 + self.__sigmas[2])
    #
    #         if self.__biases[2] != 0.0:
    #             patch_hed[:, :, 2] += self.__biases[2]
    #         # print('{f} took {s} s'.format(f='D variation', s=(time.time() - t)), flush=True)
    #
    #         # Convert back to RGB color coding and order back to channels first order.
    #         #
    #         t = time.time()
    #         patch_rgb = hed2rgb(hed=patch_hed)
    #         print('{f} took {s} s'.format(f='hed2rgb', s=(time.time() - t)), flush=True)
    #
    #         # t = time.time()
    #         patch_rgb = np.clip(a=patch_rgb, a_min=0.0, a_max=1.0)
    #         # print('{f} took {s} s'.format(f='clip', s=(time.time() - t)), flush=True)
    #
    #         # t = time.time()
    #         patch_rgb *= 255.0
    #         patch_rgb = patch_rgb.astype(dtype=np.uint8)
    #         # print('{f} took {s} s'.format(f='255 and uint8', s=(time.time() - t)), flush=True)
    #
    #         # t = time.time()
    #         patch_transformed = np.transpose(a=patch_rgb, axes=(2, 0, 1))
    #         # print('{f} took {s} s'.format(f='transpose end', s=(time.time() - t)), flush=True)
    #
    #         p = patch_transformed
    #
    #     else:
    #         # The image patch is outside the cutoff interval.
    #         #
    #         p = patch
    #
    #     print('{f} took {s} s'.format(f='all', s=(time.time() - t_init)), flush=True)
    #     return p

   

### HsbColorAugmenter 

In [14]:
"""
This file contains a class for augmenting patches from whole slide images by applying color correction in HSB color space.
"""



import skimage.color


#----------------------------------------------------------------------------------------------------

class HsbColorAugmenter(ColorAugmenterBase):
    """Apply color correction in HSB color space on the RGB patch."""

    def __init__(self, hue_sigma_range, saturation_sigma_range, brightness_sigma_range):
        """
        Initialize the object.

        Args:
            hue_sigma_range (tuple, None): Adjustment range for the Hue channel from the [-1.0, 1.0] range where 0.0 means no change. For example (-0.5, 0.5).
            saturation_sigma_range (tuple, None): Adjustment range for the Saturation channel from the [-1.0, 1.0] range where 0.0 means no change.
            brightness_sigma_range (tuple, None): Adjustment range for the Brightness channel from the [-1.0, 1.0] range where 0.0 means no change.

        Raises:
            InvalidHueSigmaRangeError: The sigma range for Hue channel adjustment is not valid.
            InvalidSaturationSigmaRangeError: The sigma range for Saturation channel adjustment is not valid.
            InvalidBrightnessSigmaRangeError: The sigma range for Brightness channel adjustment is not valid.
        """

        # Initialize base class.
        #
        super().__init__(keyword='hsb_color')

        # Initialize members.
        #
        self.__sigma_ranges = None  # Configured sigma ranges for H, S, and B channels.
        self.__sigmas = None        # Randomized sigmas for H, S, and B channels.

        # Save configuration.
        #
        self.__setsigmaranges(hue_sigma_range=hue_sigma_range, saturation_sigma_range=saturation_sigma_range, brightness_sigma_range=brightness_sigma_range)

    def __setsigmaranges(self, hue_sigma_range, saturation_sigma_range, brightness_sigma_range):
        """
        Set the sigma ranges.

        Args:
            hue_sigma_range (tuple, None): Adjustment range for the Hue channel.
            saturation_sigma_range (tuple, None): Adjustment range for the Saturation channel.
            brightness_sigma_range (tuple, None): Adjustment range for the Brightness channel.

        Raises:
            InvalidHueSigmaRangeError: The sigma range for Hue channel adjustment is not valid.
            InvalidSaturationSigmaRangeError: The sigma range for Saturation channel adjustment is not valid.
            InvalidBrightnessSigmaRangeError: The sigma range for Brightness channel adjustment is not valid.
        """

        # Check the intervals.
        #
        '''
        if hue_sigma_range is not None:
            if len(hue_sigma_range) != 2 or hue_sigma_range[1] < hue_sigma_range[0] or hue_sigma_range[0] < -1.0 or 1.0 < hue_sigma_range[1]:
                raise dptaugmentationerrors.InvalidHueSigmaRangeError(hue_sigma_range)

        if saturation_sigma_range is not None:
            if len(saturation_sigma_range) != 2 or saturation_sigma_range[1] < saturation_sigma_range[0] or saturation_sigma_range[0] < -1.0 or 1.0 < saturation_sigma_range[1]:
                raise dptaugmentationerrors.InvalidSaturationSigmaRangeError(saturation_sigma_range)

        if brightness_sigma_range is not None:
            if len(brightness_sigma_range) != 2 or brightness_sigma_range[1] < brightness_sigma_range[0] or brightness_sigma_range[0] < -1.0 or 1.0 < brightness_sigma_range[1]:
                raise dptaugmentationerrors.InvalidBrightnessSigmaRangeError(brightness_sigma_range)
        '''
        # Store the setting.
        #
        self.__sigma_ranges = [hue_sigma_range, saturation_sigma_range, brightness_sigma_range]

        self.__sigmas = [hue_sigma_range if hue_sigma_range is not None else 0.0,
                         saturation_sigma_range if saturation_sigma_range is not None else 0.0,
                         brightness_sigma_range if brightness_sigma_range is not None else 0.0]
        

    def transform(self, patch):
        """
        Apply color deformation on the patch.

        Args:
            patch (np.ndarray): Patch to transform.

        Returns:
            np.ndarray: Transformed patch.
        """
        #print('hsb self.__sigmas',self.__sigmas)
        # Reorder the patch to channel last format and convert the image patch to HSB (=HSV) color coding.
        #
        #patch_image = np.transpose(a=patch, axes=(1, 2, 0))
        patch_hsb = skimage.color.rgb2hsv(rgb=patch)

        # Augment the Hue channel.
        #
        if self.__sigmas[0] != 0.0:
            patch_hsb[:, :, 0] += self.__sigmas[0] % 1.0
            patch_hsb[:, :, 0] %= 1.0

        # Augment the Saturation channel.
        #
        if self.__sigmas[1] != 0.0:
            if self.__sigmas[1] < 0.0:
                patch_hsb[:, :, 1] *= (1.0 + self.__sigmas[1])
            else:
                patch_hsb[:, :, 1] *= (1.0 + (1.0 - patch_hsb[:, :, 1]) * self.__sigmas[1])

        # Augment the Brightness channel.
        #
        if self.__sigmas[2] != 0.0:
            if self.__sigmas[2] < 0.0:
                patch_hsb[:, :, 2] *= (1.0 + self.__sigmas[2])
            else:
                patch_hsb[:, :, 2] += (1.0 - patch_hsb[:, :, 2]) * self.__sigmas[2]

        # Convert back to RGB color coding with byte data type and order back to channels first order.
        #
        patch_rgb = skimage.color.hsv2rgb(hsv=patch_hsb)
        patch_rgb *= 255.0
        patch_rgb = patch_rgb.astype(dtype=np.uint8)
        #patch_transformed = np.transpose(a=patch_rgb, axes=(2, 0, 1))

        return patch_rgb

   

## Noise

### NoiseAugmenterBase

In [15]:
"""
This file contains base class for augmenting patches from whole slide images with color transformations.
"""


#----------------------------------------------------------------------------------------------------

class NoiseAugmenterBase(AugmenterBase):
    """Base class for noise patch augmentation."""

    def __init__(self, keyword):
        """
        Initialize the object.

        Args:
            keyword (str): Short name for the transformation.
        """

        # Initialize the base class.
        #
        super().__init__(keyword=keyword)

### GaussianBlurAugmenter

In [16]:
"""
This file contains a class for augmenting patches from whole slide images with Gaussian blurring.
"""


#----------------------------------------------------------------------------------------------------

class GaussianBlurAugmenter(NoiseAugmenterBase):
    """Apply Gaussian blur on the patch."""

    def __init__(self, sigma_range):
        """
        Initialize the object.

        Args:
            sigma_range (tuple): Range for sigma selection for Gaussian blur. For example (0.1, 0.5).

        Raises:
            InvalidBlurSigmaRangeError: The sigma range for Gaussian blur is not valid.
        """

        # Initialize base class.
        #
        super().__init__(keyword='gaussian_blur')

        # Initialize members.
        #
        self.__sigma_range = None  # Configured sigma range.
        self.__sigma = None        # Current sigma to use.

        # Save configuration.
        #
        self.__setsigmarange(sigma_range=sigma_range)

    def __setsigmarange(self, sigma_range):
        """
        Set the sigma range.

        Args:
            sigma_range (tuple): Range for sigma selection for Gaussian blur.

        Raises:
            InvalidBlurSigmaRangeError: The sigma range for Gaussian blur is not valid.
        """

        # Check the interval.
        #
       
        # Store the setting.
        #
        self.__sigma_range = list(sigma_range)
        self.__sigma = sigma_range[0]

    def transform(self, patch):
        """
        Blur the patch with a random sigma.

        Args:
            patch (np.ndarray): Patch to transform.

        Returns:
            np.ndarray: Transformed patch.
        """

        # Normalize patch range to [0.0, 1.0].
        #
        patch_normalized = patch / 255.0

        # Blur the patch by channels.
        #
        patch_transformed = scipy.ndimage.gaussian_filter(
            input=patch_normalized, 
            sigma=(0.0, self.__sigma, self.__sigma)
        )
        # Restore the [0, 255] range.
        #
        patch_transformed *= 255.0
        patch_transformed = patch_transformed.astype(dtype=np.uint8)

        return patch_transformed.astype(dtype=np.uint8)

    def randomize(self):
        """Randomize the parameters of the augmenter."""

        self.__sigma = np.random.uniform(low=self.__sigma_range[0], high=self.__sigma_range[1], size=None)

### AdditiveGaussianNoiseAugmenter

In [17]:
"""
This file contains a class for augmenting patches from whole slide images with additive Gaussian noise.
"""


#----------------------------------------------------------------------------------------------------

class AdditiveGaussianNoiseAugmenter(NoiseAugmenterBase):
    """Apply additive Gaussian noise on the patch."""

    def __init__(self, sigma_range):
        """
        Initialize the object.

        Args:
            sigma_range (tuple): Range for sigma selection for Gaussian noise. For example (0.0, 0.1).

        Raises:
            InvalidAdditiveGaussianNoiseSigmaRangeError: The sigma range for additive Gaussian noise is not valid.
        """

        # Initialize base class.
        #
        super().__init__(keyword='additive_gaussian_noise')

        # Initialize members.
        #
        self.__sigma_range = None  # Configured sigma range.
        self.__sigma = None        # Current sigma to use.

        # Save configuration.
        #
        self.__setsigmarange(sigma_range=sigma_range)

    def __setsigmarange(self, sigma_range):
        """
        Set the sigma range.

        Args:
            sigma_range (tuple): Range for sigma selection for Gaussian noise.

        Raises:
            InvalidAdditiveGaussianNoiseSigmaRangeError: The sigma range for additive Gaussian noise is not valid.
        """

        # Check the interval.
        #
       
        # Store the setting.
        #
        self.__sigma_interval = list(sigma_range)
        self.__sigma = sigma_range[0]

    def transform(self, patch):
        """
        Apply additive Gaussian noise on the patch.

        Args:
            patch (np.ndarray): Patch to transform.

        Returns:
            np.ndarray: Transformed patch.
        """

        # Normalize patch range to [0.0, 1.0].
        #
        patch_normalized = patch / 255.0

        # Add noise and clip the result to the valid (0.0, 1.0) range.
        #
        noise = np.random.normal(loc=0, scale=self.__sigma, size=patch.shape)
        patch_transformed = patch_normalized + noise
        patch_transformed = np.clip(a=patch_transformed, a_min=0.0, a_max=1.0)

        # Restore the [0, 255] range.
        #
        patch_transformed *= 255.0
        patch_transformed = patch_transformed.astype(dtype=np.uint8)

        return patch_transformed

    def randomize(self):
        """Randomize the parameters of the augmenter."""

        # Randomize the noise sigma.
        #
        self.__sigma = np.random.uniform(low=self.__sigma_interval[0], high=self.__sigma_interval[1], size=None)

## Spatial

### SpatialAugmenterBase

In [18]:
class SpatialAugmenterBase(AugmenterBase):
    """Base class for spatial patch augmentation."""

    def __init__(self, keyword):
        """
        Initialize the object.

        Args:
            keyword (str): Short name for the transformation.
        """

        # Initialize the base class.
        #
        super().__init__(keyword=keyword)

### ScalingAugmenter

In [19]:
"""
This file contains a class for augmenting patches from whole slide images with scaling.
"""
#----------------------------------------------------------------------------------------------------

class ScalingAugmenter(SpatialAugmenterBase):
    """Apply scaling on the patch."""

    def __init__(self, scaling_range, interpolation_order=1):
        """
        Initialize the object.

        Args:
            scaling_range (tuple): Range for scaling factor selection. For example (0.8, 1.2).
            interpolation_order (int): Interpolation order from the range [0, 5].

        Raises:
            InvalidScalingRangeError: The sigma range for scaling is not valid.
            InvalidScalingInterpolationOrderError: The interpolation order for scaling is not valid.
        """

        # Initialize base class.
        #
        super().__init__(keyword='scaling')

        # Initialize members.
        #
        self.__scaling_range = []       # Configured scaling range.
        self.__scaling_factor = None    # Current scaling factor to use.
        self.__interpolation_order = 0  # Interpolation order.

        # Save configuration.
        #
        self.__setscalingrange(scaling_range=scaling_range, interpolation_order=interpolation_order)

    def __setscalingrange(self, scaling_range, interpolation_order):
        """
        Set the scaling interval.

        Args:
            scaling_range (tuple): Range for scaling factor selection.
            interpolation_order (int): Interpolation order.

        Raises:
            InvalidScalingRangeError: The sigma range for scaling is not valid.
            InvalidScalingInterpolationOrderError: The interpolation order for scaling is not valid.
        """

        # Check the interval.
        #
        '''
        if len(scaling_range) != 2 or scaling_range[1] < scaling_range[0] or scaling_range[0] <= 0.0:
            raise dptaugmentationerrors.InvalidScalingRangeError(scaling_range)

        # Check the interpolation order.
        #
        if interpolation_order < 0 or 5 < interpolation_order:
            raise dptaugmentationerrors.InvalidScalingInterpolationOrderError(interpolation_order)
        '''
        # Store the setting.
        #
        self.__scaling_interval = list(scaling_range)
        self.__scaling_factor = scaling_range[0]
        self.__interpolation_order = int(interpolation_order)

    def shapes(self, target_shapes):
        """
        Calculate the required shape of the input to achieve the target output shape.

        Args:
            target_shapes (dict): Target output shape per level.

        Returns:
            (dict): Required input shape per level.
        """

        # Calculate the required input shape for each level.
        #
        return {level: (math.ceil(target_shapes[level][0] / self.__scaling_interval[0]), math.ceil(target_shapes[level][1] / self.__scaling_interval[0])) for level in target_shapes}

    def transform(self, patch):
        """
        Scale the patch with a random factor.

        Args:
            patch (np.ndarray): Patch to transform.

        Returns:
            np.ndarray: Transformed patch.
        """

        # Pad patch to keep the original shape.
        #
        if self.__scaling_factor < 1.0:
            pad_ratio = ((1.0 / self.__scaling_factor - 1.0) / 2.0)
            pad_widths = (patch.shape[1] * pad_ratio, patch.shape[2] * pad_ratio)
            pad_config = ((0, 0), (math.ceil(pad_widths[0]), math.ceil(pad_widths[0])), (math.ceil(pad_widths[1]), math.ceil(pad_widths[1])))

            patch_padded = np.pad(array=patch, pad_width=pad_config, mode='reflect')
        else:
            patch_padded = patch

        # Zoom patch.
        #
        patch_transformed = scipy.ndimage.zoom(input=patch_padded, zoom=(1.0, self.__scaling_factor, self.__scaling_factor), order=self.__interpolation_order, mode='reflect')

        # Crop zoomed patch.
        #
        if patch_transformed.shape != patch.shape:
            border = (math.floor((patch_transformed.shape[1] - patch.shape[1]) / 2.0), math.floor((patch_transformed.shape[2] - patch.shape[2]) / 2.0))
            patch_transformed = patch_transformed[:, border[0]:border[0]+patch.shape[1], border[1]:border[1]+patch.shape[2]]

        return patch_transformed

    def randomize(self):
        """Randomize the parameters of the augmenter."""

        # Randomize the scaling factor.
        #
        self.__scaling_factor = np.random.uniform(low=self.__scaling_interval[0], high=self.__scaling_interval[1], size=None)

### Rotate90Augmenter

In [20]:
"""
This file contains a class for augmenting patches from whole slide images with rotating by multiples of 90 degrees.
"""

#----------------------------------------------------------------------------------------------------

class Rotate90Augmenter(SpatialAugmenterBase):
    """Rotate patch by 90, 180 or 270 degrees."""

    def __init__(self, k_list):
        """
        Initialize the object.

        Args:
            k_list (list): List of 90 degree rotation repetition times. Example: k_list = [0, 1, 2, 3] for 0, 90,
                180 and 270 degrees.

        Raises:
            InvalidRotationRepetitionListError: The list for 90 degree rotation repetition is invalid.
        """

        # Initialize base class.
        #
        super().__init__(keyword='rotate_90')

        # Initialize members.
        #
        self.__k_list = []  # List of rotation repetitions to use.
        self.__k = None     # Current repetition number to use.

        # Save configuration.
        #
        self.__setklist(k_list=k_list)

    def __setklist(self, k_list):
        """
        Set the rotation repetition times list.

        Args:
            k_list (list): List of 90 degree rotation repetition times.

        Raises:
            InvalidRotationRepetitionListError: The list for 90 degree rotation repetition is invalid.
        """

        # Check the list.
        #
       
        # Store the setting.
        #
        self.__k_list = [int(k_item) % 4 for k_item in k_list]
        self.__k = self.__k_list[0]

    def transform(self, patch):
        """
        Rotate the patch with multiple of 90 degrees.

        Args:
            patch (np.ndarray): Patch to transform.

        Returns:
            np.ndarray: Transformed patch.
        """

        # Rotate the patch.
        #
        patch_transformed = np.rot90(m=patch, k=self.__k, axes=(1, 2))

        return patch_transformed

    def randomize(self):
        """Randomize the parameters of the augmenter."""

        # Randomize the K.
        #
        self.__k = np.random.choice(a=self.__k_list, size=None)

### FlipAugmenter

In [21]:
"""
This file contains a class for augmenting patches from whole slide images with left-right or upside-down flipping.
"""


#----------------------------------------------------------------------------------------------------

class FlipAugmenter(SpatialAugmenterBase):
    """Mirrors patch vertically, horizontally or both."""

    def __init__(self, flip_list):
        """
        Initialize the object.

        Args:
            flip_list (list): List of possible flips. Example: flip_list = ['none', 'vertical', 'horizontal', 'both'].

        Raises:
            InvalidFlipListError: The flip list is invalid.
        """

        # Initialize base class.
        #
        super().__init__(keyword='flip')

        # Initialize members.
        #
        self.__flip_list = []  # List of possible flip modes.
        self.__flip = None     # Current flip to use.

        # Save configuration.
        #
        self.__setfliplist(flip_list=flip_list)

    def __setfliplist(self, flip_list):
        """
        Save the flip direction set.

        Args:
            flip_list (list): List of possible flips. Example: flip_list = ['none', 'vertical', 'horizontal', 'both'].

        Raises:
            InvalidFlipListError: The flip list is invalid.
        """

        # Check the list.
        #
      
        # Store the setting.
        #
        self.__flip_list = flip_list
        self.__flip = self.__flip_list[0]

    def transform(self, patch):
        """
        Flip the given patch none, vertically, horizontally or both.

        Args:
            patch (np.ndarray): Patch to transform.

        Returns:
            np.ndarray: Transformed patch.
        """

        # Flip the patch.
        #
        if self.__flip == 'none':
            patch_transformed = np.transpose(a=patch, axes=(1, 2, 0))
        elif self.__flip == 'vertical':
            patch_transformed = np.flipud(np.transpose(a=patch, axes=(1, 2, 0)))
        elif self.__flip == 'horizontal':
            patch_transformed = np.fliplr(np.transpose(a=patch, axes=(1, 2, 0)))
        elif self.__flip == 'both':
            patch_transformed = np.fliplr(np.flipud(np.transpose(a=patch, axes=(1, 2, 0))))
        
        # Transpose the patch back to the right color first order.
        #
        patch_transformed = np.transpose(a=patch_transformed, axes=(2, 0, 1))

        return patch_transformed

    def randomize(self):
        """Randomize the parameters of the augmenter."""

        # Randomize the flip direction.
        #
        self.__flip = np.random.choice(a=self.__flip_list, size=None)

### ElasticAugmenter

In [22]:
"""
This file contains a class for augmenting patches from whole slide images by applying elastic transformation.
"""

#----------------------------------------------------------------------------------------------------

class ElasticAugmenter(SpatialAugmenterBase):
    """Apply elastic deformation to patch. Deformation maps are created when the first patch is deformed."""

    def __init__(self, sigma_interval, alpha_interval, map_count, interpolation_order=1):
        """
        Initialize the object.

        Args:
            sigma_interval (tuple): Interval for sigma selection for Gaussian filter map.
            alpha_interval (tuple): Interval for alpha selection for the severity of the deformation.
            map_count (int): Amount of deformation maps to precalculate.
            interpolation_order (int): Interpolation order from the range [0, 5].

        Raises:
            InvalidElasticSigmaIntervalError: The interval of sigma for elastic deformation is invalid.
            InvalidElasticAlphaIntervalError: The interval of alpha for elastic deformation is invalid.
            InvalidElasticMapCountError: The number of elastic deformation maps to precalculate is invalid.
            InvalidElasticInterpolationOrderError: The interpolation order for elastic transformation is not valid.
        """

        # Initialize base class.
        #
        super().__init__(keyword='elastic')

        # Initialize members.
        #
        self.__sigma_interval = []      # Sigma.
        self.__alpha_interval = []      # Alpha.
        self.__map_count = 0            # Number of deformation maps to pre-calculate.
        self.__interpolation_order = 0  # Interpolation order.
        self.__deformation_maps = {}    # Deformation maps per patch shape.
        self.__map_choice = 0           # Selected deformation map.

        # Save configuration.
        #
        self.__cofiguredeformationmaps(sigma_interval=sigma_interval, alpha_interval=alpha_interval, map_count=map_count, interpolation_order=interpolation_order)

    def __cofiguredeformationmaps(self, sigma_interval, alpha_interval, map_count, interpolation_order):
        """
        Configure the deformation map calculation parameters.

        Args:
            sigma_interval (tuple): Interval for sigma selection for Gaussian filter map.
            alpha_interval (tuple): Interval for alpha selection for the severity of the deformation.
            map_count (int): Amount of deformation maps to precalculate.
            interpolation_order (int): Interpolation order from the range [0, 5].

        Raises:
            InvalidElasticSigmaIntervalError: The interval of sigma for elastic deformation is invalid.
            InvalidElasticAlphaIntervalError: The interval of alpha for elastic deformation is invalid.
            InvalidElasticMapCountError: The number of elastic deformation maps to precalculate is invalid.
            InvalidElasticInterpolationOrderError: The interpolation order for elastic transformation is not valid.
        """

        # Check the sigma interval.
      

        # Store the settings.
        #
        self.__sigma_interval = list(sigma_interval)
        self.__alpha_interval = list(alpha_interval)
        self.__map_count = int(map_count)
        self.__interpolation_order = int(interpolation_order)

    def __createdeformationmaps(self, image_shape):
        """
        Elastic deformation of images as described in Simard, Steinkraus and Platt, "Best Practices for Convolutional Neural Networks applied to Visual Document Analysis",
        in Proc. of the International Conference on Document Analysis and Recognition, 2003.

        Args:
            image_shape (tuple): Image shape to deform.
        """

        self.__deformation_maps[image_shape[1:3]] = []

        for _ in range(self.__map_count):
            alpha = np.random.uniform(low=self.__alpha_interval[0], high=self.__alpha_interval[1], size=None)
            sigma = np.random.uniform(low=self.__sigma_interval[0], high=self.__sigma_interval[1], size=None)

            dx = scipy.ndimage.filters.gaussian_filter(input=(np.random.rand(*image_shape) * 2 - 1), sigma=sigma, mode='constant', cval=0) * alpha
            dy = scipy.ndimage.filters.gaussian_filter(input=(np.random.rand(*image_shape) * 2 - 1), sigma=sigma, mode='constant', cval=0) * alpha
            z, x, y = np.meshgrid(np.arange(image_shape[0]), np.arange(image_shape[1]), np.arange(image_shape[2]), indexing='ij')
            indices = (np.reshape(z, (-1, 1)), np.reshape(x + dx, (-1, 1)), np.reshape(y + dy, (-1, 1)))

            self.__deformation_maps[image_shape[1:3]].append(indices)

    def transform(self, patch):
        """
        Deform the image with a random deformation map.

        Args:
            patch (np.ndarray): Patch to transform.

        Returns:
            np.ndarray: Transformed patch.
        """

        # Initialize the deformation maps.
        #
        if patch.shape[1:3] not in self.__deformation_maps:
            self.__createdeformationmaps(patch.shape)

        # Apply elastic deformation.
        #
        indices = self.__deformation_maps[patch.shape[1:3]][self.__map_choice]
        patch_transformed = scipy.ndimage.interpolation.map_coordinates(input=patch, coordinates=indices, order=self.__interpolation_order, mode='reflect').reshape(patch.shape)

        return patch_transformed

    def randomize(self):
        """Randomize the parameters of the augmenter."""

        # Randomize the transformation map.
        #
        self.__map_choice = np.random.randint(low=0, high=self.__map_count - 1)

## RandAugment_New_Ranges

### Needs to be purged of TensorFlow Code

In [23]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function
"""
This repository is build upon RandAugment implementation
https://arxiv.org/abs/1909.13719 published here
https://github.com/tensorflow/tpu/blob/master/models/official/efficientnet/autoaugment.py
"""
#Copyright 2019 The TensorFlow Authors. All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# ==============================================================================
"""AutoAugment and RandAugment policies for enhanced image preprocessing.
AutoAugment Reference: https://arxiv.org/abs/1805.09501
RandAugment Reference: https://arxiv.org/abs/1909.13719
"""


# augmentation scheme.
_MAX_LEVEL = 10.
_REPLACE = 128

def scaling(image,factor):
    
    
    if random.random() > 0.5:
        factor = factor/60
        augmentor = ScalingAugmenter(scaling_range=(1-factor,3), interpolation_order=1)
    else:
        factor = factor/30
        augmentor = ScalingAugmenter(scaling_range=(1+factor,3), interpolation_order=1)
    image = augmentor.transform(image)
    #image = PIL.Image.fromarray(image)
    
    
    #transform=RandomScale(scale_limit=factor, interpolation=1, always_apply=False, p=1)


    #print('_hsv_h',np.max(image))
    #image = transform.apply(img=image)
    return image#['image']


def hsv_h(image, factor):
    #image=PIL.Image.fromarray(image)
    #print('image',image.shape)

    #factor = random.uniform(0, factor)
    factor=factor/30

    if random.random() > 0.5:
        factor = -factor
    #image=np.asarray(image)
    #print('hsv h factor',factor)
    augmentor= HsbColorAugmenter(hue_sigma_range = factor, saturation_sigma_range=0, brightness_sigma_range=0)
    #Not randomizing the augmentation magnitude 
    #augmentor.randomize()

    image = augmentor.transform(image)
    #image = PIL.Image.fromarray(image)
    '''
    if num < 0.01:
        image.save('/mnt/netcache/pathology/projects/autoaugmentation/data/saved_fastauto/'+'_hsv_h'+str(num)+'.jpg')

    '''
    #print('_hsv_h',np.max(image))
    return image#np.transpose(augmentor.transform(image),[1,2,0])


def hsv_s(image, factor):
    #factor = random.uniform(0, factor)
    #image=np.asarray(image)
    factor=factor/30
    if random.random() > 0.5:
        factor = -factor
    #print('image',image.shape)
    #augmentor.transform(image)#
    #print('hsv s factor',factor)
    augmentor= HsbColorAugmenter(hue_sigma_range=0, saturation_sigma_range=factor, brightness_sigma_range=0)
    #Not randomizing the augmentation magnitude 
    #augmentor.randomize()
    image = augmentor.transform(image)
    #image = PIL.Image.fromarray(image)
    '''
    if num < 0.01:
        image.save('/mnt/netcache/pathology/projects/autoaugmentation/data/saved_fastauto/'+'_hsv_s'+str(num)+'.jpg')

    '''
    #print('_hsv_s',np.max(image))
    return image#np.transpose(augmentor.transform(image),[1,2,0])  

def hsv_v(image, factor):
    #factor = random.uniform(0, factor)
    #image=np.asarray(image)
    factor=factor/30
    if random.random() > 0.5:
        factor = -factor
    
    #print('image',image.shape)
    #image=np.transpose(image,[2,0,1])
    #print('image',image.shape)
    #print('hsv v factor',factor)
    augmentor= HsbColorAugmenter(hue_sigma_range=0, saturation_sigma_range=0, brightness_sigma_range=factor)
    #Not randomizing the augmentation magnitude 
    #augmentor.randomize()
    image = augmentor.transform(image)
    #image = PIL.Image.fromarray(image)
    '''
    if num < 0.01:
        image.save('/mnt/netcache/pathology/projects/autoaugmentation/data/saved_fastauto/'+'_hsv_v'+str(num)+'.jpg')

    '''
    #print('_hsv_v',np.max(image))
    return image#np.transpose(augmentor.transform(image),[1,2,0])    


def hsv(image, factor):
    #factor = random.uniform(0, factor)
    #image=np.asarray(image)
    factor=factor/30
    if random.random() > 0.5:
        factor = -factor
    
    #print('image',image.shape)
    #image=np.transpose(image,[2,0,1])
    #print('image',image.shape)
    #print('hsv v factor',factor)
    augmentor= HsbColorAugmenter(hue_sigma_range=factor, saturation_sigma_range=factor, brightness_sigma_range=factor)
    #Not randomizing the augmentation magnitude 
    augmentor.randomize()
    image = augmentor.transform(image)
    #image = PIL.Image.fromarray(image)
    '''
    if num < 0.01:
        image.save('/mnt/netcache/pathology/projects/autoaugmentation/data/saved_fastauto/'+'_hsv_v'+str(num)+'.jpg')

    '''
    #print('_hsv_v',np.max(image))
    return image#np.transpose(augmentor.transform(image),[1,2,0])

def hed(image, factor):
    #factor = random.uniform(0, factor)
    #image=np.asarray(image)
    factor=factor/30
    if random.random() > 0.5:
        factor = -factor
    #print('applying hed_h')
    #image=np.transpose(image,[2,0,1])
    #print('imagin hed_h imagee',image.shape)
    #image=np.transpose(image,[2,0,1])
    #print('hed h factor',factor)
    augmentor= HedColorAugmenter(haematoxylin_sigma_range=factor, haematoxylin_bias_range=factor,
                                            eosin_sigma_range=factor, eosin_bias_range=factor,
                                            dab_sigma_range=factor, dab_bias_range=factor,
                                            cutoff_range=(0.15, 0.85))
    #Not randomizing the augmentation magnitude 
    augmentor.randomize()
    image = augmentor.transform(image)
    #image = PIL.Image.fromarray(image)
    '''
    if num < 0.01:
        image.save('/mnt/netcache/pathology/projects/autoaugmentation/data/saved_fastauto/'+'_hed_h'+str(num)+'.jpg')

    '''
    #print('_hed_h',np.max(image))
    return image

def hed_h(image, factor):
    #factor = random.uniform(0, factor)
    #image=np.asarray(image)
    factor=factor/30
    if random.random() > 0.5:
        factor = -factor
    #print('applying hed_h')
    #image=np.transpose(image,[2,0,1])
    #print('imagin hed_h imagee',image.shape)
    #image=np.transpose(image,[2,0,1])
    #print('hed h factor',factor)
    augmentor= HedColorAugmenter(haematoxylin_sigma_range=factor, haematoxylin_bias_range=factor,
                                            eosin_sigma_range=0, eosin_bias_range=0,
                                            dab_sigma_range=0, dab_bias_range=0,
                                            cutoff_range=(0.15, 0.85))
    #Not randomizing the augmentation magnitude 
    #augmentor.randomize()
    image = augmentor.transform(image)
    #image = PIL.Image.fromarray(image)
    '''
    if num < 0.01:
        image.save('/mnt/netcache/pathology/projects/autoaugmentation/data/saved_fastauto/'+'_hed_h'+str(num)+'.jpg')

    '''
    #print('_hed_h',np.max(image))
    return image#np.transpose(augmentor.transform(image),[1,2,0])

def hed_e(image, factor):
    #factor = random.uniform(0, factor)
    factor=factor/30
    if random.random() > 0.5:
        factor = -factor
    #print('image',image.shape)
    #image=np.transpose(image,[2,0,1])
    #image=np.asarray(image)
    #print('in hed_e image',image.shape)
    #print('hed e factor',factor)
    augmentor= HedColorAugmenter(haematoxylin_sigma_range=0, haematoxylin_bias_range=0,
                                            eosin_sigma_range=factor, eosin_bias_range=factor,
                                            dab_sigma_range=0, dab_bias_range=0,
                                            cutoff_range=(0.15, 0.85))
    #Not randomizing the augmentation magnitude 
    #augmentor.randomize()
    image = augmentor.transform(image)
    #image = PIL.Image.fromarray(image)
    '''
    if num < 0.01:
        image.save('/mnt/netcache/pathology/projects/autoaugmentation/data/saved_fastauto/'+'_hed_e'+str(num)+'.jpg')

    '''
    #print('_hed_e',np.max(image))
    return image#np.transpose(augmentor.transform(image),[1,2,0])

def hed_d(image, factor):
    #factor = random.uniform(0, factor)
    factor=factor/30
    if random.random() > 0.5:
        factor = -factor
    #print('image',image.shape)
    #image=np.transpose(image,[2,0,1])
    #image=np.asarray(image)
    #print('in hed_e image',image.shape)
    #print('hed e factor',factor)
    augmentor= HedColorAugmenter(haematoxylin_sigma_range=0, haematoxylin_bias_range=0,
                                            eosin_sigma_range=0, eosin_bias_range=0,
                                            dab_sigma_range=factor, dab_bias_range=factor,
                                            cutoff_range=(0.15, 0.85))
    #Not randomizing the augmentation magnitude 
    #augmentor.randomize()
    image = augmentor.transform(image)
    #image = PIL.Image.fromarray(image)
    '''
    if num < 0.01:
        image.save('/mnt/netcache/pathology/projects/autoaugmentation/data/saved_fastauto/'+'_hed_e'+str(num)+'.jpg')

    '''
    #print('_hed_e',np.max(image))
    return image#np.transpose(augmentor.transform(image),[1,2,0])   
def gauss_blur(image, factor):
  """Equivalent of PIL Gaussian Blur."""
  factor=factor/5
  #image=np.transpose(image,[2,0,1])
  augmentor= GaussianBlurAugmenter(sigma_range=(factor, factor*10))
  #Not randomizing the augmentation magnitude 
  #augmentor.randomize()
  return augmentor.transform(image)#np.transpose(augmentor.transform(image),[1,2,0])

def gauss_noise(image, factor):
  """Equivalent of PIL Gaussian noise."""
  factor=factor/2
  
  augmentor= AdditiveGaussianNoiseAugmenter(sigma_range=(0.1*factor,factor))
   
  return augmentor.transform(image)#(augmentor.transform(image),[1,2,0])

def elastic(image, factor):
    """Apply elastic transformation for natural images."""
    import albumentations as A
    
    # Ensure sigma is at least 1.0 (albumentations requirement)
    sigma = max(1.0, factor * 7)
    
    transform = A.ElasticTransform(
        alpha=factor * 7, 
        sigma=sigma,           # Now guaranteed >= 1
        # alpha_affine=factor * 7,
        p=1.0
    )
    
    transformed = transform(image=image)
    return transformed['image']

def color(image, factor):
  """Equivalent of PIL Color."""
  factor=factor/5+1
  image = Image.fromarray(image)
  image = ImageEnhance.Color(image).enhance(factor) 
  return np.asarray(image)


def contrast(image, factor):
  """Equivalent of PIL Contrast."""
  factor=factor/5+1
  image = Image.fromarray(image)
  image = ImageEnhance.Contrast(image).enhance(factor)
  return np.asarray(image)


def brightness(image, factor):
  """Equivalent of PIL Brightness."""
  factor=factor/10+1
  image = Image.fromarray(image)
  image = ImageEnhance.Brightness(image).enhance(factor)
  return np.asarray(image)



def rotate(image,degrees, replace=(_REPLACE,_REPLACE,_REPLACE)):
    """Equivalent of PIL Posterize."""
    if random.random() > 0.5:
        degrees = -degrees
    image = Image.fromarray(image)
    image =  image.rotate(angle=degrees*10,fillcolor =replace)
    return np.asarray(image)




def translate_x(image, pixels, replace=(_REPLACE,_REPLACE,_REPLACE)):

    """Equivalent of PIL Translate in X dimension."""
    if random.random() > 0.5:
        pixels = -pixels
    pixels=pixels*3
    image = Image.fromarray(image)
    image=image.transform(image.size, Image.AFFINE, (1, 0,pixels, 0, 1, 0), fillcolor =replace)
    return np.asarray(image)

def translate_y(image, pixels, replace=(_REPLACE,_REPLACE,_REPLACE)):
  """Equivalent of PIL Translate in Y dimension."""
  if random.random() > 0.5:
        pixels = -pixels
  pixels=pixels*3
  image = Image.fromarray(image)
  image=image.transform(image.size, Image.AFFINE, (1, 0, 0, 0, 1, pixels),fillcolor =replace)
  return np.asarray(image)

def shear_x(image, level, replace=(_REPLACE,_REPLACE,_REPLACE)):
  """Equivalent of PIL Shearing in X dimension."""
  # Shear parallel to x axis is a projective transform
  # with a matrix form of:
  # [1  level
  #  0  1].
  if random.random() > 0.5:
        level = -level
  level=level/20
  image = Image.fromarray(image)
  image=image.transform(image.size, Image.AFFINE, (1, level, 0, 0, 1, 0), Image.BICUBIC, fillcolor =replace)
  return np.asarray(image)


def shear_y(image, level, replace=(_REPLACE,_REPLACE,_REPLACE)):
  """Equivalent of PIL Shearing in Y dimension."""
  # Shear parallel to y axis is a projective transform
  # with a matrix form of:
  # [1  0
  #  level  1].
  level=level/20
  if random.random() > 0.5:
        level = -level
  image = Image.fromarray(image)
  image=image.transform(image.size, Image.AFFINE, (1, 0, 0,level,  1, 0), Image.BICUBIC, fillcolor =replace)
  return np.asarray(image)


def autocontrast(image):
  """Implements Autocontrast function from PIL using TF ops.
  Args:
    image: A 3D uint8 tensor.
  Returns:
    The image after it has had autocontrast applied to it and will be of type
    uint8.
  """
  image = Image.fromarray(image)
  image =  ImageOps.autocontrast(image)
  return np.asarray(image)


def identity(image):
  """Implements Identity
 
  """
  return image
  
def sharpness(image, factor):
  """Implements Sharpness function from PIL using TF ops."""
  image = Image.fromarray(image)
  image =  ImageEnhance.Sharpness(image).enhance(factor)
  return np.asarray(image)



def equalize(image):
  """Implements Equalize function from PIL using TF ops."""
  image = Image.fromarray(image)
  image =  ImageOps.equalize(image) 
  return np.asarray(image)
 
'''
    'HsvH': hsv_h,
    'HsvS': hsv_s,
    'HsvV': hsv_v,
    'HedH': hed_h,
    'HedE': hed_e,
    'HedD': hed_d,
'''



NAME_TO_FUNC = {
    'AutoContrast': autocontrast,
    'HsvH': hsv_h,
    'HsvS': hsv_s,
    'HsvV': hsv_v,
    'HedH': hed_h,
    'HedE': hed_e,
    'HedD': hed_d,
    'Hsv': hsv,
    'Hed': hed,
    'Identity': identity,
    'Equalize': equalize,
    'Rotate': rotate,
    'Color': color,
    'Contrast': contrast,
    'Brightness': brightness,
    'Sharpness': sharpness,
    'ShearX': shear_x,
    'ShearY': shear_y,
    'TranslateX': translate_x,
    'TranslateY': translate_y,
    'Elastic': elastic,
    'GaussBlur': gauss_blur,
    'GaussNoise': gauss_noise,
    'Scaling': scaling


}


def _randomly_negate_tensor(tensor):
  """With 50% prob turn the tensor negative."""
  rand_cva = list([1, 0])
  
  should_flip = random.choice(rand_cva)
  
  if should_flip == 1:
      final_tensor = tensor
  else:  
      final_tensor = -tensor
  return final_tensor




def _rotate_level_to_arg(level):
  level = (level/_MAX_LEVEL) * 30.
  level = _randomly_negate_tensor(level)
  return (level,)


def _shrink_level_to_arg(level):
  """Converts level to ratio by which we shrink the image content."""
  if level == 0:
    return (1.0,)  # if level is zero, do not shrink the image
  # Maximum shrinking ratio is 2.9.
  level = 2. / (_MAX_LEVEL / level) + 0.9
  return (level,)


def _enhance_level_to_arg(level):
  return ((level/_MAX_LEVEL) * 1.8 + 0.1,)
  
def _enhance_level_to_arg_hsv(level):
  return (level*0.03,)
  
def _enhance_level_to_arg_hed(level):
  return (level*0.03,)
  
def _enhance_level_to_arg_contrast(level):
  return ((level/_MAX_LEVEL) * 1.8 + 0.1,)
  
def _enhance_level_to_arg_brightness(level):
  return ((level/_MAX_LEVEL) * 1.8 + 0.1,)
  
def _enhance_level_to_arg_color(level):
  return ((level/_MAX_LEVEL) * 1.8 + 0.1,)



def _shear_level_to_arg(level):
  level = (level/_MAX_LEVEL) * 0.3
  # Flip level to negative with 50% chance.
  level = _randomly_negate_tensor(level)
  return (level,)

def _level_to_arg(level):

  return (level,)

def _translate_level_to_arg(level, translate_const):
  level = (level/_MAX_LEVEL) * float(translate_const)
  # Flip level to negative with 50% chance.
  level = _randomly_negate_tensor(level)
  return (level,)

'''
      'HsvH': _level_to_arg,
      'HsvS': _level_to_arg,
      'HsvV': _level_to_arg,
      'HedH': _level_to_arg,
      'HedE': _level_to_arg,
      'HedD': _level_to_arg,'''



def level_to_arg(hparams):
  return {
      'Identity': lambda level: (),
      'Hsv': _level_to_arg,
      'Hed': _level_to_arg,
      'HsvH': _level_to_arg,
      'HsvS': _level_to_arg,
      'HsvV': _level_to_arg,
      'HedH': _level_to_arg,
      'HedE': _level_to_arg,
      'HedD': _level_to_arg,
      'AutoContrast': lambda level: (),
      'Equalize': lambda level: (),
      'Rotate': _level_to_arg,
      'Color': _level_to_arg,
      'Contrast': _level_to_arg,
      'Brightness': _level_to_arg,
      'Sharpness': _level_to_arg,
      'ShearX': _level_to_arg,
      'ShearY': _level_to_arg,
      'TranslateX': _level_to_arg,
      'TranslateY': _level_to_arg,
      'Elastic': _level_to_arg,
      'GaussBlur': _level_to_arg,
      'GaussNoise': _level_to_arg,
      'Scaling': _level_to_arg,
  }

def _parse_policy_info(name, prob, level, replace_value, augmentation_hparams,magnitude):
  """Return the function that corresponds to `name` and update `level` param."""

  func = NAME_TO_FUNC[name]
  args = level_to_arg(augmentation_hparams)[name](level)
  if name == 'Hed':
    args = level_to_arg(augmentation_hparams)[name](magnitude)
  elif name == 'Hsv':
    args = level_to_arg(augmentation_hparams)[name](magnitude)

  # Check to see if prob is passed into function. This is used for operations
  # where we alter bboxes independently.
  # FIXED for Python 3.11
  if 'prob' in list(inspect.signature(func).parameters.keys()):
    args = tuple([prob] + list(args))

  # Add in replace arg if it is required for the function that is being called.
  # FIXED for Python 3.11
  if 'replace' in list(inspect.signature(func).parameters.keys()):
    # Make sure replace is the final argument
    assert 'replace' == list(inspect.signature(func).parameters.keys())[-1]
    args = tuple(list(args) + [replace_value])

  return (func, prob, args)

def _apply_func_with_prob(func, image, args, prob):
  """Apply `func` to image w/ `args` as input with probability `prob`."""
  assert isinstance(args, tuple)

  # If prob is a function argument, then this randomness is being handled
  # inside the function, so make sure it is always called.
  # FIXED for Python 3.11
  if 'prob' in list(inspect.signature(func).parameters.keys()):
    prob = 1.0

  # Apply the function with probability `prob`.
  should_apply_op = tf.cast(
      tf.floor(tf.random_uniform([], dtype=tf.float32) + prob), tf.bool)
  augmented_image = tf.cond(
      should_apply_op,
      lambda: func(image, *args),
      lambda: image)
  return augmented_image

def select_and_apply_random_policy(policies, image):
  """Select a random policy from `policies` and apply it to `image`."""
  policy_to_select = tf.random_uniform([], maxval=len(policies), dtype=tf.int32)
  # Note that using tf.case instead of tf.conds would result in significantly
  # larger graphs and would even break export for some larger policies.
  for (i, policy) in enumerate(policies):
    image = tf.cond(
        tf.equal(i, policy_to_select),
        lambda selected_policy=policy: selected_policy(image),
        lambda: image)
  return image





def distort_image_with_randaugment(image, num_layers, magnitude, randomize=True,randaugment_transforms_set='review'):
  """Applies the RandAugment policy to `image`.
  RandAugment is from the paper https://arxiv.org/abs/1909.13719,
  Args:
    image: `Tensor` of shape [height, width, 3] representing an image.
    num_layers: Integer, the number of augmentation transformations to apply
      sequentially to an image. Represented as (N) in the paper. Usually best
      values will be in the range [1, 3].
    magnitude: Integer, shared magnitude across all augmentation operations.
      Represented as (M) in the paper. Usually best values are in the range
      [1, 10].
  Returns:
    The augmented version of `image`.
  """
  #print(magnitude)
  replace_value = (128, 128, 128) #[128] * 3
  #tf.logging.info('Using RandAug.')
  augmentation_hparams = None #contrib_training.HParams(cutout_const=40, translate_const=10)
  #print('augmentation_hparams',augmentation_hparams)
  #The 'Default' option is the H&E tailored randaugment
  if randaugment_transforms_set=='review':
      available_ops = ['Scaling','TranslateX', 'TranslateY','ShearX', 'ShearY','Brightness', 'Sharpness','Color', 'Contrast','Rotate', 'Equalize','Identity','HsvH','HsvS','HsvV','HedH','HedE','HedD', 'Elastic','GaussBlur','GaussNoise']  
  elif randaugment_transforms_set=='midl':
      available_ops = ['Scaling','TranslateX', 'TranslateY','ShearX', 'ShearY','Brightness', 'Sharpness','Color', 'Contrast','Rotate', 'Equalize','Identity','Hsv','Hed', 'Elastic','GaussBlur','GaussNoise']  
  
  elif randaugment_transforms_set=='natural':
        available_ops = ['AutoContrast', 'Equalize', 'Rotate', 'Color', 'Contrast', 
                        'Brightness', 'Sharpness', 'ShearX', 'ShearY', 'TranslateX', 
                        'TranslateY', 'Identity', 'Elastic', 'GaussBlur', 'GaussNoise']
    
  #available_ops = ['TranslateX', 'TranslateY','ShearX', 'ShearY','Brightness', 'Sharpness','Color', 'Contrast','Rotate', 'Identity','Hsv','Hed']  

  for layer_num in range(num_layers):
    op_to_select = np.random.randint(low=0,high=len(available_ops))
    if randomize:
      random_magnitude = np.random.uniform(low=0, high=magnitude)
    else:
      random_magnitude = magnitude
    
    for (i, op_name) in enumerate(available_ops):
        prob = np.random.uniform(low=0.2, high=0.8)

        func, _, args = _parse_policy_info(op_name, prob, random_magnitude,
                                           replace_value, augmentation_hparams,magnitude)

        if  (i== op_to_select):

            selected_func=func
            selected_args=args
            image= selected_func(image, *selected_args)
        else: 
            image=image
  return image

## RandAugment Wrapper

In [24]:
class RandAugmentTransform:
    """Wrapper to apply RandAugment to PIL images in torchvision pipeline"""
    
    def __init__(self, num_layers, magnitude, randomize=True, randaugment_transforms_set='natural'):
        self.num_layers = num_layers
        self.magnitude = magnitude
        self.randomize = randomize
        self.randaugment_transforms_set = randaugment_transforms_set
    
    def __call__(self, img):
        """
        Args:
            img: PIL Image
        Returns:
            PIL Image (augmented)
        """
        # Convert PIL to numpy array (H, W, C) uint8
        img_np = np.array(img)
        
        # Apply RandAugment (returns numpy array)
        img_augmented = distort_image_with_randaugment(
            image=img_np,
            num_layers=self.num_layers,
            magnitude=self.magnitude,
            randomize=self.randomize,
            randaugment_transforms_set=self.randaugment_transforms_set
        )
        
        # Convert back to PIL Image
        return Image.fromarray(img_augmented.astype('uint8'))

# Dataset

## Base

In [25]:
ImageFile.LOAD_TRUNCATED_IMAGES = True

DATASETS2 = [
    # Debug
    "Debug28",
    "Debug224",
    # Small images
    "ColoredMNIST",
    "RotatedMNIST",
    # Big images
    "VLCS",
    "PACS",
    "OfficeHome",
    "TerraIncognita",
    "DomainNet",
    "SVIRO",
    # WILDS datasets
    "WILDSCamelyon",
    "WILDSFMoW",
    # Spawrious datasets
    "SpawriousO2O_easy",
    "SpawriousO2O_medium",
    "SpawriousO2O_hard",
    "SpawriousM2M_easy",
    "SpawriousM2M_medium",
    "SpawriousM2M_hard",
]

# OVERLAP_TYPES = ["none", "low", "mid", "high", "full", "0", "33", "66", "100"]
SPECIAL_OVERLAP_TYPES = ["0", "low", "high", "low_linked_only", "high_linked_only"]
OVERLAP_TYPES = ["0", "low", "high", "low_linked_only", "high_linked_only"]


def get_domain_classes(N_c, N_oc, repeat, N_s, seed):
    N_noc = N_c - N_oc
    Q = []
    C = list(range(N_c))

    random_state = np.random.RandomState(seed)

    # choose non overlapping classes
    C_noc = list(random_state.choice(C, replace=False, size=N_noc))
    C_oc = [x for x in C if x not in C_noc]

    # add to queue
    Q.extend(C_noc + list(np.repeat(C_oc, repeat)))

    # Round-robing distribution of classes
    domain_classes = [Q[i::N_s] for i in range(N_s)]

    # assert overlapping classes
    overlap = np.zeros(N_c)
    for cls_list in domain_classes:
        np.add.at(overlap, cls_list, 1)

    assert C_oc == list(np.where(overlap > 1)[0])

    # output
    print("C_noc", C_noc)
    print("C_oc", C_oc)
    print("Q", Q)
    print("domain_classes", domain_classes)

    return domain_classes


class DomainBedImageFolder(ImageFolder):
    """
    Custom class to allow class filtering
    """

    def __init__(
        self,
        root: str,
        transform: Optional[Callable] = None,
        target_transform: Optional[Callable] = None,
        remove_classes: List[int] = [],
        allowed_classes: List[int] = [],
        is_test_env: Optional[bool] = None,
        env_name: Optional[str] = None,
    ):
        super().__init__(root, transform, target_transform)

        # Remove specified classes
        old_samples = self.samples
        self.samples = []
        self.targets = []
        self.is_test_env = is_test_env
        self.allowed_classes = allowed_classes
        self.remove_classes = remove_classes
        self.env_name = env_name
        for sample in old_samples:
            _, target = sample

            if target not in remove_classes:
                self.samples.append(sample)
                self.targets.append(target)

        self.imgs = self.samples
        self.classes = list(set(self.targets))

    def __len__(self) -> int:
        return len(self.samples)

    def __str__(self):
        return (
            f"{self.env_name}: is_test_env={self.is_test_env}, allowed_classes={self.allowed_classes},"
            f" list(set(self.targets))={self.classes}, num_samples={len(self)}"
        )


def get_overlapping_classes(
    class_split: List[List[int]], num_classes: int
) -> List[int]:
    """
    Return the classes in multiple domains.
    """
    overlap = np.zeros(num_classes)
    for data in class_split:
        np.add.at(overlap, data, 1)

    overlapping_classes = list(np.where(overlap > 1)[0])

    return overlapping_classes


def get_dataset_class(dataset_name):
    """Return the dataset class with the given name."""
    if dataset_name not in globals():
        raise NotImplementedError("Dataset not found: {}".format(dataset_name))
    return globals()[dataset_name]


def num_environments(dataset_name):
    return len(get_dataset_class(dataset_name).ENVIRONMENTS)


class MultipleDomainDataset:
    N_STEPS = 5001  # Default, subclasses may override
    CHECKPOINT_FREQ = 100  # Default, subclasses may override
    N_WORKERS = 8  # Default, subclasses may override
    ENVIRONMENTS = None  # Subclasses should override
    INPUT_SHAPE = None  # Subclasses should override

    def __getitem__(self, index):
        return self.datasets[index]

    def __len__(self):
        return len(self.datasets)


class Debug(MultipleDomainDataset):
    def __init__(self, root, test_envs, hparams):
        super().__init__()
        self.input_shape = self.INPUT_SHAPE
        self.num_classes = 2
        self.datasets = []
        for _ in [0, 1, 2]:
            self.datasets.append(
                TensorDataset(
                    torch.randn(16, *self.INPUT_SHAPE),
                    torch.randint(0, self.num_classes, (16,)),
                )
            )


class Debug28(Debug):
    INPUT_SHAPE = (3, 28, 28)
    ENVIRONMENTS = ["0", "1", "2"]


class Debug224(Debug):
    INPUT_SHAPE = (3, 224, 224)
    ENVIRONMENTS = ["0", "1", "2"]


class MultipleEnvironmentMNIST(MultipleDomainDataset):
    def __init__(
        self,
        root,
        environments,
        dataset_transform,
        input_shape,
        num_classes,
        test_envs: List[int],
        domain_class_filter: List[List[int]],
    ):
        super().__init__()
        if root is None:
            raise ValueError("Data directory not specified!")

        original_dataset_tr = MNIST(root, train=True, download=True)
        original_dataset_te = MNIST(root, train=False, download=True)

        original_images = torch.cat(
            (original_dataset_tr.data, original_dataset_te.data)
        )

        original_labels = torch.cat(
            (original_dataset_tr.targets, original_dataset_te.targets)
        )

        shuffle = torch.randperm(len(original_images))

        original_images = original_images[shuffle]
        original_labels = original_labels[shuffle]

        assert len(test_envs) == 1, "Not performing leave-one-domain-out validation"
        num_envs = len(environments)

        self.num_classes = num_classes
        self.overlapping_classes = get_overlapping_classes(
            domain_class_filter, self.num_classes
        )

        # Dynamically associate a filter with a domain except for test_envs[0]
        num_filters = len(domain_class_filter)
        assert num_envs - 1 == num_filters  # b/c exempt first test env
        shift_filter = list(range(num_filters)) + list(range(num_filters))
        shift_filter = shift_filter[test_envs[0] : test_envs[0] + num_filters]

        self.datasets = []

        for i in range(len(environments)):
            images = original_images[i :: len(environments)]
            labels = original_labels[i :: len(environments)]
            self.datasets.append(dataset_transform(images, labels, environments[i]))

        self.input_shape = input_shape


class MultipleEnvironmentImageFolder(MultipleDomainDataset):
    def __init__(
        self,
        root,
        test_envs,
        augment,
        hparams,
        domain_class_filter: List[List[int]],
        num_classes=None,
        use_randaugment=False,  # ADD THIS
        rand_n=2,                # ADD THIS
        rand_m=9,                # ADD THIS
        randaugment_transforms_set='natural',  # 
    ):
        super().__init__()
        environments = [f.name for f in os.scandir(root) if f.is_dir()]
        environments = sorted(environments)
        num_envs = len(environments)

        assert len(test_envs) == 1, "Not performing leave-one-domain-out validation"

        self.idx_to_class = self.get_idx_to_class(
            os.path.join(root, environments[test_envs[0]])
        )
        self.num_classes = (
            len(self.idx_to_class) if num_classes is None else num_classes
        )

        self.overlapping_classes = get_overlapping_classes(
            domain_class_filter, self.num_classes
        )
        logging.info(f"Overlapping classes: {self.overlapping_classes}")

        # Dynamically associate a filter with a domain except for test_envs[0]
        num_filters = len(domain_class_filter)
        print(f"num_envs={num_envs}, num_filters={num_filters}, domain_class_filter={domain_class_filter}")
        assert num_envs - 1 == num_filters  # b/c exempt first test env
        shift_filter = list(range(num_filters)) + list(range(num_filters))
        shift_filter = shift_filter[test_envs[0] : test_envs[0] + num_filters]

        transform = transforms.Compose(
            [
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
                ),
            ]
        )

        if use_randaugment:
            # RandAugment-based augmentation
            augment_transform = transforms.Compose([
                transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
                transforms.RandomHorizontalFlip(),
                # INSERT RANDAUGMENT HERE
                RandAugmentTransform(
                    num_layers=rand_n,
                    magnitude=rand_m,
                    randomize=True,
                    randaugment_transforms_set=randaugment_transforms_set
                ),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406], 
                    std=[0.229, 0.224, 0.225]
                ),
            ])
        else:
            # Original FOND augmentation
            augment_transform = transforms.Compose([
                transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(0.3, 0.3, 0.3, 0.3),
                transforms.RandomGrayscale(),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406], 
                    std=[0.229, 0.224, 0.225]
                ),
            ])
        self.datasets = []
        for i, environment in enumerate(environments):
            path = os.path.join(root, environment)

            # setup augmentation
            if augment and (i not in test_envs):
                env_transform = augment_transform
            else:
                env_transform = transform

            # setup class filtering
            all_classes = set(list(self.idx_to_class.keys()))
            if i not in test_envs:
                filter = domain_class_filter[shift_filter.pop()]
                if filter == []:
                    continue
                remove_classes = list(all_classes - set(filter))

                env_dataset = DomainBedImageFolder(
                    path,
                    transform=env_transform,
                    remove_classes=remove_classes,
                    is_test_env=False,
                    allowed_classes=filter,
                    env_name=environment,
                )
            else:
                remove_classes = list(range(self.num_classes, len(all_classes)))
                env_dataset = DomainBedImageFolder(
                    path,
                    transform=env_transform,
                    remove_classes=remove_classes,
                    is_test_env=True,
                    allowed_classes=list(range(self.num_classes)),
                    env_name=environment,
                )
                assert self.num_classes == len(env_dataset.classes)

            # print(f"\n[info] environment: {env_dataset.env_name}, classes: {env_dataset.allowed_classes}, is_test: {env_dataset.is_test_env}")
            logging.info(f"Created domain -> {env_dataset}")
            self.datasets.append(env_dataset)

        self.input_shape = (
            3,
            224,
            224,
        )

    def get_overlapping_classes(
        self, class_split: List[List[int]], num_classes: int
    ) -> List[int]:
        """
        Return the classes in multiple domains.
        """
        overlap = np.zeros(num_classes)
        for data in class_split:
            np.add.at(overlap, data, 1)

        overlapping_classes = list(np.where(overlap > 1)[0])

        return overlapping_classes

    def get_idx_to_class(self, data_dir: str) -> Dict[int, str]:
        dataset = ImageFolder(data_dir)
        idx_to_class = {}
        for key, value in dataset.class_to_idx.items():
            idx_to_class.update({value: key})

        assert len(dataset.class_to_idx) == len(
            idx_to_class
        ), "Class and labels are not one-to-one"

        return idx_to_class

## PACS

In [26]:


class PACS(MultipleEnvironmentImageFolder):
    CHECKPOINT_FREQ = 300
    ENVIRONMENTS = sorted(["A", "C", "P", "S"])
    NUM_CLASSES = 7
    OVERLAP_CONFIG = {
        "0": [[0, 1], [2, 3], [4, 5, 6]],
        "low": [[0, 1, 2], [2, 3, 4], [4, 5, 6]],
        "high": [[0, 1, 2, 3], [2, 3, 4, 5], [4, 5, 6, 0]],
        "100": [list(range(7)), list(range(7)), list(range(7))],
        "low_linked_only": [[0, 1], [3], [5, 6]],
        "high_linked_only": [[0, 1], [], [6]],
    }

    # overlap_type
    def __init__(
        self,
        root: str,
        test_envs: list,
        hparams: dict,
        overlap_type: str,
        overlap_seed=None,
        num_classes=None,
        num_domain_linked_classes=None,
        use_randaugment=False,  # ADD THIS
        rand_n=2,                # ADD THIS
        rand_m=9,                # ADD THIS
        randaugment_transforms_set='natural',  # 
    ):
        # print(f"[info] {type(self)}, test_envs: {test_envs}, overlap: {class_overlap_id}")
        self.dir = root
        print(root)
        self._num_source_domains = 3
        self._num_classes = 7

        super().__init__(
            self.dir,
            test_envs,
            hparams["data_augmentation"],
            hparams,
            PACS.OVERLAP_CONFIG[overlap_type],
            
        use_randaugment=use_randaugment,  # ADD THIS
        rand_n=rand_n,                # ADD THIS
        rand_m=rand_m,                # ADD THIS
        randaugment_transforms_set=randaugment_transforms_set,  #
        )


## VLCS

In [27]:
class VLCS(MultipleEnvironmentImageFolder):
    CHECKPOINT_FREQ = 300
    ENVIRONMENTS = sorted(["C", "L", "S", "V"])
    NUM_CLASSES = 5
    OVERLAP_CONFIG = {
        "0": [[0, 1], [2, 3], [4]],
        "low": [[0, 1, 2], [2, 3], [3, 4]],
        "high": [[0, 1, 2], [2, 3, 4], [3, 4, 0]],
        "low_linked_only": [[0, 1], [], [4]],
        "high_linked_only": [[1], [], []],
        "100": [list(range(5)), list(range(5)), list(range(5))],
    }

    def __init__(
        self,
        root: str,
        test_envs: list,
        hparams: dict,
        overlap_type: str,
        overlap_seed=None,
        num_classes=None,
        num_domain_linked_classes=None,
        use_randaugment=True,  # ADD THIS,
        rand_n=2,                # ADD THIS
        rand_m=9,                # ADD THIS
        randaugment_transforms_set='natural',  # 
    ):
        # print(f"[info] {type(self)}, test_envs: {test_envs}, overlap: {class_overlap_id}")
        self.dir = os.path.join(root, "VLCS/")
        self._num_source_domains = 3
        self._num_classes = 5

        super().__init__(
            self.dir,
            test_envs,
            hparams["data_augmentation"],
            hparams,
            VLCS.OVERLAP_CONFIG[overlap_type],
            
            
        use_randaugment=use_randaugment,  # ADD THIS
        rand_n=rand_n,                # ADD THIS
        rand_m=rand_m,                # ADD THIS
        randaugment_transforms_set=randaugment_transforms_set,  #
        )


## OfficeHome

In [28]:


class OfficeHome(MultipleEnvironmentImageFolder):
    CHECKPOINT_FREQ = 300
    ENVIRONMENTS = sorted(["A", "C", "P", "R"])
    OVERLAP_CONFIG = {
        "0": [list(range(0, 22)), list(range(22, 44)), list(range(44, 65))],
        "low": [list(range(0, 30)), list(range(14, 44)), list(range(35, 65))],  # 25/65
        "high": [list(range(0, 38)), list(range(5, 44)), list(range(27, 65))],  # 50/65
        "low_linked_only": [
            list(range(0, 14)),
            list(range(30, 35)),
            list(range(44, 65)),
        ],
        "high_linked_only": [list(range(0, 5)), [], list(range(44, 65))],
        "100": [list(range(65)), list(range(65)), list(range(65))],
    }

    def __init__(
        self,
        root: str,
        test_envs: List[int],
        hparams: dict,
        num_classes: int,
        num_domain_linked_classes: int,
        overlap_type=None,
        overlap_seed=None,
        use_randaugment=False,
        rand_n=2,                # ADD THIS
        rand_m=9,                # ADD THIS
        randaugment_transforms_set='natural',  # 
    ):
        
        self.dir = root
        self._num_source_domains = 3
        self._num_classes = 65

        domain_class_filter = []
        if overlap_type is not None:
            logging.info(
                f"Using predefined class distributions: overlap_type={overlap_type}"
            )
            domain_class_filter = OfficeHome.OVERLAP_CONFIG[overlap_type]
        else:
            logging.info(
                f"Using dynamic class distributions: num_classes={num_classes}, num_linked={num_domain_linked_classes}, num_train_domains={self._num_source_domains}"
            )
            assert num_classes <= self._num_classes
            self._num_classes = num_classes
            domain_class_filter = create_domains(
                num_classes=num_classes,
                num_linked=num_domain_linked_classes,
                num_train_domains=self._num_source_domains,
            )
        print(f"DEBUG: overlap_type={overlap_type}")
        print(f"DEBUG: domain_class_filter={domain_class_filter}")
        print(f"DEBUG: len(domain_class_filter)={len(domain_class_filter)}")
        print(f"DEBUG: test_envs={test_envs}")


        super().__init__(
            self.dir,
            test_envs,
            hparams["data_augmentation"],
            hparams,
            domain_class_filter=domain_class_filter,
            num_classes=self._num_classes,
                     
        use_randaugment=use_randaugment,  # ADD THIS
        rand_n=rand_n,                # ADD THIS
        rand_m=rand_m,                # ADD THIS
        randaugment_transforms_set=randaugment_transforms_set,  #
            
        )


## Wilds

In [29]:

class WILDSEnvironment:
    def __init__(
        self, 
        wilds_dataset, 
        metadata_name, 
        metadata_value, 
        transform=None,
        is_test_env=False,
        allowed_classes=None  # Filter samples by class
    ):
        self.name = metadata_name + "_" + str(metadata_value)
        self.is_test_env = is_test_env

        metadata_index = wilds_dataset.metadata_fields.index(metadata_name)
        metadata_array = wilds_dataset.metadata_array
        subset_indices = torch.where(
            metadata_array[:, metadata_index] == metadata_value
        )[0]

        # Filter by allowed classes if specified
        if allowed_classes is not None:
            y_values = wilds_dataset.y_array[subset_indices]
            class_mask = torch.zeros(len(subset_indices), dtype=torch.bool)
            for allowed_class in allowed_classes:
                class_mask |= (y_values == allowed_class)
            subset_indices = subset_indices[class_mask]
            print(f"  {self.name}: Filtered to classes {allowed_classes}, {len(subset_indices)} samples")
        else:
            print(f"  {self.name}: All classes, {len(subset_indices)} samples")

        self.dataset = wilds_dataset
        self.indices = subset_indices
        self.transform = transform

    def __getitem__(self, i):
        x = self.dataset.get_input(self.indices[i])
        if type(x).__name__ != "Image":
            x = Image.fromarray(x)

        y = self.dataset.y_array[self.indices[i]]
        if self.transform is not None:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.indices)


class WILDSDataset(MultipleDomainDataset):
    INPUT_SHAPE = (3, 224, 224)

    def __init__(
        self, 
        dataset, 
        metadata_name, 
        test_envs, 
        augment, 
        hparams,
        overlap_config
    ):
        super().__init__()

        transform = transforms.Compose(
            [
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
                ),
            ]
        )

        augment_transform = transforms.Compose(
            [
                transforms.Resize((224, 224)),
                transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(0.3, 0.3, 0.3, 0.3),
                transforms.RandomGrayscale(),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
                ),
            ]
        )

        self.datasets = []
        metadata_values = self.metadata_values(dataset, metadata_name)
        
        print(f"[WILDS] Creating {len(metadata_values)} environments with overlap config: {overlap_config}")

        # Map environment index to allowed classes
        # Test environments get all classes, source environments get restricted classes
        source_env_idx = 0
        for i, metadata_value in enumerate(metadata_values):
            if augment and (i not in test_envs):
                env_transform = augment_transform
            else:
                env_transform = transform

            # Determine allowed classes for this environment
            if i in test_envs:
                # Test environment gets all classes
                allowed_classes = None
                print(f"[WILDS] Env {i} (TEST): all classes")
            else:
                # Source environment gets restricted classes from overlap_config
                if source_env_idx < len(overlap_config):
                    allowed_classes = overlap_config[source_env_idx]
                    print(f"[WILDS] Env {i} (SOURCE): classes {allowed_classes}")
                else:
                    allowed_classes = None
                    print(f"[WILDS] Env {i} (SOURCE): all classes (no config)")
                source_env_idx += 1

            env_dataset = WILDSEnvironment(
                dataset, 
                metadata_name, 
                metadata_value, 
                env_transform,
                is_test_env=(i in test_envs),
                allowed_classes=allowed_classes
            )

            self.datasets.append(env_dataset)

        self.input_shape = (3, 224, 224)
        self.num_classes = dataset.n_classes
        
        # Set overlapping classes from overlap_config (same as PACS/VLCS)
        all_overlapping = set()
        for domain_classes in overlap_config:
            all_overlapping.update(domain_classes)
        self.overlapping_classes = sorted(list(all_overlapping))
        
        print(f"[WILDS] Total overlapping classes: {self.overlapping_classes}")
        print(f"[WILDS] Dataset created with {len(self.datasets)} environments")

    def metadata_values(self, wilds_dataset, metadata_name):
        metadata_index = wilds_dataset.metadata_fields.index(metadata_name)
        metadata_vals = wilds_dataset.metadata_array[:, metadata_index]
        return sorted(list(set(metadata_vals.view(-1).tolist())))


class WILDSCamelyon(WILDSDataset):
    CHECKPOINT_FREQ = 300
    ENVIRONMENTS = [
        "hospital_0",
        "hospital_1",
        "hospital_2",
        "hospital_3",
        "hospital_4",
    ]
    NUM_CLASSES = 2  # Binary: 0=normal, 1=tumor
    
    # Overlap configurations matching PACS/VLCS structure
    # For 4 source domains (when 1 is held out as test)
    OVERLAP_CONFIG = {
        "0": [[0], [1], [0, 1], [0]],  # Minimal overlap
        "low": [[0], [0, 1], [1], [0, 1]],  # Some overlap
        "high": [[0, 1], [0, 1], [0, 1], [0, 1]],  # Full overlap
        "100": [[0, 1], [0, 1], [0, 1], [0, 1]],  # All classes everywhere
        "low_linked_only": [[0], [], [1], []],  # Sparse
        "high_linked_only": [[0], [], [], []],  # Very sparse
    }

    def __init__(
        self, 
        root: str,
        test_envs: list,
        hparams: dict,
        overlap_type: str,
        overlap_seed=None,
        num_classes=None,
        num_domain_linked_classes=None
    ):
        self.dir = os.path.join(root, "camelyon17_v1.0/")
        self._num_source_domains = 4  # 5 hospitals - 1 test = 4 source
        self._num_classes = 2
        
        print(f"[WILDSCamelyon] Initializing with overlap_type: {overlap_type}")
        print(f"[WILDSCamelyon] Test environments: {test_envs}")
        
        dataset = Camelyon17Dataset(root_dir=root)
        
        print(f"[WILDSCamelyon] Loaded base dataset: {len(dataset)} total samples")
        print(f"[WILDSCamelyon] Number of classes: {dataset.n_classes}")
        
        super().__init__(
            dataset,
            "hospital",
            test_envs,
            hparams["data_augmentation"],
            hparams,
            WILDSCamelyon.OVERLAP_CONFIG[overlap_type],
        )


# if __name__ == '__main__':
#     parser = argparse.ArgumentParser()
#     parser.add_argument("--data_dir", type=str, default="./data")
#     parser.add_argument("--overlap", type=str, default="high")
#     args = parser.parse_args()
    
#     dataset = WILDSCamelyon(
#         root=args.data_dir, 
#         test_envs=[0], 
#         hparams={'data_augmentation': True},
#         overlap_type=args.overlap
#     )
    
#     print(f"\n=== Dataset Summary ===")
#     print(f"Number of classes: {dataset.num_classes}")
#     print(f"Overlapping classes: {dataset.overlapping_classes}")
#     print(f"Number of environments: {len(dataset.datasets)}")
    
#     for i, env in enumerate(dataset.datasets):
#         print(f"  {env.name}: {len(env)} samples (test={env.is_test_env})")

## Init

In [30]:
DATASETS = {
    "PACS": PACS,
    "VLCS": VLCS,
    "OfficeHome": OfficeHome,
    "WILDSCamelyon": WILDSCamelyon
}


# Networks

## Base


In [31]:

class Identity(nn.Module):
    """An identity layer"""

    def __init__(self):
        super(Identity, self).__init__()

    def forward(self, x):
        return x


class ResNet(torch.nn.Module):
    """ResNet with the softmax chopped off and the batchnorm frozen"""

    def __init__(self, input_shape, hparams):
        """
        If you're using Compute Canada you will need to manually download
        and store the appropriate resnet18 and resnet50 weight files. Currently
        we are using:
            ResNet18_Weights.IMAGENET1K_V1
                "https://download.pytorch.org/models/resnet18-f37072fd.pth",
            ResNet50_Weights.IMAGENET1K_V2
                "https://download.pytorch.org/models/resnet50-11ad3fa6.pth",

        """
        super(ResNet, self).__init__()
        resnet18 = hparams["resnet18"]
        if resnet18:
            pretrain_weight = os.path.expanduser(
                "~/scratch/saved/resnet18-f37072fd.pth"
            )
            # assert os.path.isfile(pretrain_weight), f"File not found: {pretrain_weight}"
            if os.path.exists(pretrain_weight):
                print(
                    f"[info] loading weights resnet18: {resnet18}, from {pretrain_weight}"
                )
                self.network = torchvision.models.resnet18()
                self.network.load_state_dict(torch.load(pretrain_weight))
            else:
                self.network = torchvision.models.resnet18(
                    weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1
                )

            self.n_outputs = 512
        else:
            pretrain_weight = os.path.expanduser(
                "~/scratch/saved/resnet50-11ad3fa6.pth"
            )
            # assert os.path.isfile(pretrain_weight), f"File not found: {pretrain_weight}"
            if os.path.exists(pretrain_weight):
                print(
                    f"[info] loading weights resnet50: {resnet18}, from {pretrain_weight}"
                )
                self.network = torchvision.models.resnet50()
                self.network.load_state_dict(torch.load(pretrain_weight))
            else:
                self.network = torchvision.models.resnet50(
                    weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2
                )

            self.n_outputs = 2048

        # self.network = remove_batch_norm_from_resnet(self.network)

        # adapt number of channels
        nc = input_shape[0]
        if nc != 3:
            tmp = self.network.conv1.weight.data.clone()

            self.network.conv1 = nn.Conv2d(
                nc, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
            )

            for i in range(nc):
                self.network.conv1.weight.data[:, i, :, :] = tmp[:, i % 3, :, :]

        # save memory
        del self.network.fc
        self.network.fc = Identity()

        self.freeze_bn()
        self.hparams = hparams
        self.dropout = nn.Dropout(hparams["resnet_dropout"])

    def forward(self, x):
        """Encode x into a feature vector of size n_outputs."""
        return self.dropout(self.network(x))

    def train(self, mode=True):
        """
        Override the default train() to freeze the BN parameters
        """
        super().train(mode)
        self.freeze_bn()

    def freeze_bn(self):
        for m in self.network.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()


def Featurizer(input_shape, hparams):
    """Auto-select an appropriate featurizer for the given input shape."""
    # if len(input_shape) == 1:
    #     return MLP(input_shape[0], hparams["mlp_width"], hparams)
    # elif input_shape[1:3] == (28, 28):
    #     return MNIST_CNN(input_shape)
    # elif input_shape[1:3] == (32, 32):
    #     return wide_resnet.Wide_ResNet(input_shape, 16, 2, 0.)
    # elif input_shape[1:3] == (224, 224):
    if input_shape[1:3] == (224, 224):
        return ResNet(input_shape, hparams)
    else:
        raise NotImplementedError


def Classifier(in_features, out_features, is_nonlinear=False):
    if is_nonlinear:
        return torch.nn.Sequential(
            torch.nn.Linear(in_features, in_features // 2),
            torch.nn.ReLU(),
            torch.nn.Linear(in_features // 2, in_features // 4),
            torch.nn.ReLU(),
            torch.nn.Linear(in_features // 4, out_features),
        )
    else:
        return torch.nn.Linear(in_features, out_features)


class Algorithm(torch.nn.Module):
    """
    A subclass of Algorithm implements a domain generalization algorithm.
    Subclasses should implement the following:
    - update()
    - predict()
    """

    def __init__(self, input_shape, num_classes, num_domains, hparams):
        super(Algorithm, self).__init__()
        self.hparams = hparams

    def update(self, minibatches, unlabeled=None):
        """
        Perform one update step, given a list of (x, y) tuples for all
        environments.

        Admits an optional list of unlabeled minibatches from the test domains,
        when task is domain_adaptation.
        """
        raise NotImplementedError

    def predict(self, x):
        raise NotImplementedError


class ERM(Algorithm):
    """
    Empirical Risk Minimization (ERM)
    """

    def __init__(self, input_shape, num_classes, num_domains, hparams):
        super(ERM, self).__init__(input_shape, num_classes, num_domains, hparams)
        self.featurizer = Featurizer(input_shape, self.hparams)
        self.classifier = Classifier(
            self.featurizer.n_outputs, num_classes, self.hparams["nonlinear_classifier"]
        )

        self.network = nn.Sequential(self.featurizer, self.classifier)
        self.optimizer = torch.optim.Adam(
            self.network.parameters(),
            lr=self.hparams["lr"],
            weight_decay=self.hparams["weight_decay"],
        )

    def update(self, minibatches, unlabeled=None):
        all_x = torch.cat([x for x, y in minibatches])
        all_y = torch.cat([y for x, y in minibatches])
        loss = F.cross_entropy(self.predict(all_x), all_y)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return {"loss": loss.item()}

    def predict(self, x):
        return self.network(x)


## CDSA

In [32]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import Dict, List, Tuple, Optional


class CoVarianceMatrix:
    """
    Class to compute and maintain class-conditional covariance matrices
    for semantic augmentation across multiple domains.
    """
    def __init__(self, num_classes: int, num_domains: int, feature_dim: int, momentum: float = 0.9):
        """
        Args:
            num_classes: Number of classes in the dataset
            num_domains: Number of source domains
            feature_dim: Dimension of the feature space
            momentum: Momentum for exponential moving average
        """
        self.num_classes = num_classes
        self.num_domains = num_domains
        self.feature_dim = feature_dim
        self.momentum = momentum
        
        # Initialize covariance matrices for each domain and class
        # Shape: [num_domains, num_classes, feature_dim, feature_dim]
        self.covariance = {}
        self.mean = {}
        self.count = {}
        
        for domain_idx in range(num_domains):
            self.covariance[domain_idx] = {}
            self.mean[domain_idx] = {}
            self.count[domain_idx] = {}
            for class_idx in range(num_classes):
                self.covariance[domain_idx][class_idx] = torch.eye(feature_dim).cuda()
                self.mean[domain_idx][class_idx] = torch.zeros(feature_dim).cuda()
                self.count[domain_idx][class_idx] = 0
        self.device = None
    
    def update(self, features, labels, domain_ids):
        # Set device on first update
        if self.device is None:
            self.device = features.device
            # Move all tensors to the correct device
            for domain_idx in range(self.num_domains):
                for class_idx in range(self.num_classes):
                    self.covariance[domain_idx][class_idx] = self.covariance[domain_idx][class_idx].to(self.device)
                    self.mean[domain_idx][class_idx] = self.mean[domain_idx][class_idx].to(self.device)

        batch_size = features.size(0)
        
        for i in range(batch_size):
            feature = features[i].detach()
            label = labels[i].item()
            domain_id = domain_ids[i].item()
            
            # Update mean
            if self.count[domain_id][label] == 0:
                self.mean[domain_id][label] = feature
                self.count[domain_id][label] = 1
            else:
                self.mean[domain_id][label] = (
                    self.momentum * self.mean[domain_id][label] + 
                    (1 - self.momentum) * feature
                )
            
            # Compute centered feature
            centered = feature - self.mean[domain_id][label]
            
            # Update covariance matrix
            outer_product = torch.outer(centered, centered)
            self.covariance[domain_id][label] = (
                self.momentum * self.covariance[domain_id][label] + 
                (1 - self.momentum) * outer_product
            )
            
            self.count[domain_id][label] += 1
    
    def get_covariance(self, class_idx: int, domain_idx: int) -> torch.Tensor:
        """Get covariance matrix for specific class and domain."""
        return self.covariance[domain_idx][class_idx]
    
    def get_all_domain_covariances(self, class_idx: int) -> List[torch.Tensor]:
        """Get covariance matrices for a class across all domains."""
        return [self.covariance[domain_idx][class_idx] for domain_idx in range(self.num_domains)]


class SemanticAugmentation(nn.Module):
    """
    Semantic Augmentation module implementing ISDA-style augmentation.
    """
    def __init__(self, num_classes: int, feature_dim: int, num_domains: int = 1):
        super(SemanticAugmentation, self).__init__()
        self.num_classes = num_classes
        self.feature_dim = feature_dim
        self.num_domains = num_domains
        
        # Covariance matrix tracker
        self.cov_matrix = CoVarianceMatrix(num_classes, num_domains, feature_dim)
    
    def forward(self, features: torch.Tensor, labels: torch.Tensor, 
                weight: torch.Tensor, domain_ids: torch.Tensor) -> torch.Tensor:
        """
        Compute augmented features using covariance matrices.
        
        Args:
            features: Input features [batch_size, feature_dim]
            labels: Labels [batch_size]
            weight: Weight matrix from classification layer [num_classes, feature_dim]
            domain_ids: Domain IDs [batch_size]
            
        Returns:
            Augmented features [batch_size, feature_dim]
        """
        # Update covariance matrices
        self.cov_matrix.update(features, labels, domain_ids)
        
        batch_size = features.size(0)
        augmented_features = torch.zeros_like(features)
        
        for i in range(batch_size):
            feature = features[i]
            label = labels[i].item()
            domain_id = domain_ids[i].item()
            
            # Get covariance matrix for this class and domain
            sigma = self.cov_matrix.get_covariance(label, domain_id)
            
            # Compute W - W_hat (Equation 2)
            W = weight.t()  # [feature_dim, num_classes]
            W_y = weight[label].unsqueeze(1).repeat(1, self.num_classes)  # [feature_dim, num_classes]
            W_diff = W - W_y  # [feature_dim, num_classes]
            
            # Compute transformation: (W - W_hat)^T @ Sigma @ (W - W_hat)
            # Result shape: [num_classes, num_classes]
            transform = W_diff.t() @ sigma @ W_diff
            
            # Sample from N(0, I)
            noise = torch.randn(self.num_classes, device=features.device)
            
            # Compute sqrt of transformation matrix using Cholesky decomposition
            try:
                L = torch.linalg.cholesky(transform + 1e-6 * torch.eye(self.num_classes, device=features.device))
                augmented_direction = L @ noise
            except:
                # Fallback to eigenvalue decomposition if Cholesky fails
                eigenvalues, eigenvectors = torch.linalg.eigh(transform)
                eigenvalues = torch.clamp(eigenvalues, min=1e-6)
                L = eigenvectors @ torch.diag(torch.sqrt(eigenvalues))
                augmented_direction = L @ noise
            
            # Apply augmentation direction to feature
            augmented_features[i] = feature + W_diff @ augmented_direction
        
        return augmented_features


class CrossSmooth(nn.Module):
    """
    CrossSmooth module for inter-class semantic augmentation.
    Implements Equation 5 from the paper.
    """
    def __init__(self, num_classes: int, feature_dim: int, num_domains: int, lambda_0: float = 1.0):
        super(CrossSmooth, self).__init__()
        self.num_classes = num_classes
        self.feature_dim = feature_dim
        self.num_domains = num_domains
        self.lambda_0 = lambda_0
        
        # All-ones matrix M
        self.register_buffer('M', torch.ones(feature_dim, feature_dim))
        
        # Covariance matrix tracker
        self.cov_matrix = CoVarianceMatrix(num_classes, num_domains, feature_dim)
    
    def forward(self, features: torch.Tensor, labels: torch.Tensor, 
                weight: torch.Tensor, domain_ids: torch.Tensor, lambda_aug: float = 1.0) -> torch.Tensor:
        """
        Apply CrossSmooth augmentation.
        
        Args:
            features: Input features [batch_size, feature_dim]
            labels: Labels [batch_size]
            weight: Weight matrix from classification layer [num_classes, feature_dim]
            domain_ids: Domain IDs [batch_size]
            lambda_aug: Augmentation strength
            
        Returns:
            Augmented features [batch_size, feature_dim]
        """
        # Update covariance matrices
        self.cov_matrix.update(features, labels, domain_ids)
        
        batch_size = features.size(0)
        augmented_features = torch.zeros_like(features)
        
        for i in range(batch_size):
            feature = features[i]
            label = labels[i].item()
            domain_id = domain_ids[i].item()
            
            # Get covariance matrix for this class and domain
            sigma = self.cov_matrix.get_covariance(label, domain_id)
            
            # Add lambda_0 * M to covariance (Equation 5)
            sigma_augmented = sigma + self.lambda_0 * self.M
            
            # Compute W - W_hat
            W = weight.t()  # [feature_dim, num_classes]
            W_y = weight[label].unsqueeze(1).repeat(1, self.num_classes)
            W_diff = W - W_y
            
            # Compute transformation
            transform = W_diff.t() @ sigma_augmented @ W_diff
            
            # Sample and apply augmentation
            noise = torch.randn(self.num_classes, device=features.device)
            
            try:
                L = torch.linalg.cholesky(transform + 1e-6 * torch.eye(self.num_classes, device=features.device))
                augmented_direction = L @ noise
            except:
                eigenvalues, eigenvectors = torch.linalg.eigh(transform)
                eigenvalues = torch.clamp(eigenvalues, min=1e-6)
                L = eigenvectors @ torch.diag(torch.sqrt(eigenvalues))
                augmented_direction = L @ noise
            
            # Apply augmentation with strength lambda_aug / 2
            augmented_features[i] = feature + (lambda_aug / 2) * (W_diff @ augmented_direction)
        
        return augmented_features


class CrossVariance(nn.Module):
    """
    CrossVariance module for inter-domain semantic augmentation.
    Implements Equation 6 from the paper.
    """
    def __init__(self, num_classes: int, feature_dim: int, num_domains: int):
        super(CrossVariance, self).__init__()
        self.num_classes = num_classes
        self.feature_dim = feature_dim
        self.num_domains = num_domains
        
        # Covariance matrix tracker
        self.cov_matrix = CoVarianceMatrix(num_classes, num_domains, feature_dim)
    
    def forward(self, features: torch.Tensor, labels: torch.Tensor, 
                weight: torch.Tensor, domain_ids: torch.Tensor, lambda_aug: float = 1.0) -> torch.Tensor:
        """
        Apply CrossVariance augmentation.
        
        Args:
            features: Input features [batch_size, feature_dim]
            labels: Labels [batch_size]
            weight: Weight matrix from classification layer [num_classes, feature_dim]
            domain_ids: Domain IDs [batch_size]
            lambda_aug: Augmentation strength
            
        Returns:
            Augmented features [batch_size, feature_dim]
        """
        # Update covariance matrices
        self.cov_matrix.update(features, labels, domain_ids)
        
        batch_size = features.size(0)
        augmented_features = torch.zeros_like(features)
        
        for i in range(batch_size):
            feature = features[i]
            label = labels[i].item()
            domain_id = domain_ids[i].item()
            
            # Compute W - W_hat
            W = weight.t()  # [feature_dim, num_classes]
            W_y = weight[label].unsqueeze(1).repeat(1, self.num_classes)
            W_diff = W - W_y
            
            # Sum covariance matrices across all domains (Equation 6)
            sigma_sum = torch.zeros(self.feature_dim, self.feature_dim, device=features.device)
            for k in range(self.num_domains):
                sigma_k = self.cov_matrix.get_covariance(label, k)
                sigma_sum += sigma_k
            
            # Compute transformation
            transform = W_diff.t() @ sigma_sum @ W_diff
            
            # Sample and apply augmentation
            noise = torch.randn(self.num_classes, device=features.device)
            
            try:
                L = torch.linalg.cholesky(transform + 1e-6 * torch.eye(self.num_classes, device=features.device))
                augmented_direction = L @ noise
            except:
                eigenvalues, eigenvectors = torch.linalg.eigh(transform)
                eigenvalues = torch.clamp(eigenvalues, min=1e-6)
                L = eigenvectors @ torch.diag(torch.sqrt(eigenvalues))
                augmented_direction = L @ noise
            
            # Apply augmentation with strength lambda_aug / 2
            augmented_features[i] = feature + (lambda_aug / 2) * (W_diff @ augmented_direction)
        
        return augmented_features


class CDSA(nn.Module):
    """
    Complete CDSA module combining CrossSmooth and CrossVariance.
    Implements Equation 7 from the paper.
    """
    def __init__(self, num_classes: int, feature_dim: int, num_domains: int, lambda_0: float = 1.0):
        super(CDSA, self).__init__()
        self.num_classes = num_classes
        self.feature_dim = feature_dim
        self.num_domains = num_domains
        self.lambda_0 = lambda_0
        
        # Shared covariance matrix tracker
        self.cov_matrix = CoVarianceMatrix(num_classes, num_domains, feature_dim)
        
        # All-ones matrix M for CrossSmooth
        self.register_buffer('M', torch.ones(feature_dim, feature_dim))
    
   
            
    def forward(self, features, labels, weight, domain_ids, lambda_aug=1.0):
        # Update covariance matrices
        self.cov_matrix.update(features, labels, domain_ids)
        
        batch_size = features.size(0)
        augmented_features = torch.zeros_like(features)
        
        for i in range(batch_size):
            feature = features[i]
            label = labels[i].item()
            domain_id = domain_ids[i].item()
            
            # CORRECTED: W and W_diff calculation
            W = weight  # [num_classes, feature_dim]
            W_y = weight[label]  # [feature_dim]
            W_diff = W - W_y  # [num_classes, feature_dim]
            
            # Combine CrossSmooth and CrossVariance (Equation 7)
            sigma_total = torch.zeros(self.feature_dim, self.feature_dim, device=features.device)
            for k in range(self.num_domains):
                sigma_k = self.cov_matrix.get_covariance(label, k)
                # Add lambda_0 * M for inter-class augmentation
                sigma_k_augmented = sigma_k + self.lambda_0 * self.M
                sigma_total += sigma_k_augmented
            
            # Compute transformation: W_diff @ sigma_total @ W_diff.T
            # W_diff: [num_classes, feature_dim]
            # sigma_total: [feature_dim, feature_dim]
            # Result: [num_classes, num_classes]
            transform = W_diff @ sigma_total @ W_diff.T
            
            # Sample and apply augmentation
            noise = torch.randn(self.num_classes, device=features.device)
            
            try:
                L = torch.linalg.cholesky(transform + 1e-6 * torch.eye(self.num_classes, device=features.device))
                augmented_direction = L @ noise
            except:
                eigenvalues, eigenvectors = torch.linalg.eigh(transform)
                eigenvalues = torch.clamp(eigenvalues, min=1e-6)
                L = eigenvectors @ torch.diag(torch.sqrt(eigenvalues))
                augmented_direction = L @ noise
            
            # Apply augmentation: feature + (lambda_aug/2) * (W_diff.T @ augmented_direction)
            # W_diff.T: [feature_dim, num_classes]
            # augmented_direction: [num_classes]
            # Result: [feature_dim]
            augmentation = (lambda_aug / 2) * (W_diff.T @ augmented_direction)
            augmented_features[i] = feature + augmentation
        
        return augmented_features

class SupervisedContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.1):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, features, labels):
        # Normalize features
        features = F.normalize(features, dim=1)
        
        # Compute similarity matrix
        sim_matrix = torch.matmul(features, features.T) / self.temperature
        
        # Mask for positive pairs (same label)
        labels_expanded = labels.unsqueeze(0)
        pos_mask = (labels_expanded == labels_expanded.T).float()
        
        # Remove self-comparisons
        pos_mask.fill_diagonal_(0)
        
        # Compute loss
        exp_sim = torch.exp(sim_matrix)
        log_prob = sim_matrix - torch.log(exp_sim.sum(dim=1, keepdim=True))
        
        # Mean over positive pairs
        loss = -(pos_mask * log_prob).sum(dim=1) / (pos_mask.sum(dim=1) + 1e-8)
        return loss.mean()
        
class SupervisedContrastiveLoss2(nn.Module):
    """
    Supervised Contrastive Learning Loss.
    Implements Equation 4 from the paper.
    """
    def __init__(self, temperature: float = 0.1):
        super(SupervisedContrastiveLoss, self).__init__()
        self.temperature = temperature
    
    def forward(self, features: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        """
        Compute supervised contrastive loss.
        
        Args:
            features: Normalized feature vectors [2*batch_size, feature_dim]
            labels: Labels [2*batch_size]
            
        Returns:
            Loss value
        """
        batch_size = features.size(0)
        
        # Normalize features
        features = F.normalize(features, dim=1)
        
        # Compute similarity matrix
        similarity_matrix = torch.matmul(features, features.T) / self.temperature
        
        # Create mask for positive pairs
        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float()
        
        # Remove diagonal elements (self-similarity)
        logits_mask = torch.ones_like(mask)
        logits_mask.fill_diagonal_(0)
        
        # Apply masks
        mask = mask * logits_mask
        
        # Compute log probabilities
        exp_logits = torch.exp(similarity_matrix) * logits_mask
        log_prob = similarity_matrix - torch.log(exp_logits.sum(1, keepdim=True))
        
        # Compute mean of log-likelihood over positive pairs
        mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-6)
        
        # Compute number of positives per sample
        B_yi = mask.sum(1)
        
        # Loss
        loss = -mean_log_prob_pos
        loss = loss.view(batch_size).mean()
        
        return loss


class FACTAugmentation(nn.Module):
    """
    FACT-style Fourier-based data augmentation.
    This is used as part of the L_FACT loss.
    """
    def __init__(self, alpha: float = 1.0):
        super(FACTAugmentation, self).__init__()
        self.alpha = alpha
    
    def forward(self, images: torch.Tensor, alpha: Optional[float] = None) -> torch.Tensor:
        """
        Apply Fourier-based augmentation.
        
        Args:
            images: Input images [batch_size, channels, height, width]
            alpha: Mixing coefficient (if None, uses self.alpha)
            
        Returns:
            Augmented images
        """
        if alpha is None:
            alpha = self.alpha
        
        batch_size = images.size(0)
        
        # Get random permutation
        indices = torch.randperm(batch_size)
        
        # Apply FFT
        fft_1 = torch.fft.fft2(images, dim=(-2, -1))
        fft_2 = torch.fft.fft2(images[indices], dim=(-2, -1))
        
        # Extract amplitude and phase
        amp_1, phase_1 = torch.abs(fft_1), torch.angle(fft_1)
        amp_2, phase_2 = torch.abs(fft_2), torch.angle(fft_2)
        
        # Mix amplitudes
        amp_mixed = alpha * amp_1 + (1 - alpha) * amp_2
        
        # Reconstruct with original phase
        fft_mixed = amp_mixed * torch.exp(1j * phase_1)
        
        # Apply inverse FFT
        images_mixed = torch.fft.ifft2(fft_mixed, dim=(-2, -1)).real
        
        return images_mixed


class CDSANetwork(nn.Module):
    """
    Complete CDSA network combining backbone, augmentation, and loss computation.
    """
    def __init__(self, backbone: nn.Module, num_classes: int, feature_dim: int, 
                 num_domains: int, lambda_0: float = 1.0, lambda_aug: float = 1.0,
                 beta: float = 1.0, temperature: float = 0.1):
        super(CDSANetwork, self).__init__()
        
        self.backbone = backbone
        self.num_classes = num_classes
        self.feature_dim = feature_dim
        self.num_domains = num_domains
        self.lambda_aug = lambda_aug
        self.beta = beta
        
        # CDSA module
        self.cdsa = CDSA(num_classes, feature_dim, num_domains, lambda_0)
        
        # Classification head
        self.classifier = nn.Linear(feature_dim, num_classes)
        
        # Losses
        self.contrastive_loss = SupervisedContrastiveLoss(temperature)
        self.ce_loss = nn.CrossEntropyLoss()
        
        # FACT augmentation
        self.fact_aug = FACTAugmentation()
    
    def forward(self, images: torch.Tensor, labels: torch.Tensor, 
                domain_ids: torch.Tensor, training: bool = True) -> Dict[str, torch.Tensor]:
        """
        Forward pass.
        
        Args:
            images: Input images
            labels: Ground truth labels
            domain_ids: Domain IDs
            training: Whether in training mode
            
        Returns:
            Dictionary containing logits and losses
        """
        # Extract features
        features = self.backbone(images)
        
        # Get classification weight
        weight = self.classifier.weight  # [num_classes, feature_dim]
        
        if training:
            # Apply CDSA augmentation
            features_aug = self.cdsa(features, labels, weight, domain_ids, self.lambda_aug)
            
            # Classification on augmented features
            logits_aug = self.classifier(features_aug)
            
            # Compute contrastive loss (L_CSCV_CL)
            # Stack original and augmented features
            features_combined = torch.cat([features, features_aug], dim=0)
            labels_combined = torch.cat([labels, labels], dim=0)
            loss_cl = self.contrastive_loss(features_combined, labels_combined)
            
            # Classification loss
            logits = self.classifier(features)
            loss_ce = self.ce_loss(logits, labels)
            loss_ce_aug = self.ce_loss(logits_aug, labels)
            
            # Total loss (Equation 8): L_CDSA = L_FACT + β * L_CSCV_CL
            # L_FACT includes both CE losses and potentially co-teacher regularization
            loss_fact = loss_ce + loss_ce_aug
            loss_total = loss_fact + self.beta * loss_cl
            
            return {
                'logits': logits,
                'logits_aug': logits_aug,
                'loss_total': loss_total,
                'loss_ce': loss_ce,
                'loss_ce_aug': loss_ce_aug,
                'loss_cl': loss_cl,
                'loss_fact': loss_fact
            }
        else:
            # Inference mode
            logits = self.classifier(features)
            return {'logits': logits}


# Example usage
def create_cdsa_model(backbone_name: str = 'resnet18', num_classes: int = 7, 
                      num_domains: int = 3, pretrained: bool = True) -> CDSANetwork:
    """
    Create a CDSA model with specified backbone.
    
    Args:
        backbone_name: Name of the backbone ('resnet18', 'resnet50', etc.)
        num_classes: Number of classes
        num_domains: Number of source domains
        pretrained: Whether to use pretrained weights
        
    Returns:
        CDSANetwork model
    """
    import torchvision.models as models
    
    # Create backbone
    if backbone_name == 'resnet18':
        backbone = models.resnet18(pretrained=pretrained)
        feature_dim = 512
    elif backbone_name == 'resnet50':
        backbone = models.resnet50(pretrained=pretrained)
        feature_dim = 2048
    else:
        raise ValueError(f"Unsupported backbone: {backbone_name}")
    
    # Remove the final classification layer
    backbone = nn.Sequential(*list(backbone.children())[:-1])
    
    # Add adaptive pooling and flatten
    class BackboneWithPooling(nn.Module):
        def __init__(self, backbone, feature_dim):
            super().__init__()
            self.backbone = backbone
            self.pool = nn.AdaptiveAvgPool2d((1, 1))
            self.feature_dim = feature_dim
        
        def forward(self, x):
            x = self.backbone(x)
            x = self.pool(x)
            x = x.view(x.size(0), -1)
            return x
    
    backbone = BackboneWithPooling(backbone, feature_dim)
    
    # Create CDSA network
    model = CDSANetwork(
        backbone=backbone,
        num_classes=num_classes,
        feature_dim=feature_dim,
        num_domains=num_domains,
        lambda_0=1.0,  # Default value, adjust based on validation
        lambda_aug=1.0,
        beta=1.0,
        temperature=0.1
    )
    
    return model


# if __name__ == "__main__":
#     # Example: Create model and test forward pass
#     model = create_cdsa_model('resnet18', num_classes=7, num_domains=3)
#     model = model.cuda()
    
#     # Dummy input
#     batch_size = 16
#     images = torch.randn(batch_size, 3, 224, 224).cuda()
#     labels = torch.randint(0, 7, (batch_size,)).cuda()
#     domain_ids = torch.randint(0, 3, (batch_size,)).cuda()
    
#     # Forward pass
#     model.train()
#     outputs = model(images, labels, domain_ids, training=True)
    
#     print("Training outputs:")
#     print(f"Total loss: {outputs['loss_total'].item():.4f}")
#     print(f"CE loss: {outputs['loss_ce'].item():.4f}")
#     print(f"CE aug loss: {outputs['loss_ce_aug'].item():.4f}")
#     print(f"Contrastive loss: {outputs['loss_cl'].item():.4f}")
    
#     # Inference
#     model.eval()
#     with torch.no_grad():
#         outputs = model(images, labels, domain_ids, training=False)
#         print(f"\nInference logits shape: {outputs['logits'].shape}")

## Fond

In [33]:
    

class AbstractXDom(ERM):
    def __init__(self, input_shape, num_classes, num_domains, hparams):
        super(AbstractXDom, self).__init__(
            input_shape, num_classes, num_domains, hparams
        )

        self.domain_relations = hparams.get("domain_relations", None)

        self.temperature = hparams["temperature"]
        self.base_temperature = hparams["base_temperature"]

        encoder_output = 512 if hparams["resnet18"] else 2048
        self.projector = nn.Sequential(
            nn.Linear(encoder_output, encoder_output),
            nn.ReLU(),
            nn.Linear(encoder_output, 256),
        )

        def weight_init(m):
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

        self.projector.apply(weight_init)

        self.optimizer = torch.optim.Adam(
            (
                list(self.featurizer.parameters())
                + list(self.classifier.parameters())
                + list(self.projector.parameters())
            ),
            lr=self.hparams["lr"],
            weight_decay=self.hparams["weight_decay"],
        )

    def get_masks(self, Y, D):
        """
        Generate masks relating samples and their domains/classes
        """
        # mask-out self-contrast cases
        self_mask = (~torch.eye(Y.shape[0], dtype=torch.bool)).to(Y.device)

        # mask out dot products between different classes
        same_Y_mask = torch.eq(Y.view(-1, 1), Y.view(-1, 1).T)
        # mask out dot products between different domains
        same_D_mask = torch.eq(D.view(-1, 1), D.view(-1, 1).T)

        return {
            "self_mask": self_mask,
            "same_class_mask": same_Y_mask,
            "same_class_exclude_self_mask": same_Y_mask * self_mask,
            "same_domain_mask": same_D_mask,
            "diff_domain_mask": ~same_D_mask,
            "diff_domain_same_class_mask": same_Y_mask * ~same_D_mask,
            "same_domain_diff_class_mask": ~same_Y_mask * same_D_mask,
            "same_domain_same_class_mask": same_Y_mask * same_D_mask,
        }

    def supcon_loss(
        self,
        projections,
        positive_mask,
        negative_mask,
        alpha: torch.Tensor,
        beta: torch.Tensor,
        epsilon: float = 1e-6,
    ):
        """
        Regular FOND_FBA Loss with custom masks for positive and A(i) "negative" samples
        """

        mean_positives_per_sample = (
            torch.count_nonzero(positive_mask) / projections.shape[0]
        )

        # count the number of samples with no positives
        num_zero_positives = projections.shape[0] - torch.count_nonzero(
            positive_mask.sum(1)
        )

        # proj_dot is cos similarity b/c features are normalized
        # find the dot product with respect to every x
        proj_dot = torch.div(torch.matmul(projections, projections.T), self.temperature)

        # for numeric stability so sum is never zero
        logits_max, _ = torch.max(proj_dot, dim=1, keepdim=True)
        logits = proj_dot - logits_max.detach()

        # compute exp per element (i.e. over each cosine similarity)
        exp_logits = torch.exp(logits) * negative_mask * beta
        # weigh intra domain negatives higher = same domain different class

        # decompose log(exp(x)/y) = x - log(y)
        # y = summation of cos similarities excluding self
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True))

        # compute mean of log-likelihood over positives
        mean_log_prob_pos = (positive_mask * alpha * log_prob).sum(1) / (
            positive_mask.sum(1) + epsilon
        )

        loss = -(self.temperature / self.base_temperature) * mean_log_prob_pos
        loss = loss.mean()

        return loss, mean_positives_per_sample, num_zero_positives

    def preprocess(self, minibatches):
        # NOTE: current implementations doesn't create duplicates
        # check SelfReg

        num_domains = len(minibatches)
        features = [self.featurizer(xi) for xi, _ in minibatches]

        projections = [F.normalize(self.projector(fi)) for fi in features]
        classifs = [self.classifier(fi) for fi in features]
        targets = [yi for _, yi in minibatches]

        # create domain labels
        domains = [
            torch.zeros(len(x), dtype=torch.long).to(x.device) + i
            for i, x in enumerate(targets)
        ]

        # match domains
        if self.domain_relations is not None:
            for old, new in self.domain_relations.items():
                for d in domains:
                    d[d == old] = new

        return {
            "features": features,
            "projections": projections,
            "classifs": classifs,
            "targets": targets,
            "domains": domains,
            "num_domains": num_domains,
        }

    def update(self, minibatches, unlabeled=None):
        raise NotImplementedError()


class FONDBase(AbstractXDom):
    def __init__(self, input_shape, num_classes, num_domains, hparams):
        super(FONDBase, self).__init__(input_shape, num_classes, num_domains, hparams)

        # hparams
        self.xdom_lmbd = hparams["xdom_lmbd"]
        self.error_lmbd = hparams["error_lmbd"]
        self.xda_alpha = hparams["xda_alpha"]
        self.xda_beta = hparams["xda_beta"]
        self.C_oc = hparams["C_oc"]

        # create class masks
        oc_weight = torch.zeros(num_classes, dtype=torch.bool)
        oc_weight[self.C_oc] = True
        noc_weight = ~oc_weight
        self.oc_weight = oc_weight.type(torch.float)
        self.noc_weight = noc_weight.type(torch.float)

    def update(self, minibatches, unlabeled=None):
        values = self.preprocess(minibatches)

        domains = torch.cat(values["domains"])
        targets = torch.cat(values["targets"])
        classifs = torch.cat(values["classifs"])
        projections = torch.cat(values["projections"])

        masks = self.get_masks(Y=targets, D=domains)

        alpha = (
            ~masks["diff_domain_same_class_mask"]
            + masks["diff_domain_same_class_mask"] * self.xda_alpha
        )

        beta = (
            ~masks["same_domain_diff_class_mask"]
            + masks["same_domain_diff_class_mask"] * self.xda_beta
        )

        xdom_loss, mean_positives_per_sample, num_zero_positives = self.supcon_loss(
            projections=projections,
            positive_mask=masks["same_class_mask"] * masks["self_mask"],
            negative_mask=masks["self_mask"],
            alpha=alpha,
            beta=beta,
        )

        oc_class_loss = F.cross_entropy(
            classifs, targets, weight=self.oc_weight.to(targets.device)
        )
        noc_class_loss = F.cross_entropy(
            classifs, targets, weight=self.noc_weight.to(targets.device)
        )
        error_loss = oc_class_loss - noc_class_loss
        if torch.isnan(error_loss):
            error_loss = torch.tensor(0).to(targets.device)

        class_loss = F.cross_entropy(classifs, targets)

        loss = (
            class_loss
            + self.xdom_lmbd * xdom_loss
            + self.error_lmbd * torch.abs(error_loss)
        )

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return {
            "loss": loss.item(),
            "class_loss": class_loss.item(),
            "xdom_loss": xdom_loss.item(),
            "error_loss": error_loss.item(),
            "mean_p": mean_positives_per_sample.item(),
            "zero_p": num_zero_positives.item(),
        }


class FOND(FONDBase):
    def __init__(self, input_shape, num_classes, num_domains, hparams):
        super(FOND, self).__init__(input_shape, num_classes, num_domains, hparams)


class FOND_NC(FONDBase):
    """
    Based on FOND however we replace the fairness loss with the domain-linked
    classification loss
    """

    def __init__(self, input_shape, num_classes, num_domains, hparams):
        super(FOND_NC, self).__init__(input_shape, num_classes, num_domains, hparams)

    def update(self, minibatches, unlabeled=None):
        values = self.preprocess(minibatches)

        domains = torch.cat(values["domains"])
        targets = torch.cat(values["targets"])
        classifs = torch.cat(values["classifs"])
        projections = torch.cat(values["projections"])

        masks = self.get_masks(Y=targets, D=domains)

        alpha = (
            ~masks["diff_domain_same_class_mask"]
            + masks["diff_domain_same_class_mask"] * self.xda_alpha
        )

        beta = (
            ~masks["same_domain_diff_class_mask"]
            + masks["same_domain_diff_class_mask"] * self.xda_beta
        )

        xdom_loss, mean_positives_per_sample, num_zero_positives = self.supcon_loss(
            projections=projections,
            positive_mask=masks["same_class_mask"] * masks["self_mask"],
            negative_mask=masks["self_mask"],
            alpha=alpha,
            beta=beta,
        )

        oc_class_loss = F.cross_entropy(
            classifs, targets, weight=self.oc_weight.to(targets.device)
        )
        noc_class_loss = F.cross_entropy(
            classifs, targets, weight=self.noc_weight.to(targets.device)
        )
        error_loss = oc_class_loss - noc_class_loss
        if torch.isnan(error_loss):
            error_loss = torch.tensor(0).to(targets.device)
        class_loss = F.cross_entropy(classifs, targets)
        if torch.isnan(noc_class_loss):
            noc_class_loss = torch.tensor(0).to(targets.device)

        loss = (
            class_loss + self.xdom_lmbd * xdom_loss + self.error_lmbd * noc_class_loss
        )

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return {
            "loss": loss.item(),
            "class_loss": class_loss.item(),
            "xdom_loss": xdom_loss.item(),
            "error_loss": error_loss.item(),
            "noc_class_loss": noc_class_loss.item(),
            "mean_p": mean_positives_per_sample.item(),
            "zero_p": num_zero_positives.item(),
        }


# FOND with a non overlapping class loss, but no overall class loss
class FOND_N(FONDBase):
    """
    Guiding Question: Why optimize for domain-shared class accuracy if
    we are only interested in domain-linked classes? Does including domain-shared
    classes for the contrastive objective only improve domain-linked class
    performance?

    Based on FOND however we keep the domain-shared and domain-linked
    contrastive loss and only optimize for the domain-linked classification
    loss.
    """

    def __init__(self, input_shape, num_classes, num_domains, hparams):
        super(FOND_N, self).__init__(input_shape, num_classes, num_domains, hparams)

    def update(self, minibatches, unlabeled=None):
        values = self.preprocess(minibatches)

        domains = torch.cat(values["domains"])
        targets = torch.cat(values["targets"])
        classifs = torch.cat(values["classifs"])
        projections = torch.cat(values["projections"])

        masks = self.get_masks(Y=targets, D=domains)

        alpha = (
            ~masks["diff_domain_same_class_mask"]
            + masks["diff_domain_same_class_mask"] * self.xda_alpha
        )

        beta = (
            ~masks["same_domain_diff_class_mask"]
            + masks["same_domain_diff_class_mask"] * self.xda_beta
        )

        xdom_loss, mean_positives_per_sample, num_zero_positives = self.supcon_loss(
            projections=projections,
            positive_mask=masks["same_class_mask"] * masks["self_mask"],
            negative_mask=masks["self_mask"],
            alpha=alpha,
            beta=beta,
        )

        oc_class_loss = F.cross_entropy(
            classifs, targets, weight=self.oc_weight.to(targets.device)
        )
        noc_class_loss = F.cross_entropy(
            classifs, targets, weight=self.noc_weight.to(targets.device)
        )
        error_loss = oc_class_loss - noc_class_loss
        if torch.isnan(error_loss):
            error_loss = torch.tensor(0).to(targets.device)
        class_loss = F.cross_entropy(classifs, targets)
        if torch.isnan(noc_class_loss):
            noc_class_loss = torch.tensor(0).to(targets.device)

        loss = self.xdom_lmbd * xdom_loss + noc_class_loss

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return {
            "loss": loss.item(),
            "class_loss": class_loss.item(),
            "xdom_loss": xdom_loss.item(),
            "error_loss": error_loss.item(),
            "noc_class_loss": noc_class_loss.item(),
            "mean_p": mean_positives_per_sample.item(),
            "zero_p": num_zero_positives.item(),
        }


## Fond_ADV

In [34]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GradientReversalLayer(torch.autograd.Function):
    """
    Gradient Reversal Layer (GRL).
    Forward pass: identity function.
    Backward pass: reverses the gradient sign and scales it by lambda.
    """
    @staticmethod
    def forward(ctx, x, lambda_grl):
        ctx.lambda_grl = lambda_grl
        return x.clone()

    @staticmethod
    def backward(ctx, grad_output):
        # Reverse and scale the gradient
        lambda_grl = ctx.lambda_grl
        return grad_output.neg() * lambda_grl, None

class AdversarialDomainClassifier(nn.Module):
    """
    Domain Classifier with integrated Gradient Reversal.
    """
    def __init__(self, feature_dim, num_domains, hidden_dim=256):
        super().__init__()
        self.grl = GradientReversalLayer.apply
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, num_domains)
        )

    def forward(self, x, lambda_grl=1.0):
        # Apply GRL before the classifier
        x_rev = self.grl(x, lambda_grl)
        return self.classifier(x_rev)

class FOND_ADV(AbstractXDom):
    """
    FOND base class with integrated Domain-Adversarial (DANN) component.
    Inherit from this to add adversarial domain confusion to any FOND variant.
    """
    def __init__(self, input_shape, num_classes, num_domains, hparams):
        super(FOND_ADV, self).__init__(input_shape, num_classes, num_domains, hparams)

        # ==================== ADVERSARIAL SETUP ====================
        # Hyperparameter for the strength of the adversarial objective
        self.adv_lambda = hparams.get("adv_lambda", 0.1)  # Tune: [0.05, 0.2, 0.5]
        # Hyperparameter for the GRL scaling (usually 1.0, can be scheduled)
        self.grl_lambda = hparams.get("grl_lambda", 1.0)

        # Build the domain classifier
        feature_dim = 512 if hparams["resnet18"] else 2048
        self.domain_classifier = AdversarialDomainClassifier(
            feature_dim=feature_dim,
            num_domains=num_domains,
            hidden_dim=256
        )

        # Add the domain classifier's parameters to the optimizer
        self.optimizer = torch.optim.Adam(
            (
                list(self.featurizer.parameters()) +
                list(self.classifier.parameters()) +
                list(self.projector.parameters()) +
                list(self.domain_classifier.parameters())  # NEW
            ),
            lr=self.hparams["lr"],
            weight_decay=self.hparams["weight_decay"],
        )
        # ==================== END ADVERSARIAL SETUP ====================

        # Keep your existing FOND-specific initialization if needed
        # This will be handled by the child class (FOND or FOND_CDSA)
        # that inherits from FOND_ADV

    def compute_adversarial_loss(self, features, domains):
        """
        Computes the domain adversarial loss.
        Returns:
            domain_loss: Cross-entropy loss for domain classification.
            This single loss, when backpropagated through the GRL,
            trains the featurizer adversarially.
        """
        domain_logits = self.domain_classifier(features, lambda_grl=self.grl_lambda)
        domain_loss = F.cross_entropy(domain_logits, domains)
        return domain_loss

    def update(self, minibatches, unlabeled=None):
        """
        Adversarial update method. Child classes should call
        super().update() at the start of their update() method.
        """
        # Preprocess to get features, domains, etc. (from AbstractXDom)
        values = self.preprocess(minibatches)

        # Prepare tensors for the rest of the computation
        self.current_features = torch.cat(values["features"])
        self.current_domains = torch.cat(values["domains"])
        self.current_targets = torch.cat(values["targets"])
        self.current_projections = torch.cat(values["projections"])
        self.current_classifs = torch.cat(values["classifs"])

        # Compute and store adversarial loss
        self.adv_loss = self.compute_adversarial_loss(
            self.current_features,
            self.current_domains
        )

        # Return the preprocessed values for the child class to use
        return values

## Fond_DANN

In [42]:
class FOND_DANN(FOND_ADV):
    """
    Original FOND algorithm with added adversarial domain confusion.
    """
    def __init__(self, input_shape, num_classes, num_domains, hparams):
        # Initialize FOND hyperparameters (xdom_lmbd, error_lmbd, etc.)
        # FOND_ADV.__init__ will handle the adversarial setup
        super(FOND_DANN, self).__init__(input_shape, num_classes, num_domains, hparams)

        # Initialize FOND-specific parameters (from FONDBase)
        self.xdom_lmbd = hparams["xdom_lmbd"]
        self.error_lmbd = hparams["error_lmbd"]
        self.xda_alpha = hparams["xda_alpha"]
        self.xda_beta = hparams["xda_beta"]
        self.C_oc = hparams["C_oc"]

        # Create class masks for OC/NOC weighting
        oc_weight = torch.zeros(num_classes, dtype=torch.bool)
        oc_weight[self.C_oc] = True
        noc_weight = ~oc_weight
        self.oc_weight = oc_weight.type(torch.float)
        self.noc_weight = noc_weight.type(torch.float)

    def update(self, minibatches, unlabeled=None):
        # 1. Let FOND_ADV preprocess and compute adversarial loss
        values = super().update(minibatches, unlabeled)

        # 2. Compute original FOND losses using the stored tensors
        masks = self.get_masks(Y=self.current_targets, D=self.current_domains)
        alpha = (~masks["diff_domain_same_class_mask"] +
                 masks["diff_domain_same_class_mask"] * self.xda_alpha)
        beta = (~masks["same_domain_diff_class_mask"] +
                masks["same_domain_diff_class_mask"] * self.xda_beta)

        xdom_loss, mean_positives, num_zero = self.supcon_loss(
            projections=self.current_projections,
            positive_mask=masks["same_class_mask"] * masks["self_mask"],
            negative_mask=masks["self_mask"],
            alpha=alpha,
            beta=beta,
        )

        oc_class_loss = F.cross_entropy(
            self.current_classifs, self.current_targets,
            weight=self.oc_weight.to(self.current_targets.device)
        )
        noc_class_loss = F.cross_entropy(
            self.current_classifs, self.current_targets,
            weight=self.noc_weight.to(self.current_targets.device)
        )
        error_loss = oc_class_loss - noc_class_loss
        if torch.isnan(error_loss):
            error_loss = torch.tensor(0).to(self.current_targets.device)

        class_loss = F.cross_entropy(self.current_classifs, self.current_targets)

        # 3. Combine FOND loss with adversarial loss
        fond_loss = (class_loss +
                     self.xdom_lmbd * xdom_loss +
                     self.error_lmbd * torch.abs(error_loss))

        total_loss = fond_loss + self.adv_lambda * self.adv_loss

        # 4. Optimize
        self.optimizer.zero_grad()
        total_loss.backward()
        self.optimizer.step()
        # print(f"adv_loss is: {self.adv_loss.item()}")
        return {
            "loss": total_loss.item(),
            "class_loss": class_loss.item(),
            "xdom_loss": xdom_loss.item(),
            "error_loss": error_loss.item(),
            "adv_loss": self.adv_loss.item(),
            "mean_p": mean_positives.item(),
            "zero_p": num_zero.item(),
        }

## Fond_CDSA

In [36]:
class FOND_CDSA(FONDBase):
    def __init__(self, input_shape, num_classes, num_domains, hparams):
        # Instead of: super(FOND_CDSA, self).__init__(...)
        # Use direct call:
        super(FOND_CDSA, self).__init__(input_shape, num_classes, num_domains, hparams)
        
        # Rest of your init code...
        # Get feature dimension
        encoder_output = 512 if hparams["resnet18"] else 2048
        
        # Use Claude's CDSA exactly
        self.cdsa_module = CDSA(
            num_classes=num_classes,
            feature_dim=encoder_output,
            num_domains=num_domains,
            lambda_0=hparams.get("cdsa_lambda_0", 1.0)
        )
        
        # Use Claude's supervised contrastive loss
        self.cdsa_contrastive = SupervisedContrastiveLoss(
            temperature=hparams.get("cdsa_temperature", 0.1)
        )
        
        # CDSA hyperparameters from paper
        self.cdsa_lambda = hparams.get("cdsa_lambda", 0.5)
        self.cdsa_beta = hparams.get("cdsa_beta", 1.0)
    
  
     
    def update(self, minibatches, unlabeled=None):
        # Get features
        values = self.preprocess(minibatches)
        
        domains = torch.cat(values["domains"])
        targets = torch.cat(values["targets"])
        features = torch.cat(values["features"])
        
        # Apply CDSA to all features
        augmented_features = self.cdsa_module(
            features=features,
            labels=targets,
            weight=self.classifier.weight,
            domain_ids=domains,
            lambda_aug=self.cdsa_lambda
        )
        
        # Compute supervised contrastive loss (L_CL^{CSCV})
        features_combined = torch.cat([features, augmented_features], dim=0)
        labels_combined = torch.cat([targets, targets], dim=0)
        cdsa_contrastive_loss = self.cdsa_contrastive(features_combined, labels_combined)
        
        # Use augmented features for projections and classification
        projections = F.normalize(self.projector(augmented_features))
        classifs = self.classifier(augmented_features)
        
        # Compute masks
        masks = self.get_masks(Y=targets, D=domains)
        
        alpha = (
            ~masks["diff_domain_same_class_mask"]
            + masks["diff_domain_same_class_mask"] * self.xda_alpha
        )
        
        beta = (
            ~masks["same_domain_diff_class_mask"]
            + masks["same_domain_diff_class_mask"] * self.xda_beta
        )
        
        xdom_loss, mean_positives_per_sample, num_zero_positives = self.supcon_loss(
            projections=projections,
            positive_mask=masks["same_class_mask"] * masks["self_mask"],
            negative_mask=masks["self_mask"],
            alpha=alpha,
            beta=beta,
        )
        
        oc_class_loss = F.cross_entropy(
            classifs, targets, weight=self.oc_weight.to(targets.device)
        )
        noc_class_loss = F.cross_entropy(
            classifs, targets, weight=self.noc_weight.to(targets.device)
        )
        error_loss = oc_class_loss - noc_class_loss
        if torch.isnan(error_loss):
            error_loss = torch.tensor(0).to(targets.device)
        
        class_loss = F.cross_entropy(classifs, targets)
        
        # Base FOND loss
        base_loss = (
            class_loss
            + self.xdom_lmbd * xdom_loss
            + self.error_lmbd * torch.abs(error_loss)
        )
        
        # Total loss: L_CDSA = L_FACT + β * L_CL^{CSCV}
        total_loss = base_loss + self.cdsa_beta * cdsa_contrastive_loss
        
        self.optimizer.zero_grad()
        total_loss.backward()
        self.optimizer.step()
        
        return {
            "loss": total_loss.item(),
            "base_loss": base_loss.item(),
            "cdsa_loss": cdsa_contrastive_loss.item(),
            "class_loss": class_loss.item(),
            "xdom_loss": xdom_loss.item(),
            "error_loss": error_loss.item(),
            "mean_p": mean_positives_per_sample.item(),
            "zero_p": num_zero_positives.item(),
        }

## Init

In [37]:


ALGORITHMS = {
    "ERM": ERM,
    "FOND": FOND,
    "FOND_CDSA": FOND_CDSA,
    "FOND_DANN": FOND_DANN
    # "FOND_NC": "FOND_NC",
    # "FOND_N": "FOND_N",

    # "FOND_Distillation_Separate_Projector": "FOND_Distillation_Separate_Projector",
    # "FOND_Distillation_Teacher_Projector": "FOND_Distillation_Teacher_Projector",
    # "FOND_Distillation_Student_Projector": "FOND_Distillation_Student_Projector",
}


# Train

## Fit Simple

In [43]:

def fit_simple(
    exp_dir: str,
    logger,  # CSVLogger or PrintLogger
    seed: int,
    trial_seed: int,
    hparams_seed: int,
    algorithm_name: str,
    dataset_name: str,
    data_dir: str,
    num_workers: int,
    test_envs: list,
    overlap_type: str,
    holdout_fraction: float = 0.2,
    n_steps: int = 5001,  
    checkpoint_freq: int = 300,  
    model_checkpoint: Optional[Dict] = None,
    teacher_paths: Optional[Dict] = None,
    num_domain_linked_classes: Optional[int] = None,
    num_classes: Optional[int] = None,
    auto_augment: bool = False,
    augment_search_epochs: int = 10,
):
    # seed everything
    L.seed_everything(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # get device
    if torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"
    
    # Setup hyper parameters
    if hparams_seed == 0:
        hparams = default_hparams(algorithm_name, dataset_name)
    else:
        hparams = random_hparams(
            algorithm_name, dataset_name, seed_hash(hparams_seed, trial_seed)
        )
    logging.info(f"hparams: {hparams}")

    # Add these lines:
    # In your hparams configuration
# Use THESE EXACT parameters from paper
    hparams.update({
        'cdsa_lambda': 0.1,      # λ in paper (base strength)
        'cdsa_lambda_0': 1.5,    # λ₀ for CrossSmooth (start with 0.5, tune in {0.5,1.0,1.5,2.0})
        'cdsa_beta': 1.0,        # β in paper (always 1.0)
    })
    if "cdsa_y_l_multiplier" in config:
        hparams["cdsa_y_l_multiplier"] = config["cdsa_y_l_multiplier"]
    if "cdsa_lambda" in config:
        hparams["cdsa_lambda"] = config["cdsa_lambda"]
    if "cdsa_lambda_0" in config:
        hparams["cdsa_lambda_0"] = config["cdsa_lambda_0"]
    use_autoaugment = config["auto_augment"]
    # Get dataset
    dataset = DATASETS[dataset_name](
        root=data_dir,
        test_envs=test_envs,
        hparams=hparams,
        overlap_type=overlap_type,
        num_classes=num_classes,
        num_domain_linked_classes=num_domain_linked_classes,
        use_randaugment=use_autoaugment
    )
    
    # get overlapping classes
    hparams["C_oc"] = dataset.overlapping_classes
    logging.info(f"Loaded {dataset_name}")
    
    # Split each env into an 'in-split' and an 'out-split'
    in_splits = []
    out_splits = []
    relative_test_env = None
    log_dir = logger.return_root()
    
    for env_i, env in enumerate(dataset):
        out, in_ = split_dataset(
            env, int(len(env) * holdout_fraction), seed_hash(trial_seed, env_i)
        )
        
        if hparams["class_balanced"]:
            in_weights = make_weights_for_balanced_classes(in_)
            out_weights = make_weights_for_balanced_classes(out)
        else:
            in_weights, out_weights = None, None
        
        in_splits.append((in_, in_weights))
        out_splits.append((out, out_weights))
        
        # determine the relative test env id
        if env.is_test_env:
            relative_test_env = env_i
    
    assert relative_test_env is not None, "No testing domains"
    logging.info(f"test_envs={test_envs}, relative_test_env={relative_test_env}")
    
    # Setup data loaders
    train_loaders = [
        InfiniteDataLoader(
            dataset=env,
            weights=env_weights,
            batch_size=hparams["batch_size"],
            num_workers=num_workers,
        )
        for i, (env, env_weights) in enumerate(in_splits)
        if i != relative_test_env
    ]
    
    eval_loaders = [
        FastDataLoader(
            dataset=env,
            batch_size=hparams["batch_size"],
            num_workers=num_workers,
        )
        for env, _ in (in_splits + out_splits)
    ]
    
    eval_weights = [None for _, weights in (in_splits + out_splits)]
    
    eval_loader_names = ["env{}_in".format(i) for i in range(len(in_splits))]
    eval_loader_names += ["env{}_out".format(i) for i in range(len(out_splits))]
    logging.info(f"Created data loaders:  {eval_loader_names}")
    
    train_minibatches_iterator = zip(*train_loaders)
    steps_per_epoch = min([len(env) / hparams["batch_size"] for env, _ in in_splits])
    
    # Setup the algorithm
    if "distillation" in algorithm_name.lower():
        assert len(test_envs) == 1
        teacher_path = teacher_paths[dataset_name][str(test_envs[0])]
        teacher_algorithm = torch.load(teacher_path, map_location=device)
        
        algorithm = ALGORITHMS[algorithm_name](
            input_shape=dataset.input_shape,
            num_classes=dataset.num_classes,
            num_domains=len(dataset) - len(test_envs),
            hparams=hparams,
            teacher=teacher_algorithm,
        )
    else:
        algorithm = ALGORITHMS[algorithm_name](
            input_shape=dataset.input_shape,
            num_classes=dataset.num_classes,
            num_domains=len(dataset) - len(test_envs),
            hparams=hparams,
        )
    
    algorithm.to(device)
    logging.info(f"Algorithm {algorithm_name} setup")
    
    # Calculate model statistics
    total_params = sum(p.numel() for p in algorithm.parameters())
    trainable_params = sum(p.numel() for p in algorithm.parameters() if p.requires_grad)
    
    # Calculate model size in MB
    param_size = sum(p.nelement() * p.element_size() for p in algorithm.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in algorithm.buffers())
    model_size_mb = (param_size + buffer_size) / (1024 ** 2)
    
    logging.info(f"Total parameters: {total_params:,}")
    logging.info(f"Trainable parameters: {trainable_params:,}")
    logging.info(f"Model size: {model_size_mb:.2f} MB")
    
    # Save model stats
    model_stats = {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "model_size_mb": model_size_mb,
        "dataset": dataset_name,
        "algorithm": algorithm_name,
    }
    
    import json
    with open(os.path.join(log_dir, "model_statistics.json"), "w") as f:
        json.dump(model_stats, f, indent=4)
    
    # Training loop
    logging.info(f"Begining training loop, with {steps_per_epoch} steps per epoch")
    checkpoint_vals = collections.defaultdict(lambda: [])
    
    # track the model checkpoint value
    best_model_checkpoint_value = None
    best_model_checkpoint_path = ""
    
    # get checkpoint parameters
    checkpoint_metric = model_checkpoint["metric"].split("/")
    stage, metric = checkpoint_metric[0], checkpoint_metric[1]
    print(f"Model checkpoint frequency: {checkpoint_freq}")
    
    # Initialize epoch-based tracking
    epoch_history = {
        "train": {"loss": [], "acc": [], "precision": [], "recall": [], "f1": []},
        "val": {"loss": [], "acc": [], "precision": [], "recall": [], "f1": []},
        "test": {"loss": [], "acc": [], "precision": [], "recall": [], "f1": []},
        "epochs": [],
        "epoch_times": [],
    }
    
    # Track training time
    training_start_time = time.time()
    epoch_start_time = time.time()
    current_epoch = 0
    steps_in_current_epoch = 0
    
    for step in tqdm(list(range(n_steps))):
        step_start_time = time.time()
        
        # Get batches
        minibatches_device = [
            (x.to(device), y.to(device)) for x, y in next(train_minibatches_iterator)
        ]
        
        # Perform an update
        step_vals = algorithm.update(minibatches_device, None)
        for key, val in step_vals.items():
            checkpoint_vals[key].append(val)
        
        # Track epochs
        steps_in_current_epoch += 1
        
        # Check if we've completed an epoch
        if steps_in_current_epoch >= steps_per_epoch:
            epoch_time = time.time() - epoch_start_time
            epoch_history["epoch_times"].append(epoch_time)
            current_epoch += 1
            steps_in_current_epoch = 0
            epoch_start_time = time.time()
        
        # log
        if (step % checkpoint_freq == 0) or (step == n_steps - 1):
            results_dict = {
                key: {
                    _key: []
                    for _key in ["acc", "f1", "nacc", "oacc", "recall", "precision", "cm", "loss"]
                    + [f"acc-{i}" for i in range(dataset.num_classes)]
                }
                for key in ["train", "val", "test", "other"]
            }
            
            # Calculate training value averages
            for key, val in checkpoint_vals.items():
                if np.isnan(np.mean(val)):
                    raise Exception(f"{key}: {np.mean(val)}")
                # Store the MEAN of training values, not the entire list
                results_dict["train"][str(key)] = [np.mean(val)]
            
            # Evaluation
            for name, loader, weights in zip(
                eval_loader_names,
                eval_loaders,
                eval_weights,
            ):
                # Compute metrics using existing accuracy function
                (acc, recall, f1, precision, oacc, nacc, per_class_acc, cm) = accuracy(
                    algorithm, loader, weights, device, dataset
                )
                
                # Compute loss separately
                loss = compute_loss(algorithm, loader, device)
                
                # env{domain_idx}_{rest_of_string} is how name is formatted
                domain_idx = int(name[3])
                
                loader_type = ""
                if domain_idx == relative_test_env:
                    if "in" in name:  # Note 'in' is the 80%
                        loader_type = "test"
                    else:
                        loader_type = "other"
                elif "out" in name:  # Note 'out' is the 20%
                    loader_type = "val"
                else:
                    loader_type = "train"
                
                # update results
                results_dict[loader_type]["acc"].append(float(acc))
                results_dict[loader_type]["recall"].append(float(recall))
                results_dict[loader_type]["f1"].append(float(f1))
                results_dict[loader_type]["precision"].append(float(precision))
                results_dict[loader_type]["nacc"].append(float(nacc))
                results_dict[loader_type]["oacc"].append(float(oacc))
                results_dict[loader_type]["cm"].append(cm)
                results_dict[loader_type]["loss"].append(float(loss))
                
                for class_id, class_acc in per_class_acc.items():
                    results_dict[loader_type]["acc-" + str(class_id)].append(class_acc)
            
            # Store epoch-level metrics
            current_epoch_num = step / steps_per_epoch
            epoch_history["epochs"].append(current_epoch_num)
            
            for split in ["train", "val", "test"]:
                if results_dict[split]["acc"]:
                    epoch_history[split]["acc"].append(np.mean(results_dict[split]["acc"]))
                    epoch_history[split]["precision"].append(np.mean(results_dict[split]["precision"]))
                    epoch_history[split]["recall"].append(np.mean(results_dict[split]["recall"]))
                    epoch_history[split]["f1"].append(np.mean(results_dict[split]["f1"]))
                    epoch_history[split]["loss"].append(np.mean(results_dict[split]["loss"]))
                if split == "train" and "adv_loss" in results_dict[split]:
                    tqdm.write(f"  adv_loss: {np.mean(results_dict[split]['adv_loss']):.4f}")
                    if "class_loss" in results_dict[split]:
                        tqdm.write(f"  class_loss: {np.mean(results_dict[split]['class_loss']):.4f}")
                    if "xdom_loss" in results_dict[split]:
                        tqdm.write(f"  xdom_loss: {np.mean(results_dict[split]['xdom_loss']):.4f}")
            
            # log metrics - using tqdm.write to avoid breaking progress bar
            tqdm.write(f"\n=== Step {step} (Epoch {current_epoch_num:.2f}) ===")
            tqdm.write(f"step_time: {time.time() - step_start_time:.4f}s")
            
            for stage in ["train", "val", "test", "other"]:
                if results_dict[stage]["acc"]:
                    tqdm.write(f"{stage.upper()}:")
                    tqdm.write(f"  loss: {np.mean(results_dict[stage]['loss']):.4f}")
                    tqdm.write(f"  acc: {np.mean(results_dict[stage]['acc']):.4f}")
                    tqdm.write(f"  precision: {np.mean(results_dict[stage]['precision']):.4f}")
                    tqdm.write(f"  recall: {np.mean(results_dict[stage]['recall']):.4f}")
                    tqdm.write(f"  f1: {np.mean(results_dict[stage]['f1']):.4f}")
                    tqdm.write(f"  oacc: {np.mean(results_dict[stage]['oacc']):.4f}")
                    tqdm.write(f"  nacc: {np.mean(results_dict[stage]['nacc']):.4f}")
            
            logger.log(results_dict, step)
            
            # Save model checkpoint
            if model_checkpoint is not None:
                # get checkpoint parameters
                checkpoint_metric = model_checkpoint["metric"].split("/")
                maximize = model_checkpoint["maximize"]
                stage, metric = checkpoint_metric[0], checkpoint_metric[1]
                
                # get current value
                current_model_checkpoint_value = np.nanmean(results_dict[stage][metric])
                ckpt_path = os.path.join(log_dir, f"{id}_model_step{step}.ckpt")
                
                if best_model_checkpoint_value == None:
                    best_model_checkpoint_value = current_model_checkpoint_value
                    best_model_checkpoint_path = ckpt_path
                    torch.save(algorithm, ckpt_path)
                elif (
                    best_model_checkpoint_value < current_model_checkpoint_value
                ) and maximize:
                    torch.save(algorithm, ckpt_path)
                    # remove previous checkpoint
                    os.remove(best_model_checkpoint_path)
                    # update
                    logging.info(
                        "Updated {} from {} to {}".format(
                            model_checkpoint["metric"],
                            best_model_checkpoint_value,
                            current_model_checkpoint_value,
                        )
                    )
                    best_model_checkpoint_value = current_model_checkpoint_value
                    best_model_checkpoint_path = ckpt_path
                elif (
                    best_model_checkpoint_value > current_model_checkpoint_value
                ) and not maximize:
                    torch.save(algorithm, ckpt_path)
                    # remove previous checkpoint
                    os.remove(best_model_checkpoint_path)
                    # update
                    logging.info(
                        "Updated {} from {} to {}".format(
                            model_checkpoint["metric"],
                            best_model_checkpoint_value,
                            current_model_checkpoint_value,
                        )
                    )
                    best_model_checkpoint_value = current_model_checkpoint_value
                    best_model_checkpoint_path = ckpt_path
            
            checkpoint_vals = collections.defaultdict(lambda: [])
    
    # TRAINING COMPLETE - NOW PRINT EVERYTHING
    
    # Calculate total training time
    total_training_time = time.time() - training_start_time
    avg_epoch_time = np.mean(epoch_history["epoch_times"]) if epoch_history["epoch_times"] else 0
    
    print(f"\n{'='*60}")
    print(f"TRAINING COMPLETED")
    print(f"{'='*60}")
    print(f"Total training time: {total_training_time:.2f}s ({total_training_time/60:.2f} minutes)")
    print(f"Average time per epoch: {avg_epoch_time:.2f}s")
    print(f"Total epochs: {current_epoch}")
    print(f"{'='*60}\n")
    
    # Save training time statistics
    time_stats = {
        "total_training_time_seconds": total_training_time,
        "total_training_time_minutes": total_training_time / 60,
        "average_epoch_time_seconds": avg_epoch_time,
        "total_epochs": current_epoch,
        "epoch_times": epoch_history["epoch_times"],
    }
    
    with open(os.path.join(log_dir, "training_time_statistics.json"), "w") as f:
        json.dump(time_stats, f, indent=4)
    
    # Print and save final confusion matrices
    print("\n" + "="*60)
    print("FINAL EVALUATION - CONFUSION MATRICES")
    print("="*60)
    
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    for name, loader, weights in zip(eval_loader_names, eval_loaders, eval_weights):
        domain_idx = int(name[3])
        
        (acc, recall, f1, precision, oacc, nacc, per_class_acc, cm) = accuracy(
            algorithm, loader, weights, device, dataset
        )
        loss = compute_loss(algorithm, loader, device)
        
        # Determine split type
        if domain_idx == relative_test_env:
            if "in" in name:
                split_name = "TEST"
            else:
                split_name = "OTHER"
        elif "out" in name:
            split_name = "VAL"
        else:
            split_name = "TRAIN"
        
        print(f"\n{split_name} - {name}:")
        print(f"  Loss: {loss:.4f}")
        print(f"  Accuracy: {acc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
        print(f"  Confusion Matrix:")
        print(cm)
        
        # Save confusion matrix as CSV
        cm_path = os.path.join(log_dir, f"confusion_matrix_{split_name}_{name}.csv")
        np.savetxt(cm_path, cm, delimiter=",", fmt="%d")
        print(f"  Saved CSV to: {cm_path}")
        
        # Plot confusion matrix as heatmap
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, 
                    cbar_kws={'label': 'Count'})
        ax.set_xlabel('Predicted Label', fontsize=12)
        ax.set_ylabel('True Label', fontsize=12)
        ax.set_title(f'Confusion Matrix - {split_name} - {name}', fontsize=14)
        
        # Save confusion matrix plot
        cm_plot_path = os.path.join(log_dir, f"confusion_matrix_{split_name}_{name}.png")
        plt.savefig(cm_plot_path, dpi=300, bbox_inches='tight')
        print(f"  Saved plot to: {cm_plot_path}")
        plt.show()  
        plt.close()
    
    # Generate and save plots
    epochs = epoch_history["epochs"]
    
    # Create figure with subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(f'Training Curves - {dataset_name}', fontsize=16)
    
    # Plot 1: Accuracy
    ax = axes[0, 0]
    if epoch_history["train"]["acc"]:
        ax.plot(epochs, epoch_history["train"]["acc"], label='Train', marker='o', markersize=4)
    if epoch_history["val"]["acc"]:
        ax.plot(epochs, epoch_history["val"]["acc"], label='Val', marker='s', markersize=4)
    if epoch_history["test"]["acc"]:
        ax.plot(epochs, epoch_history["test"]["acc"], label='Test', marker='^', markersize=4)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_title('Accuracy vs Epochs')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Loss
    ax = axes[0, 1]
    if epoch_history["train"]["loss"]:
        ax.plot(epochs, epoch_history["train"]["loss"], label='Train', marker='o', markersize=4)
    if epoch_history["val"]["loss"]:
        ax.plot(epochs, epoch_history["val"]["loss"], label='Val', marker='s', markersize=4)
    if epoch_history["test"]["loss"]:
        ax.plot(epochs, epoch_history["test"]["loss"], label='Test', marker='^', markersize=4)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Loss vs Epochs')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 3: F1 Score
    ax = axes[1, 0]
    if epoch_history["train"]["f1"]:
        ax.plot(epochs, epoch_history["train"]["f1"], label='Train', marker='o', markersize=4)
    if epoch_history["val"]["f1"]:
        ax.plot(epochs, epoch_history["val"]["f1"], label='Val', marker='s', markersize=4)
    if epoch_history["test"]["f1"]:
        ax.plot(epochs, epoch_history["test"]["f1"], label='Test', marker='^', markersize=4)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('F1 Score')
    ax.set_title('F1 Score vs Epochs')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 4: Precision and Recall
    ax = axes[1, 1]
    if epoch_history["val"]["precision"]:
        ax.plot(epochs, epoch_history["val"]["precision"], label='Val Precision', marker='s', linestyle='--', markersize=4)
    if epoch_history["val"]["recall"]:
        ax.plot(epochs, epoch_history["val"]["recall"], label='Val Recall', marker='o', linestyle='--', markersize=4)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Score')
    ax.set_title('Precision & Recall vs Epochs')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save figure
    plot_path = os.path.join(log_dir, f'training_curves_{dataset_name}.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"\nTraining curves saved to: {plot_path}")
    plt.close()
    
    # Create a separate plot for training time per epoch
    if epoch_history["epoch_times"]:
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(range(1, len(epoch_history["epoch_times"]) + 1), 
                epoch_history["epoch_times"], marker='o', markersize=4)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Time (seconds)')
        ax.set_title(f'Training Time per Epoch - {dataset_name}')
        ax.grid(True, alpha=0.3)
        
        # Add average line
        avg_time = np.mean(epoch_history["epoch_times"])
        ax.axhline(y=avg_time, color='r', linestyle='--', 
                   label=f'Average: {avg_time:.2f}s')
        ax.legend()
        
        plt.tight_layout()
        time_plot_path = os.path.join(log_dir, f'epoch_times_{dataset_name}.png')
        plt.savefig(time_plot_path, dpi=300, bbox_inches='tight')
        print(f"Epoch time plot saved to: {time_plot_path}")
        plt.close()
    
    # Save epoch history for later analysis
    with open(os.path.join(log_dir, "epoch_history.json"), "w") as f:
        history_serializable = {
            "epochs": epoch_history["epochs"],
            "epoch_times": epoch_history["epoch_times"],
        }
        for split in ["train", "val", "test"]:
            history_serializable[split] = {}
            for metric in ["loss", "acc", "precision", "recall", "f1"]:
                history_serializable[split][metric] = [
                    float(x) if not np.isnan(x) else None 
                    for x in epoch_history[split][metric]
                ]
        json.dump(history_serializable, f, indent=4)
    
    print(f"\nAll results saved to: {log_dir}")
    print(f"\nSummary of saved files:")
    print(f"  - model_statistics.json")
    print(f"  - training_time_statistics.json")
    print(f"  - epoch_history.json")
    print(f"  - training_curves_{dataset_name}.png")
    print(f"  - epoch_times_{dataset_name}.png")
    print(f"  - confusion_matrix_*.csv (for all splits)")
    print(f"  - confusion_matrix_*.png (heatmaps for all splits)")
# In your config dictionary, or in the hparams you pass to the algorithm, ADD:
config["temperature"] = 0.1  # A typical default value
config["base_temperature"] = 0.1  # Often the same as temperature

# Train AutoFOND

## OfficeHome

In [39]:
output_dir = "/kaggle/working/experiments"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
exp_name_oh = input()
config["name"] = exp_name_oh

exp_name_oh = config["name"] or f"{config['algo']}_{config['dataset']}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
exp_dir = Path(output_dir) / exp_name_oh
exp_dir.mkdir(parents=True, exist_ok=True)
log_dir = exp_dir / 'logs'
log_dir.mkdir(parents=True, exist_ok=True)
# Setup logging
config_logging()
logging.info(f"Starting experiment: {exp_name_oh}")
logging.info(f"Output directory: {exp_dir}")

# Save config to experiment directory
with open(exp_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)


logger = CSVLogger(log_dir / 'metrics.csv',  root_dir=log_dir)


 exp_officehome_adv_1


In [40]:
dataset_name = config["dataset"][2]
dataset_name

'OfficeHome'

In [41]:


fit_simple(
exp_dir=str(exp_dir),
logger=logger,
seed=config.get("seed", 0),
trial_seed=config.get("trial_seed", 0),
hparams_seed=config.get("hparams_seed", 0),
algorithm_name=config["algo"],
dataset_name=dataset_name,
data_dir=config["data_dir"][dataset_name],
num_workers=config.get("num_workers", 4),
test_envs=[config["test_set_id"]],
overlap_type=config.get("overlap", "none"),
holdout_fraction=config.get("holdout_fraction", 0.2),
n_steps=config.get("n_epochs"),  # Changed to epochs!
checkpoint_freq=config.get("checkpoint_freq"),  # In epochs
model_checkpoint=config.get("model_checkpoint", {
    "metric": "val/acc",
    "maximize": True
}),
num_domain_linked_classes=config.get("num_domain_linked_classes"),
num_classes=config.get("num_classes"),
# AutoAugmentation specific
auto_augment=config.get("auto_augment", False),
augment_search_epochs=config.get("augment_search_epochs", 10),
)

logging.info(f"Experiment complete. Results saved to {exp_dir}")
print(f"\n✓ Experiment complete!")
print(f"  Results: {exp_dir / 'metrics_oh.csv'}")
print(f"  Best model: {exp_dir / 'best_model_oh.ckpt'}")
print(f"  Config: {exp_dir / 'config_oh.json'}")



import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image
import os

log_dir = f"/kaggle/working/experiments/{exp_name_oh}/logs"

# Get all confusion matrix plot files
cm_files = [f for f in os.listdir(log_dir) if f.startswith('confusion_matrix_') and f.endswith('.png')]

# Display all confusion matrices
print("CONFUSION MATRICES:")
print("="*60)
for cm_file in sorted(cm_files):
    print(f"\n{cm_file}")
    display(Image(filename=os.path.join(log_dir, cm_file)))

# Display training curves
print("\n\nTRAINING CURVES:")
print("="*60)
display(Image(filename=os.path.join(log_dir, 'training_curves_OfficeHome.png')))





Seed set to 0


DEBUG: overlap_type=high
DEBUG: domain_class_filter=[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37], [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43], [27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64]]
DEBUG: len(domain_class_filter)=3
DEBUG: test_envs=[0]
num_envs=4, num_filters=3, domain_class_filter=[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37], [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43], [27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,

  0%|          | 0/5001 [02:38<?, ?it/s]


=== Step 0 (Epoch 0.00) ===
step_time: 158.5509s
TRAIN:
  loss: 5.2388
  acc: 0.0072
  precision: 0.0081
  recall: 0.0072
  f1: 0.0048
  oacc: 0.0078
  nacc: 0.0039
VAL:
  loss: 4.3776
  acc: 0.0103
  precision: 0.0118
  recall: 0.0103
  f1: 0.0072
  oacc: 0.0111
  nacc: 0.0045
TEST:
  loss: 4.4325
  acc: 0.0086
  precision: 0.0029
  recall: 0.0086
  f1: 0.0034
  oacc: 0.0050
  nacc: 0.0141
OTHER:
  loss: 4.3910
  acc: 0.0063
  precision: 0.0007
  recall: 0.0063
  f1: 0.0013
  oacc: 0.0000
  nacc: 0.0154


  6%|▌         | 300/5001 [09:05<55:30,  1.41it/s]   

...


=== Step 3600 (Epoch 59.32) ===
step_time: 149.4503s
TRAIN:
  loss: 0.5011
  acc: 0.7114
  precision: 0.7212
  recall: 0.7114
  f1: 0.7156
  oacc: 0.7182
  nacc: 0.3169
VAL:
  loss: 0.7707
  acc: 0.6126
  precision: 0.6394
  recall: 0.6126
  f1: 0.6162
  oacc: 0.6203
  nacc: 0.2537
TEST:
  loss: 2.8560
  acc: 0.3926
  precision: 0.4448
  recall: 0.3926
  f1: 0.3738
  oacc: 0.4692
  nacc: 0.2778
OTHER:
  loss: 2.8303
  acc: 0.3970
  precision: 0.4498
  recall: 0.3970
  f1: 0.3689
  oacc: 0.4569
  nacc: 0.3071


 78%|███████▊  | 3901/5001 [1:26:29<13:55:56, 45.60s/it]


=== Step 3900 (Epoch 64.26) ===
step_time: 150.1159s
TRAIN:
  loss: 0.5136
  acc: 0.7654
  precision: 0.7772
  recall: 0.7654
  f1: 0.7695
  oacc: 0.7196
  nacc: 0.3031
VAL:
  loss: 0.8366
  acc: 0.6363
  precision: 0.6634
  recall: 0.6363
  f1: 0.6381
  oacc: 0.6127
  nacc: 0.2435
TEST:
  loss: 3.2406
  acc: 0.3798
  precision: 0.4513
  recall: 0.3798
  f1: 0.3507
  oacc: 0.4904
  nacc: 0.2141
OTHER:
  loss: 3.1128
  acc: 0.3785
  precision: 0.4347
  recall: 0.3785
  f1: 0.3520
  oacc: 0.4528
  nacc: 0.2671


 81%|████████  | 4061/5001 [1:28:45<20:32,  1.31s/it]   


KeyboardInterrupt: 

## PACS

In [ ]:
output_dir = "/kaggle/working/experiments"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
exp_name_pacs = input()
config["name"] = exp_name_pacs

exp_name_pacs = config["name"] or f"{config['algo']}_{config['dataset']}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
exp_dir = Path(output_dir) / exp_name_pacs
exp_dir.mkdir(parents=True, exist_ok=True)
log_dir = exp_dir / 'logs'
log_dir.mkdir(parents=True, exist_ok=True)
# Setup logging
config_logging()
logging.info(f"Starting experiment: {exp_name_pacs}")
logging.info(f"Output directory: {exp_dir}")

# Save config to experiment directory
with open(exp_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)


logger = CSVLogger(log_dir / 'metrics.csv',  root_dir=log_dir)## VLCS

In [ ]:
dataset_name_pacs = config["dataset"][0]
dataset_name_pacs

In [ ]:


fit_simple(
exp_dir=str(exp_dir),
logger=logger,
seed=config.get("seed", 0),
trial_seed=config.get("trial_seed", 0),
hparams_seed=config.get("hparams_seed", 0),
algorithm_name=config["algo"],
dataset_name=dataset_name_pacs,
data_dir=config["data_dir"][dataset_name_pacs],
num_workers=config.get("num_workers", 4),
test_envs=[config["test_set_id"]],
overlap_type=config.get("overlap", "none"),
holdout_fraction=config.get("holdout_fraction", 0.2),
n_steps=config.get("n_epochs"),  # Changed to epochs!
checkpoint_freq=config.get("checkpoint_freq"),  # In epochs​

Draft saved

model_checkpoint=config.get("model_checkpoint", {
    "metric": "val/acc",
    "maximize": True
}),
num_domain_linked_classes=config.get("num_domain_linked_classes"),
num_classes=config.get("num_classes"),
# AutoAugmentation specific
auto_augment=config.get("auto_augment", False),
augment_search_epochs=config.get("augment_search_epochs", 10),
)

logging.info(f"Experiment complete. Results saved to {exp_dir}")
print(f"\n✓ Experiment complete!")
print(f"  Results: {exp_dir / 'metrics_pacs.csv'}")
print(f"  Best model: {exp_dir / 'best_model_pacs.ckpt'}")
print(f"  Config: {exp_dir / 'config_pacs.json'}")





import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image
import os

log_dir = f"/kaggle/working/experiments/{exp_name_pacs}/logs"

# Get all confusion matrix plot files
cm_files = [f for f in os.listdir(log_dir) if f.startswith('confusion_matrix_') and f.endswith('.png')]

# Display all confusion matrices
print("CONFUSION MATRICES:")
print("="*60)
for cm_file in sorted(cm_files):
    print(f"\n{cm_file}")
    display(Image(filename=os.path.join(log_dir, cm_file)))

# Display training curves
print("\n\nTRAINING CURVES:")
print("="*60)
display(Image(filename=os.path.join(log_dir, 'training_curves_PACS.png')))

# # Display epoch times
# print("\n\nEPOCH TIMES:")
# print("="*60)
# display(Image(filename=os.path.join(log_dir, 'epoch_times_PACS.png')))


In [ ]:
import os
import subprocess
from IPython.display import FileLink, display

# Directory to zip
directory_to_zip = "/kaggle/working/experiments/experiment_with_pacs_low"

# Zip file should be OUTSIDE or at parent level
zip_name = "/kaggle/working/experiment_with_pacs_low.zip"

# Create the zip file
command = f"zip -r {zip_name} {directory_to_zip}"
result = subprocess.run(command, shell=True, capture_output=True, text=True)

# Check if successful
if result.returncode == 0:
    print(f"Successfully created {zip_name}")
    # Generate the downloadable link
    display(FileLink(zip_name))
else:
    print(f"Error: {result.stderr}")

## VLCS

In [ ]:
output_dir = "/kaggle/working/experiments"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
exp_name_vlcs = input()
config["name"] = exp_name_vlcs

exp_name_vlcs = config["name"] or f"{config['algo']}_{config['dataset']}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
exp_dir = Path(output_dir) / exp_name_vlcs
exp_dir.mkdir(parents=True, exist_ok=True)
log_dir = exp_dir / 'logs'
log_dir.mkdir(parents=True, exist_ok=True)
# Setup logging
config_logging()
logging.info(f"Starting experiment: {exp_name_vlcs}")
logging.info(f"Output directory: {exp_dir}")

# Save config to experiment directory
with open(exp_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)


logger = CSVLogger(log_dir / 'metrics.csv',  root_dir=log_dir)## VLCS

In [ ]:
dataset_name = config["dataset"][1]
dataset_name

In [ ]:


fit_simple(
exp_dir=str(exp_dir),
logger=logger,
seed=config.get("seed", 0),
trial_seed=config.get("trial_seed", 0),
hparams_seed=config.get("hparams_seed", 0),
algorithm_name=config["algo"],
dataset_name=dataset_name,
data_dir=config["data_dir"][dataset_name],
num_workers=config.get("num_workers", 4),
test_envs=[config["test_set_id"]],
overlap_type=config.get("overlap", "none"),
holdout_fraction=config.get("holdout_fraction", 0.2),
n_steps=config.get("n_epochs"),  # Changed to epochs!
checkpoint_freq=config.get("checkpoint_freq"),  # In epochs
model_checkpoint=config.get("model_checkpoint", {
    "metric": "val/acc",
    "maximize": True
}),
num_domain_linked_classes=config.get("num_domain_linked_classes"),
num_classes=config.get("num_classes"),
# AutoAugmentation specific
auto_augment=config.get("auto_augment", False),
augment_search_epochs=config.get("augment_search_epochs", 10),
)

logging.info(f"Experiment complete. Results saved to {exp_dir}")
print(f"\n✓ Experiment complete!")
print(f"  Results: {exp_dir / 'metrics_vlcs.csv'}")
print(f"  Best model: {exp_dir / 'best_model_vlcs.ckpt'}")
print(f"  Config: {exp_dir / 'config_vlcs.json'}")



import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image
import os

log_dir = f"/kaggle/working/experiments/{exp_name_vlcs}/logs"

# Get all confusion matrix plot files
cm_files = [f for f in os.listdir(log_dir) if f.startswith('confusion_matrix_') and f.endswith('.png')]

# Display all confusion matrices
print("CONFUSION MATRICES:")
print("="*60)
for cm_file in sorted(cm_files):
    print(f"\n{cm_file}")
    display(Image(filename=os.path.join(log_dir, cm_file)))

# Display training curves
print("\n\nTRAINING CURVES:")
print("="*60)
display(Image(filename=os.path.join(log_dir, 'training_curves_VLCS.png')))

In [ ]:
dataset_name = config["dataset"][3]
dataset_name

# Birjis's Tests

## Keep Alive

In [ ]:
import threading
import time
from IPython.display import clear_output
import datetime

class KaggleKeepAlive:
    def __init__(self, interval_minutes=30):
        self.interval = interval_minutes * 60
        self.running = False
        self.thread = None
        self.start_time = datetime.datetime.now()
        
    def _keep_alive(self):
        iteration = 0
        while self.running:
            iteration += 1
            timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            elapsed = datetime.datetime.now() - self.start_time
            hours = elapsed.total_seconds() / 3600
            
            # Print status
            clear_output(wait=True)
            print(f"{'='*70}")
            print(f"⏰ KEEP-ALIVE HEARTBEAT #{iteration}")
            print(f"🕐 Current Time: {timestamp}")
            print(f"⏱️  Elapsed Time: {hours:.2f} hours")
            print(f"📊 Status: Experiments running...")
            print(f"🔄 Next heartbeat: {self.interval//60} minutes")
            print(f"{'='*70}\n")
            
            # Write to file
            with open("heartbeat.log", "a") as f:
                f.write(f"{timestamp} | Heartbeat #{iteration} | Elapsed: {hours:.2f}h\n")
            
            time.sleep(self.interval)
    
    def start(self):
        if not self.running:
            self.running = True
            self.thread = threading.Thread(target=self._keep_alive, daemon=True)
            self.thread.start()
            print("✅ Keep-Alive Started - Your notebook won't timeout!")
            print(f"📍 Heartbeat interval: Every {self.interval//60} minutes\n")
    
    def stop(self):
        self.running = False
        if self.thread:
            self.thread.join(timeout=5)
        elapsed = datetime.datetime.now() - self.start_time
        print(f"\n🛑 Keep-Alive Stopped - Total runtime: {elapsed.total_seconds()/3600:.2f} hours")


## Strategies

In [63]:
"""Experiment Runner - Follows SOLID principles
Allows running either single experiments or full paper-style sweeps
WITHOUT modifying existing code"""
import os
import json
import numpy as np
from typing import Dict, List, Optional, Any
from dataclasses import dataclass, asdict
from abc import ABC, abstractmethod

from dataclasses import dataclass, field

@dataclass
class ExperimentConfig:
    """Immutable experiment configuration"""
    algo: str
    dataset: str
    test_set_id: int
    data_dir: str
    log_dir: str
    n_epochs: int
    checkpoint_freq: int
    holdout_fraction: float
    model_checkpoint: Dict[str, Any]
    overall_seed: int
    trial_id: int
    hparam_id: int
    num_workers: int
    overlap: str
    num_classes: Optional[int]
    num_domain_linked_classes: Optional[int]
    cdsa_y_l_multiplier: float
    auto_augment: bool
    augment_search_epochs: int
    augment_policy_size: int
    augment_num_policies: int
    teacher_paths: Dict
    # Catch-all for any other config fields
    extra_params: Dict[str, Any] = field(default_factory=dict)

class ExperimentStrategy(ABC):
    """Abstract base class for experiment execution strategies"""

    @abstractmethod
    def run(self, base_config: Dict, dataset_index: int = 0) -> List[Dict]:
        """Execute experiment strategy and return list of results

        Args:
            base_config: Your config dict with dataset as list, data_dir as dict
            dataset_index: Which dataset from the list to use
        """
        pass

    @abstractmethod
    def get_description(self) -> str:
        """Get human-readable description of this strategy"""
        pass






class ExperimentRunner:
    """Main runner that delegates to strategies"""

    def __init__(self, strategy: ExperimentStrategy):
        self.strategy = strategy

    def run(self, config: Dict, dataset_index: int = 0) -> List[Dict]:
        """Execute experiment using configured strategy

        Args:
            config: Your config dict with dataset as list
            dataset_index: Which dataset from the list to use (default 0)
        """
        print(f"\nExperiment Strategy: {self.strategy.get_description()}\n")
        return self.strategy.run(config, dataset_index)


# # =============================================================================
# # USAGE EXAMPLES
# # =============================================================================
# if __name__ == "__main__":
#     # Your original config (unchanged)
#     base_config = {
#         "algo": "FOND_DANN",
#         "dataset": ["PACS", "VLCS", "OfficeHome"],  # List of datasets
#         "test_set_id": 0,
#         "data_dir": {  # Dict mapping dataset names to paths
#             "PACS": "/kaggle/input/pacs-dataset/kfold",
#             "VLCS": "/kaggle/input/vlcsdataset/",
#             "OfficeHome": "/kaggle/input/officehome/OfficeHome/",
#         },
#         "log_dir": "./logs",
#         "n_epochs": 5001,
#         "checkpoint_freq": 300,
#         "holdout_fraction": 0.2,
#         "model_checkpoint": {"metric": "val/oacc", "maximize": True},
#         "overall_seed": 1,
#         "trial_id": 0,
#         "hparam_id": 1,
#         "num_workers": 4,
#         "overlap": "high",
#         "num_classes": None,
#         "num_domain_linked_classes": None,
#         "cdsa_y_l_multiplier": 1.0,
#         "auto_augment": True,
#         "augment_search_epochs": 5,
#         "augment_policy_size": 5,
#         "augment_num_policies": 3,
#         "teacher_paths": {}
#     }

#     # Run single experiment on OfficeHome (index 2)
#     print("\n" + "="*60)
#     print("OPTION 1: Single Experiment on OfficeHome")
#     print("="*60)
#     runner = ExperimentRunner(SingleExperimentStrategy())
#     results = runner.run(base_config, dataset_index=2)

#     # Run single experiment on PACS (index 0)
#     print("\n" + "="*60)
#     print("OPTION 2: Single Experiment on PACS")
#     print("="*60)
#     runner = ExperimentRunner(SingleExperimentStrategy())
#     results = runner.run(base_config, dataset_index=0)

#     # Run full paper replication on VLCS (index 1)
#     print("\n" + "="*60)
#     print("OPTION 3: Full Paper Replication on VLCS")
#     print("="*60)
#     runner = ExperimentRunner(PaperReplicationStrategy(num_hparams=5, num_trials=3))
#     results = runner.run(base_config, dataset_index=1)

##  SingleExperimentStrategy

In [73]:
class SingleExperimentStrategy(ExperimentStrategy):
    """Run a single experiment with given configuration"""

    def run(self, base_config: Dict, dataset_index: int = 0) -> List[Dict]:
        """Run single experiment"""

        # Extract dataset name from list using the index
        dataset_name = base_config['dataset'][dataset_index]

        print(f"\n{'='*60}")
        print("STRATEGY: Single Experiment")
        print(f"{'='*60}")
        print(f"Dataset: {dataset_name}")
        print(f"Test domain: {base_config['test_set_id']}")
        print(f"Hparam ID: {base_config['hparam_id']}")
        print(f"Trial ID: {base_config['trial_id']}")
        print(f"{'='*60}\n")


        # Create experiment config object
        config = self._create_config(base_config, dataset_index,
                                     base_config['test_set_id'],
                                     base_config['hparam_id'],
                                     base_config['trial_id'])

        # Run the experiment
        result = self._run_single_experiment(config, fit_simple, CSVLogger)

        return [result]



    def _create_config(self, base_config: Dict, dataset_index: int,
                      test_domain_id: int, hparam_id: int, trial_id: int) -> ExperimentConfig:
        # Extract dataset/path
        dataset_name = base_config['dataset'][dataset_index]
        data_dir_path = base_config['data_dir'][dataset_name]

        config_dict = base_config.copy()
        config_dict['dataset'] = dataset_name
        config_dict['data_dir'] = data_dir_path
        config_dict['test_set_id'] = test_domain_id
        config_dict['hparam_id'] = hparam_id
        config_dict['trial_id'] = trial_id

        # Separate known fields from extra
        known_fields = ExperimentConfig.__dataclass_fields__.keys()
        extra = {k: v for k, v in config_dict.items() if k not in known_fields and k != 'extra_params'}
        config_for_dataclass = {k: v for k, v in config_dict.items() if k in known_fields}
        config_for_dataclass['extra_params'] = extra

        return ExperimentConfig(**config_for_dataclass)
    def _run_single_experiment(self, config: ExperimentConfig, fit_simple, CSVLogger) -> Dict:
        """Execute a single experiment"""

        # Setup logger - FIXED
        log_dir = os.path.join(config.log_dir,
            f"{config.algo}_{config.dataset}_test{config.test_set_id}_h{config.hparam_id}_t{config.trial_id}")
        os.makedirs(log_dir, exist_ok=True)

        # Create CSV file path
        csv_file_path = os.path.join(log_dir, "metrics.csv")
        logger = CSVLogger(csv_file_path, root_dir=log_dir)

        # Run training
        fit_simple(
            exp_dir=log_dir,
            logger=logger,
            seed=config.overall_seed,
            trial_seed=config.trial_id,
            hparams_seed=config.hparam_id,
            algorithm_name=config.algo,
            dataset_name=config.dataset,
            data_dir=config.data_dir,
            num_workers=config.num_workers,
            test_envs=[config.test_set_id],
            overlap_type=config.overlap,
            holdout_fraction=config.holdout_fraction,
            n_steps=config.n_epochs,
            checkpoint_freq=config.checkpoint_freq,
            model_checkpoint=config.model_checkpoint,
            teacher_paths=config.teacher_paths,
            num_domain_linked_classes=config.num_domain_linked_classes,
            num_classes=config.num_classes,
            auto_augment=config.auto_augment,
            augment_search_epochs=config.augment_search_epochs,
        )

        # Load results
        result = self._load_results(log_dir)
        result['config'] = asdict(config)

        return result

    def _load_results(self, log_dir: str) -> Dict:
        """Load results from experiment directory"""
        results = {}

        # Load epoch history if it exists
        epoch_history_path = os.path.join(log_dir, "epoch_history.json")
        if os.path.exists(epoch_history_path):
            with open(epoch_history_path, 'r') as f:
                results['epoch_history'] = json.load(f)

        # Extract final metrics
        if 'epoch_history' in results:
            for split in ['train', 'val', 'test']:
                if split in results['epoch_history']:
                    results[f'{split}_final'] = {
                        metric: values[-1] if values else None
                        for metric, values in results['epoch_history'][split].items()
                    }

        return results

    def get_description(self) -> str:
        return "Single experiment run"


## PaperReplicationStrategy

In [ ]:
class PaperReplicationStrategy(ExperimentStrategy):
    """Replicate the paper's experimental protocol:
    - For each test domain (4 domains total)
    - Run 5 hyperparameter configurations (hparam_id = 0,1,2,3,4)
    - Select best hparam based on validation Y_L accuracy
    - Run 3 trials with best hparam (trial_id = 0,1,2)
    - Average results across 3 trials
    - Report average across all 4 test domains"""

    def __init__(self, num_hparams: int = 5, num_trials: int = 3):
        self.num_hparams = num_hparams
        self.num_trials = num_trials

    def run(self, base_config: Dict, dataset_index: int = 0) -> List[Dict]:
        """Run full paper replication protocol"""

        # Extract dataset name from list
        dataset_name = base_config['dataset'][dataset_index]

        print(f"\n{'='*60}")
        print("STRATEGY: Paper Replication")
        print(f"{'='*60}")
        print(f"Dataset: {dataset_name}")
        print(f"Hyperparameter configs: {self.num_hparams}")
        print(f"Trials per config: {self.num_trials}")
        print(f"{'='*60}\n")


        num_domains = self._get_num_domains(dataset_name)
        all_results = []

        # For each test domain
        for test_domain_id in range(num_domains):
            print(f"\n{'#'*60}")
            print(f"TEST DOMAIN: {test_domain_id}")
            print(f"{'#'*60}\n")

            # Phase 1: Hyperparameter search
            best_hparam_id = self._find_best_hparam(base_config, dataset_index,
                                                     test_domain_id, fit_simple, CSVLogger)

            print(f"\n>>> Best hparam_id for test domain {test_domain_id}: {best_hparam_id}\n")

            # Phase 2: Run multiple trials with best hparam
            domain_results = self._run_trials_with_best_hparam(base_config, dataset_index,
                                                                test_domain_id, best_hparam_id,
                                                                fit_simple, CSVLogger)

            all_results.extend(domain_results)

        # Aggregate and report
        self._report_aggregated_results(all_results, dataset_name)

        return all_results

    def _get_num_domains(self, dataset_name: str) -> int:
        """Get number of domains for dataset"""
        domain_counts = {
            'PACS': 4,
            'VLCS': 4,
            'OfficeHome': 4,
            'Camelyon17': 5,
        }
        return domain_counts.get(dataset_name, 4)

    def _find_best_hparam(self, base_config: Dict, dataset_index: int,
                         test_domain_id: int, fit_simple, CSVLogger) -> int:
        """Find best hyperparameter configuration based on validation Y_L accuracy

        Uses WandB-style model selection:
        - For each hparam_id, find the step with best validation nacc
        - Compare the best validation nacc across all hparam_ids
        - Return the hparam_id that achieved the highest validation nacc
        """

        print(f"  Phase 1: Hyperparameter search (testing {self.num_hparams} configs)")

        best_val_nacc = -float('inf')
        best_hparam_id = 0

        for hparam_id in range(self.num_hparams):
            print(f"    Testing hparam_id={hparam_id}...", end=" ")

            # Create config for this hparam
            config = self._create_config(base_config, dataset_index, test_domain_id,
                                        hparam_id, trial_id=0)

            # Run experiment
            result = self._run_single_experiment(config, fit_simple, CSVLogger)

            # Find best validation step (like WandB does)
            log_path = os.path.join(config.log_dir,
                f"{config.algo}_{config.dataset}_test{config.test_set_id}_h{config.hparam_id}_t{config.trial_id}")

            best_step_info = self._find_best_validation_step(log_path, metric='nacc', maximize=True)
            val_nacc = best_step_info['best_val_metric']

            if val_nacc is None:
                val_nacc = -float('inf')

            print(f"best_val_nacc={val_nacc:.4f} (at step {best_step_info.get('best_step', 'N/A')})")

            if val_nacc > best_val_nacc:
                best_val_nacc = val_nacc
                best_hparam_id = hparam_id

        return best_hparam_id

    def _run_trials_with_best_hparam(self, base_config: Dict, dataset_index: int,
                                     test_domain_id: int, best_hparam_id: int,
                                     fit_simple, CSVLogger) -> List[Dict]:
        """Run multiple trials with the best hyperparameter configuration

        Uses WandB-style evaluation:
        - For each trial, find step with best validation nacc
        - Report test performance at that step
        """

        print(f"  Phase 2: Running {self.num_trials} trials with hparam_id={best_hparam_id}")

        trial_results = []

        for trial_id in range(self.num_trials):
            print(f"    Running trial {trial_id}...", end=" ")

            # Create config for this trial
            config = self._create_config(base_config, dataset_index, test_domain_id,
                                        best_hparam_id, trial_id)

            # Run experiment
            result = self._run_single_experiment(config, fit_simple, CSVLogger)

            # Find best validation step and get test performance at that step
            log_path = os.path.join(config.log_dir,
                f"{config.algo}_{config.dataset}_test{config.test_set_id}_h{config.hparam_id}_t{config.trial_id}")

            best_step_info = self._find_best_validation_step(log_path, metric='nacc', maximize=True)

            result['test_domain_id'] = test_domain_id
            result['best_hparam_id'] = best_hparam_id
            result['trial_id'] = trial_id
            result['best_val_nacc'] = best_step_info['best_val_metric']
            result['best_step'] = best_step_info['best_step']
            result['test_nacc_at_best_step'] = best_step_info['test_nacc']
            result['test_oacc_at_best_step'] = best_step_info['test_oacc']

            test_nacc = best_step_info['test_nacc'] or 0
            print(f"test_nacc={test_nacc:.4f} (at step {best_step_info['best_step']})")

            trial_results.append(result)

        return trial_results



    def _create_config(self, base_config: Dict, dataset_index: int,
                      test_domain_id: int, hparam_id: int, trial_id: int) -> ExperimentConfig:
        # Extract dataset/path
        dataset_name = base_config['dataset'][dataset_index]
        data_dir_path = base_config['data_dir'][dataset_name]

        config_dict = base_config.copy()
        config_dict['dataset'] = dataset_name
        config_dict['data_dir'] = data_dir_path
        config_dict['test_set_id'] = test_domain_id
        config_dict['hparam_id'] = hparam_id
        config_dict['trial_id'] = trial_id

        # Separate known fields from extra
        known_fields = ExperimentConfig.__dataclass_fields__.keys()
        extra = {k: v for k, v in config_dict.items() if k not in known_fields and k != 'extra_params'}
        config_for_dataclass = {k: v for k, v in config_dict.items() if k in known_fields}
        config_for_dataclass['extra_params'] = extra

        return ExperimentConfig(**config_for_dataclass)
    def _run_single_experiment(self, config: ExperimentConfig, fit_simple, CSVLogger) -> Dict:
        """Execute a single experiment"""

        # Setup logger - FIXED: Create directory and separate file path
        log_dir = os.path.join(config.log_dir,
            f"{config.algo}_{config.dataset}_test{config.test_set_id}_h{config.hparam_id}_t{config.trial_id}")
        os.makedirs(log_dir, exist_ok=True)

        # Create CSV file path inside the log directory
        csv_file_path = os.path.join(log_dir, "metrics.csv")
        logger = CSVLogger(csv_file_path, root_dir=log_dir)  # Assuming CSVLogger takes root_dir

        # Run training
        fit_simple(
            exp_dir=log_dir,  # Pass directory, not file path
            logger=logger,
            seed=config.overall_seed,
            trial_seed=config.trial_id,
            hparams_seed=config.hparam_id,
            algorithm_name=config.algo,
            dataset_name=config.dataset,
            data_dir=config.data_dir,
            num_workers=config.num_workers,
            test_envs=[config.test_set_id],
            overlap_type=config.overlap,
            holdout_fraction=config.holdout_fraction,
            n_steps=config.n_epochs,
            checkpoint_freq=config.checkpoint_freq,
            model_checkpoint=config.model_checkpoint,
            teacher_paths=config.teacher_paths,
            num_domain_linked_classes=config.num_domain_linked_classes,
            num_classes=config.num_classes,
            auto_augment=config.auto_augment,
            augment_search_epochs=config.augment_search_epochs,
        )

        # Load results
        result = self._load_results(log_dir)
        result['config'] = asdict(config)

        return result
    def _load_results(self, log_dir: str) -> Dict:
        """Load results from experiment directory"""
        results = {}

        # Load epoch history if it exists
        epoch_history_path = os.path.join(log_dir, "epoch_history.json")
        if os.path.exists(epoch_history_path):
            with open(epoch_history_path, 'r') as f:
                results['epoch_history'] = json.load(f)

        # Extract final metrics
        if 'epoch_history' in results:
            for split in ['train', 'val', 'test']:
                if split in results['epoch_history']:
                    results[f'{split}_final'] = {
                        metric: values[-1] if values else None
                        for metric, values in results['epoch_history'][split].items()
                    }

        return results
    def _find_best_validation_step(self, log_path: str, metric: str = 'nacc', maximize: bool = True) -> Dict:
        """Find the step with best validation performance and return corresponding test metrics

        This replicates WandB's model selection logic:
        - Find step with best validation metric
        - Return test performance at that step

        Args:
            log_path: Path to experiment logs
            metric: Which validation metric to use for selection (default: 'nacc')
            maximize: Whether to maximize (True) or minimize (False) the metric

        Returns:
            Dict with best_val_metric, best_step, test_metrics_at_best_step
        """
        epoch_history_path = os.path.join(log_path, "epoch_history.json")

        if not os.path.exists(epoch_history_path):
            return {
                'best_val_metric': None,
                'best_step': None,
                'test_nacc': None,
                'test_oacc': None,
                'test_acc': None,
            }

        with open(epoch_history_path, 'r') as f:
            history = json.load(f)

        # Get validation metric history
        if 'val' not in history or metric not in history['val']:
            return {
                'best_val_metric': None,
                'best_step': None,
                'test_nacc': None,
                'test_oacc': None,
                'test_acc': None,
            }

        val_metrics = history['val'][metric]

        # Find best validation step
        if maximize:
            best_idx = np.argmax(val_metrics)
            best_val_metric = np.max(val_metrics)
        else:
            best_idx = np.argmin(val_metrics)
            best_val_metric = np.min(val_metrics)

        best_step = history['epochs'][best_idx] if 'epochs' in history else best_idx

        # Get test metrics at that step
        test_nacc = history['test']['nacc'][best_idx] if 'test' in history and 'nacc' in history['test'] else None
        test_oacc = history['test']['oacc'][best_idx] if 'test' in history and 'oacc' in history['test'] else None
        test_acc = history['test']['acc'][best_idx] if 'test' in history and 'acc' in history['test'] else None

        return {
            'best_val_metric': best_val_metric,
            'best_step': best_step,
            'best_step_idx': best_idx,
            'test_nacc': test_nacc,
            'test_oacc': test_oacc,
            'test_acc': test_acc,
        }

    def _report_aggregated_results(self, all_results: List[Dict], dataset_name: str):
        """Aggregate and report results across all test domains

        Reports test performance at the step with best validation performance,
        matching WandB's model selection strategy.
        """

        print(f"\n{'='*60}")
        print(f"FINAL RESULTS FOR {dataset_name}")
        print(f"{'='*60}\n")

        # Group results by test domain
        domain_results = {}
        for result in all_results:
            test_domain = result['test_domain_id']
            if test_domain not in domain_results:
                domain_results[test_domain] = []
            domain_results[test_domain].append(result)

        # Calculate statistics for each domain
        domain_stats = {}
        for domain_id, results in domain_results.items():
            # Extract test metrics AT BEST VALIDATION STEP from each trial
            test_naccs = [r.get('test_nacc_at_best_step', 0) for r in results]
            test_oaccs = [r.get('test_oacc_at_best_step', 0) for r in results]

            domain_stats[domain_id] = {
                'nacc': {
                    'mean': np.mean(test_naccs),
                    'std': np.std(test_naccs),
                    'values': test_naccs,
                },
                'oacc': {
                    'mean': np.mean(test_oaccs),
                    'std': np.std(test_oaccs),
                    'values': test_oaccs,
                },
            }

        # Print per-domain results
        for domain_id in sorted(domain_stats.keys()):
            stats = domain_stats[domain_id]
            print(f"Test Domain {domain_id}:")
            print(f"  Y_L accuracy (nacc): {stats['nacc']['mean']:.4f} ± {stats['nacc']['std']:.4f}")
            print(f"  Overall accuracy:    {stats['oacc']['mean']:.4f} ± {stats['oacc']['std']:.4f}")
            print()

        # Calculate overall average (as reported in paper)
        overall_nacc_mean = np.mean([s['nacc']['mean'] for s in domain_stats.values()])
        overall_oacc_mean = np.mean([s['oacc']['mean'] for s in domain_stats.values()])

        print(f"{'='*60}")
        print(f"OVERALL AVERAGE (as reported in paper):")
        print(f"  Y_L accuracy (nacc): {overall_nacc_mean:.4f}")
        print(f"  Overall accuracy:    {overall_oacc_mean:.4f}")
        print(f"{'='*60}\n")

        # Save aggregated results
        output_dir = all_results[0]['config']['log_dir']
        output_path = os.path.join(output_dir, f"aggregated_results_{dataset_name}.json")

        with open(output_path, 'w') as f:
            json.dump({
                'dataset': dataset_name,
                'domain_stats': {int(k): v for k, v in domain_stats.items()},
                'overall_mean_nacc': overall_nacc_mean,
                'overall_mean_oacc': overall_oacc_mean,
                'note': 'Test metrics are from the step with best validation nacc (WandB-style model selection)'
            }, f, indent=4)

        print(f"Aggregated results saved to: {output_path}\n")

    def get_description(self) -> str:
        return f"Paper replication: {self.num_hparams} hparams × {self.num_trials} trials × all test domains"


## Run Experiment

In [74]:
keep_alive = KaggleKeepAlive(interval_minutes=30)
keep_alive.start()

print("\n" + "="*60)
print("OPTION 3: Full Paper Replication on VLCS")
print("="*60)
output_dir = "/kaggle/working/experiments/"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
exp_name_oh = input()
config["name"] = exp_name_oh

exp_name_oh = config["name"] or f"{config['algo']}_{config['dataset']}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
exp_dir = Path(output_dir) / exp_name_oh
exp_dir.mkdir(parents=True, exist_ok=True)
log_dir = exp_dir / 'logs'
log_dir.mkdir(parents=True, exist_ok=True)
# Setup logging
config_logging()
logging.info(f"Starting experiment: {exp_name_oh}")
logging.info(f"Output directory: {exp_dir}")

# Save config to experiment directory
with open(exp_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)


logger = CSVLogger(log_dir / 'metrics.csv',  root_dir=log_dir)

runner = ExperimentRunner(PaperReplicationStrategy(num_hparams=5, num_trials=3))
results = runner.run(config, dataset_index=2)


OPTION 3: Full Paper Replication on VLCS


 random


Seed set to 1



Experiment Strategy: Paper replication: 5 hparams × 3 trials × all test domains


STRATEGY: Paper Replication
Dataset: OfficeHome
Hyperparameter configs: 5
Trials per config: 3


############################################################
TEST DOMAIN: 0
############################################################

  Phase 1: Hyperparameter search (testing 5 configs)
    Testing hparam_id=0... DEBUG: overlap_type=high
DEBUG: domain_class_filter=[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37], [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43], [27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64]]
DEBUG: len(domain_class_filter)=3
DEBUG: test_envs=[0]
num_envs=4, num_filters=3, domain_class_fil

  0%|          | 0/5001 [02:37<?, ?it/s]

...


=== Step 600 (Epoch 9.89) ===
step_time: 148.2102s
TRAIN:
  loss: 1.5101
  acc: 0.4579
  precision: 0.5155
  recall: 0.4579
  f1: 0.4748
  oacc: 0.5505
  nacc: 0.2373
VAL:
  loss: 1.1945
  acc: 0.4999
  precision: 0.5644
  recall: 0.4999
  f1: 0.5112
  oacc: 0.5315
  nacc: 0.2161
TEST:
  loss: 2.6848
  acc: 0.3547
  precision: 0.4075
  recall: 0.3547
  f1: 0.3309
  oacc: 0.4280
  nacc: 0.2449
OTHER:
  loss: 2.5944
  acc: 0.3618
  precision: 0.4009
  recall: 0.3618
  f1: 0.3255
  oacc: 0.4225
  nacc: 0.2706


 18%|█▊        | 900/5001 [22:17<1:11:44,  1.05s/it] 


=== Step 900 (Epoch 14.83) ===
step_time: 149.3607s
TRAIN:
  loss: 1.2346
  acc: 0.5531
  precision: 0.6003
  recall: 0.5531
  f1: 0.5677
  oacc: 0.6004
  nacc: 0.2556
VAL:
  loss: 1.0712
  acc: 0.5322
  precision: 0.6011
  recall: 0.5322
  f1: 0.5432
  oacc: 0.5413
  nacc: 0.2165
TEST:
  loss: 3.0869
  acc: 0.3459
  precision: 0.4123
  recall: 0.3459
  f1: 0.3192
  oacc: 0.4315
  nacc: 0.2176
OTHER:
  loss: 3.0775
  acc: 0.3490
  precision: 0.4512
  recall: 0.3490
  f1: 0.3283
  oacc: 0.4350
  nacc: 0.2067


 19%|█▉        | 955/5001 [23:01<1:37:32,  1.45s/it] 


KeyboardInterrupt: 